# Actividad 8 | Continuidad operativa y recuperación del modelo de riesgo crediticio

**Propuesta de negocio.** Un modelo predictivo aporta valor cuando sus resultados se mantienen confiables y sus respuestas son oportunas. Este proyecto integra un modelo de riesgo crediticio, indicadores de desempeño, alertas preventivas y procedimientos para responder a dos incidentes controlados: alteración de información de pagos y aumento de latencia.

**Resultados disponibles:** evaluación sobre datos históricos etiquetados, monitorización de solicitudes de prueba, dashboard de SLO, registros en MLflow y comparación antes/durante/después de cada incidente.

El notebook ejecuta el flujo completo en una misma sesión de Google Colab. El modelo validado y sus parámetros de referencia están incorporados; los 30,000 registros del conjunto público se obtienen automáticamente. No requiere cargar archivos de otras etapas ni ejecutar otros notebooks. Las pruebas no representan decisiones crediticias automatizadas en producción.

## 1. Preparación del entorno
Se instalan las bibliotecas necesarias para ejecutar el modelo, almacenar las evidencias y registrar los experimentos.

In [ ]:
%pip -q install --upgrade "scikit-learn==1.6.1" "mlflow==3.5.1" "SQLAlchemy==2.0.44" "kagglehub>=0.3,<1" "numpy>=1.26,<3" "pandas>=2,<3" "scipy>=1.11,<2" "matplotlib>=3.7,<4" "joblib>=1.3,<2" "psutil>=5.9,<8" "requests>=2.31,<3"

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from zipfile import ZipFile, ZIP_DEFLATED
from IPython.display import display
import base64, hashlib, io, json, platform, shutil, time, uuid, warnings

import joblib
import kagglehub
import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
import psutil
import requests
import scipy
import sklearn
import sqlalchemy
from scipy.spatial.distance import jensenshannon
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, average_precision_score,
                             confusion_matrix)
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore', category=FutureWarning)
plt.rcParams.update({'figure.figsize': (9, 4.5), 'figure.dpi': 125})
SEMILLA = 42
NUM_SOLICITUDES = 500
TAMANO_VENTANA = 100
BASE = Path('/content') if Path('/content').is_dir() else Path.cwd()
SALIDA = BASE / 'actividad8_resultados_integrales'
REFERENCIA = SALIDA / 'referencia'
ARTEFACTOS = SALIDA / 'modelo'
LOGS = SALIDA / 'logs'
FIGURAS = SALIDA / 'figuras'
RUTA_EVIDENCIAS = SALIDA / 'incidentes'
RUTA_FIGURAS = FIGURAS
RUTA_LOGS = LOGS
RUTA_SALIDA = SALIDA
RUTA_RUNBOOKS = RUTA_EVIDENCIAS / 'runbooks'
for carpeta in (SALIDA, REFERENCIA, ARTEFACTOS, LOGS, FIGURAS, RUTA_EVIDENCIAS, RUTA_RUNBOOKS):
    carpeta.mkdir(parents=True, exist_ok=True)
assert sklearn.__version__ == '1.6.1', 'Es necesario reiniciar el entorno si había otra versión de scikit-learn importada.'
assert sqlalchemy.__version__ == '2.0.44', 'Es necesario reiniciar el entorno si había otra versión de SQLAlchemy importada.'
with sqlalchemy.create_engine('sqlite+pysqlite:///:memory:').connect() as conexion:
    assert conexion.execute(sqlalchemy.text('SELECT 1')).scalar_one() == 1
np.random.seed(SEMILLA)
print('Python:', platform.python_version(), '| sklearn:', sklearn.__version__, '| MLflow:', mlflow.__version__)
print('Evidencias:', SALIDA)

## 2. Modelo de referencia y parámetros aprobados
Se conserva el modelo original y se verifica su integridad antes de utilizarlo.

In [ ]:
# Modelo y referencias integradas en este notebook; no es necesario cargar ZIPs.
import base64 as _b64
ARCHIVO_INTEGRADO_B85 = (
    'P)h>@6aWAK2mp97cU{pW*XYp%008+3000#L003=ebYWy+bZ>KCZEs{{Y;P`Vb8l{aR@-jeHV}Q!R|xymqFIG^iSE<d36cfuCP<v3MNtqGMXi}iqM(ZrBh9~e'
    'sGGIhqKkddaLx?R+(>?V5k<SE^-tFHes>k^T4%eHSEjYO>z#346}q#^Ic0m@XoZzFy?b=I&Q-d5Ihz}s+uArCJly%*mOg5&t^3y2I`>g-yWB>1y0@*<imSuw'
    'Ld33ls<X*au6x+44^9T-s`6EC3q6$m-WU6c(B9}a-<#5b?{2@~N8{R8S!<Q)9)tboo1?lJP1FsT#9?`abq|_wvA{Gckj|LQ1tmfxI2R^QX=b@e)7%s|MJPdp'
    '7glOct;vOy7)z5T7$?GLJozlFdC+R;O>jvN5qpgHgx(`|h2<3?v7lTs{u4r12u&uBw(Xp+RrAz8_&V6p7{fn3S9gDvR=1-b$0DBh)8)yxuCJg>V-#;&orjD>'
    '*WP!%t4|ZG8VjLL10(>dFvi>VkWwgh_iEP8_1;t~{^GOJWoPM{z`=t|62u`@hFZxJVlXOXW~>&PCl)b8Q;rFinV?ykTbXE@TF5Yg4D*cTVs>jQSC*k&7@2wq'
    'N_rY>H$}w9HRH>ns&%KDP)H!%!nduJZ=u-$^#_0ec6+=#-mB}w?dTd7J{*2mXe*GdK(_+h3VbVwMYulR-m1ghJ-%EZmkacAfn6@}%LTFBa=XbalM9(%$m~Mq'
    '7qZxa_~&^x?N?I{Fnrx}3XiWByH9U^TRz{s{(N(|KfeEES-3lV{CGUPeY41$&v{VeMn6^1NO5hy_HpIw{^6OxqlV$nN~?VK#lrlL?esr}qfu)c=vl9(?)2(W'
    'm(UJ_&GgDX3@Ms;h_kPC=Lgr=7X0YS_N|8ul=kFu9}@6w;y(|A8mA2gNVRZQff~h#zdVOj`m1tqkbxEx^dCy_QK%Xgf`9xka|ctctEPlr56CbAf$8^=Y3~|{'
    '6=t}CK@3~$-np({=6f0Mn^Je3D;$s*tp+&xATPSKZLgn?ZaoHo*<Ah(re3*n=z0ha(!UA6&}HXUZBKfx3y+t`CRO7*H>+^`OgH4ynQ=nrlFSYpUj_=}eeem='
    'ZQvpqZ-Sd&qrXQ|(GS-@F5NTn->?nGIf&2Y-1R{wER_qbCdfEl;Kih0vfT$YMk$$PZOqVW^EB?kCC6g(n=zR#bqyDpIs=P@Aw&hol1XSZI)mXzGA<m^BZyK='
    'B*6$Vk_x_(1dJ}HxKhk%LJ=_LG=&3np^Zl^gZqTzgpYR+fan@!JLyFN6cU_B&dA!4aQW%E=-m_;V|4plXn3RaU`9xiBrM?q!#)II!PanA1m~0`R5D=9Il@aU'
    'S?laX22)OPijgEq%5W-^88LR4>U|HbxM?%6p$y5jp(a@4TLt+-q1QwjpVed2UU_(d)T)GO49^SYi=uS3eQvaRsBG)if}Cb<9m_CIp#@!C*mm>~xjGL|k3pBw'
    '?CfP!`{>>M{fD(3>+m!Qc2&2mH5{obIHxg@3BiFo!U<(~DP<%1gy1oinDI0mOIV5|SpjkZBr;_pC8?B>O1cyw1fiE*xFig5IVJ}Bg0D%bA;E#$ht@-`A;3r*'
    'ox^=<{LfLTqa{~S>7n~VY{u7cZ1lsEjfOh<bo)Nuz4-nQP)h>@6aWAK2mp97cU=_KL*6d|001BY000;O003=ebaH89VRK(>X>Mg<Ut(c%WiD!SZ*EnP-D<=j'
    '5QXpeDMGGVD9en__|wbw5emHxuA7B`U4t=&()QhJG?ZPhJf2VI9KR1;*UvFbdGfA*?)tYejNVW2>dvuyeeZrcI3jfqFOU7x#;zZy<U%30<9c!}LIFS(c~gRn'
    't?Dvw3QYqq&Ep1c7J?B~Iq%ox{8xRKEu*?wl{mTJu6{fZcc{_?AQht}Ys#*oa~l1$lnND*5``5AK*A{AgB8Ms*1{l5Euo}jX@J|b3s#RbBBN%Zl*WR{y;W+i'
    '(q*(pZJ7k@wcO$c7sgaJtqhU8U2S~ygIj$CwAQ3qVVNKR73~%NeP&5oS}Hl20bBl4n#L9aB}+62R%k<LtggO~OO8X1*@a;W>ynep(`PI<B!NA`4K&9zl;45k'
    'u`<WIpFjNm0Z>Z=1QY-O00;nhF?U^wllzBP0{{Sg2mk;h0001RWpZX|Y+rI^W@U0^ZewX-UuAA|a%FB|ZE0n0bZ;(Wb9R+gJ#XVU6zunZY{w9M-(;EX0%U<J'
    'wz<mi<x=F3*o%BD?&sf77AYCmY#v}swxk)p;k?nW<LlG$ujj|~aQ-;Gr<cd$DZRb_onD?!$-XAW$I90fI5vD<j{l~g=huhF)BEwC$HVF4`TUlC9iPvyk8g*M'
    ')6>KG?aTe$_q!j5Z@2e}YAGd`L>>~7X%;8qX6`x6#GRO_CMuV^c|F>CBpnyViOPD?@t2eSiTdS_+dmIw(%#^UO1-L<2<vcRB59&pjWykHzkB(j6>Q<_4C^`3'
    '0?vDI66JkS$8H{#ta+&|O5;yp%N4epEpSoXi%D41piiLb9u4x+zPLks)uIhoN{EPk0^6>z-C|d@LrZDFtyabE*7|$UWr6l9=vv9QiVm%<Q8Y3pb20r*=x=xT'
    '_lMi>KWM^RacY@M^;MG;EY4Y4g($3K6^Ph{Iv_>OVA#Yg((v2elrlpMbXsb*HmW9P3T=a5A@p3rW+9jlQ-NoTE#=8uW6o7oMR3(%A8!zKG}WwhLhhrIExBls'
    '+!sJ$oEfyLVUw0iOxn#|>gJjiD(a;G)X%eeX+?x#Ht-S@8oXxFJEzv9iwt{fS^lVw3Clb?M+?G5$4+3z!sik;OKP9~9JR@qN|(xLQ>+yuS5Z$lm7u~{f^e6U'
    'YFk!i(sUt!&Db(##koSg$s4-0zS&h8e3@I=sN_Y$F=nJ#ctbK=(^4#S(5mqoNzny&D^2ojJ2X{>8_VQJJClF&v{b2{oA9c<CaY*RQ;iS=$f7*lg;fnCF0>G~'
    '<u9!P?{(!7FcpI7SVA(d5OFR5nY#?swdq|g!&kdu#nz*dY7P$t?BsKVXiZ%jX?)p$HF-l|&-G3Vnq-^3(OSS7f<VWxB8sIa4zy|;M=gx*(Bn{d-Q$vdii^m`'
    '^GLtJhDaG~o2!lCD7ZI98e`M6n>vth?3k-MS*4Y=l`g6A#mmu0hjr&=7_;cu2@Lyea_8h|p21UPtwacGc&yLYBCgvOO0h_2mnX-&@p5#g?Nw$)%HXmc7e=IF'
    '=C(Z4z$ja%@0NGldPGQFamkKLk=Z<EMVn1IH>a8u%|bd0SBwAxiM?=to$QRl|8Fb$q^WL^XxtY*NwJ6j0Z>Z=1QY-O00;nhF?U^?w91UR0002>0RR9d0001F'
    'X>)XPX<~I_X>V>{aB^>AVPa`)X=Gt!WpiI}a&={5VJ>5Hc5PAZ3d1l6{O_|!FlkKFrx|7KAkcNA>)wA}x<CC0i07&2L8kI7>v${kV_eE>;c2Ts-}hGE`*|%x'
    'Kae<K3*{y&RUpo@4;l!-LS^>?``C4K)esjrYeUEu$}wXwK(pBoOXMIpapsLXC^sCErz>7uWS?Lhdr{3e1|u#ix4;!DYsDziUc|Fv|HK+)3b2^4)gJpn5?#<n'
    'nTr&u0&)3oDFf?KqjvNp(b*CqS)96bL3;cm{Qyu)0|XQR000O8crkZf>DHhIp8x;=<NyEw6#xJLY+-a|ZewX-UvFk+Y-w&~E^2dcZZ*z54udcZ1>l`iM41uF'
    'f8w5?V+VE=ksw(jNd=`#)w`EKd7J&|S^hMH(7dI%lp)1{BbFRjdLvx0w%QAyWgjW#DK0UO=n3Y4Zw#BCNtvb@b8Ja2<lgrWdUUpo!H3#P>$!PqHl%u1b^@&6'
    'obBOODsD~(eY)->aHXBA4#wI)!^IYyGEO)EkRa8;^>KgQ*B}(rz|9^ljBdU^P)h>@6aWAK2mp97cU|a%b+J870RRck0{|-k003cfbY*5?V{~tGFKusRWo&O>'
    'a%p9AXK!C}Wo2%3a%FB|WN$8NZ(?j|V(gs>Tug8K|EG;gB^8Py2_b2d2s4qSP4=Qp+tfrVEtGYvg+Vc;h{m3^P-Hz>%bq>^P{@{$WdG01bxqD0b$t6i&+~iz'
    'f4}cDuNR*=Gjlg*&biNhz3<O;opX4~sYetT(g$BVW>PpOAUeV#DlCc<9u~n7@KwEJPge=L&K1lOnDJF&VmVPU0zdK7reTw#;$j1$1$;eOQ|T)l3wB^YjJUbx'
    'NbzzwXQ=eW=wE+BYjRjjOjtxHdul*<94AIFWKJ!vgJwW*Fgq+FI4me2HZoeUn6EpD!--;ZCP&34utPWjv2oGj&oAby#>7Sk#BxFt1pFqGIl*B85dyw?NLY9{'
    '{7Hc)UpXi;DnXFMZ-K4Y5do7qG2*|?7Vvc^Mh1q51zJSJO^!-nM}-B6*IvMH;VD^kjE)XS7#kfB6-8cS)tI=zpzwg07y(~J+DgE08W9{I`K<Uu;*G)$h~fy`'
    'bQg-BM@9#WoAMO~iJu3@CWv3>wv{~-e^zmry<l-tqF{u;%uR`}FjL^^>FN4Y{E+_V#*7sN^3?;v!y~7$Cr=KD693q?u%6OyV+VwXg+@%~M8vWK6Jo{N!c99a'
    'kf)OBCEJUwq#w3m2SrA6EGEZ=$A-aL@-^7eoS?`E@rH{Fij}VP??1$iu~0BbAy5Eory%(~T&2dX`(jDYP;(#ZF?1ArprgC^YrGtN*p{f)s5Yp!sJ&3_1bnrD'
    'L*3okjvk||NYAZF&uvK0ZAs62k)GR0`mXrz!Jm;mwf^<g=GRl(Ur&4edTJ-&h<}Ym>Oux?SgnY<4k+zKZn}JRc1Tz(8~3p(UonKTb^6I|9fowz)^%>SxRW-E'
    'i4BMii-`>jieU@5t)a7V+i;Dz?WEh@nA?GC!nNdfOX7By>}c(uau4aMwN`Ls{EbIp$xW6-jpyBvYT?xDQsqs(CRN7DtMK~V5?JO6L8G!jjWcn1^>3P*uP7T;'
    '*f=(@r*fiW!Xm{7s*06`orRSkE^v|L=#32KghvYaePu%kCnAP3IWSy2h;YSQh8-Fm5F94$?vat=4HXe;@!Kl`emfWOM~C3+1M%fSlH)ZjB#e9<D~Cr)j$zgC'
    '$WS&s)s%j#WH6EpD1sr(?HQ8C8ouh}fOvM8WZZI#7mrxt1@R|C*b(CO6(4oY`D&7vf;myKT=9<7o-FPXF#(d%E=Kw}ftwk>MYy&2m<x>y4~Pz%F5b`L-#$dP'
    '@%b8(=5-_55M~v=F8$iTun6&Y{N!s+j*N(mjf@BjVh6><iod6MScG_u0)ixe3}4a<k_?2A(c)ee{_78lN5kmI$=Emf>eB+EC$q(!Ejm^_o@j}Gad?7wF)}Jj'
    'viTPCo5qNT3&~TiBVSiMf`$c4SC<`vUnO3@L%3PABcvNeydBs<+<=Hs@u;WHSBi}c7YrFN{r3AcM{`{GO~t!9Fj5==+{C{#Iv^rAQu^iM&Fj{UuOS^r+0!^-'
    'p<MV$F>LZr5Wl3!j*8|)MMnm4Vq(Nmz>}}fmitgNXR>634i<QF&BX_zs`PJiem{s5>kcCE;Tc9cJh@iCADnfEnHu%@5<hTsWu%%b){V35*l3RUE5&g_x<z1%'
    '3EUJUBQ5@Bj(80l9+8{Gh<{3eq<2czF76L&H%K+){$sY}y9JBHpW?TY?QFI<HbK{8M@tR{J@`IY2aCG!rp;!Dav~(_DIR-*B+*s;y-|EUwz#*&{{9wUSMJXQ'
    'h1y9+>p?@rx=OH6ybh8hN|VyM?gJ7|knC{Du9&r?g?hYJ)~(K_Ph*OrI6Ji_?qcwj!XknMOIkEdR6E#RXU2hB!!^&h(Vv#bh~uks17g^5;*M@(E$Kk2lBb+d'
    '$#E!tvkhgf<i5Ktn;n%9&yI_L-^?B<-u@#wQ{p%gL6Y#royXUZd&$*g<4Yw^=~m~4awiT_-QWM^g}QC*#*phy%H-}8XZshsllrgmI!U+4^k90v`Ns4ODH)@)'
    'rliit{^X|IRorBeAW*Vh`OVnBf{i55sNi6wBO4^8%NFpWwo5>G$iT3O$jM;=;qH=`e!*Ih^dsDKzRsY8`aUN9E@d&_P5S*$OJ7uTj|>fqb(cJs{!PA?_^6AH'
    'jAwJjN2Ub27W1{GOc5X+j(@$TCLWvMfE2hX;+Q2KRoDsQLq-f|+)aGdq4?UbbyX4*k7R!<bHxLL<S^2dAdtA1gvZ5%P34GDPD}C*`IWdp${!aG8o>hYF5F`|'
    ';=Pyaroi31n7eOrT%h=~I>C`_>d?m)udE(BE{4M<4SHe;5Pw@U$tcX83?Rt2{LA4<+$usEl+@Yct?eNPl1(CDaQ9#+@YU)LT!AY~Q%Q^%)KN|nS0+V7PK&5}'
    'QM^GtxnsByk}Vv|jphUdi?^@>*SkS<kaSZkQa3fD>847g@s9LJ9VH7{t*O{9-9BN#@oY(?YA#ujtWpd&GCWv35Q{NTZmm4(0h;FZ@l5bJ;0OWm2V-I>XzRzI'
    'pPz!>dkK2%EpW@Q_n?K<pjMwiH+}}y{RX<=2XM`FhLoFF`+=VdUn#<}sWL44s=#t;6Ii~~6qcK)!SVogSf17lmUA><xk3|`jhjnVXs#_)Q8B17r-fAg6?EYJ'
    'b)cd?y72xjQ2%H>cweg}yuSm~ct|UFzYJ73r!~CaRv+Fk0QDbb0Pk0Uiui5d{jP1{{WGA#07H2HJE-v*BY5Ag9lU=HR20!(s>v$GQq3I+TCxGO>Mf}DfDTf%'
    '<b!(M1&uN@f%a2Db5Dbo=yZhk{-D}9pqAf3z1&QpeFkXqbI{ygouK`E(5kDQaQk$Ic1J;@)Vttz>Vn(J%rUhic51mks$NeYZ6@D88MOqcNTy7D9aS0nB|`&M'
    '2elPy8`O5J#;k|Zf4~2K?EBRk_5D@<(eM3p|NHlS|Bt`&{oKWc&^Qc~HyD>qP^$>W|4$DQ8-BM}C}_!KP|IkX$Hsz+P`%<o{pGYO0q4Q9ussiyH4ij)0ch<a'
    'Y?lO@xfqnc6yIM4s+|lvAO+*xO3=JjpoXcSyfn~VIKS?k0b0EV^Z?GoJFf>liSu*44fy&-&_dLtO&Cu%g9dH^U9%PR(Kb+}?VuVvK&^LzPTK`qwi}d_0~(hL'
    '@s+(7)NvoE&VJAbpp5)HSe~8_%YzSq8XSbzKcQYIz<7QL-d}qdme(8sjXnym`y2yxISy(@!22475a*LY8DBwJ(@(&%?n(IE7F7LH@OdF9&+asQZX+nq_6)pV'
    '3Cdc27SyK*Ue_yz&%Xv`JvfJd?>xLd=>ojo?IP$QP+ssQ_*}=!@OmL=VfYoP3RS>=;rW6Z=YSS|2W7dVPm_jPhTPN!d>l~_`Z+nEg$m&FFnrPP*$3)x1U?XJ'
    'DX4J=^oQ1h@|15u`?a8~uD9Xu=YR@rfzw1gk=L~Ez~>i$3O|DKCfo%*2wG@&58HtnYu<<TnFY$K#ATlc@VR18<G~N1{Z&x^%a1@kMez5xff_3SHx_b0{Y#J|'
    'dp(hAl@Qd6Q6|-rnV{OwK#e`irJ5@QEolP$X*>~BcnW#6%`>S+1%k42L6hHuimac*au}#(DyaVfTz&>>+^PcB$GHO62XzOi@Fr-X&I?%oNuZ3kprS<JVt@Nr'
    '@cL~~FZOF_f9o~&+c%&kpvDs_;r;iZ+S$m}ovL8{=7Mt1f@bT!gXJ-x+B-q@n*)c7qCiWFa9QJnR0}*nv(r$^K}F+$+bd3iGR!|p)jJ6^>H}zrcMY^F1oiR*'
    '&M&?ITG6#us%FWcRVu*$taMO+-7iv2+79Y(_Z5C$K4{c1m?so|1x?EO4(kyC^M=}<KjH7c05u!{^9cS*Q2Q64W<y|JVVeVbx)aPVMy&<SG*yDn9|FxA5A%*Z'
    '5okFd<{w^NVg6B43~HGK^AO{<s!}a227R3j^AgKFFfVbcgn7wDEtrp-4$+cobs5Y{xT-J@S#AmQk2Tsb-%vdV^NRQr29Ay4ggQP-UI84SAB}u461l({Rj-io'
    'zv_Yf#0$W<jvSFI#}S1wpasYgRnzL{2z%s+QsfBJxuE*<@$W4}A3YIt0dmBGCCG2far{Pp@I`*mKyJ_%V4O#OP)2UhNe3-Peps{?*J~YUE98b4^!bl%pmM_;'
    '<c621t&kh~AvbW58w#>P6_6X;ksET68@%^`-UV*pt<FUbL2h6nH*`f_xQu!QxPh@3xq*k=FadahF&cP*;Rd|GYYn`>P(@xCg1j&gdBF#H!2x-p19F0l7j#b`'
    'KY+4Y0Vgo_ATRVqPB?f9xe+*lbql#6890I0A2@;c5tOkWd4U6*!0Qd1z*7TGD7*#A%0ganLQYTuPGA+FdH^RdDnWUtfEQRjkPl{p`k%u5jVf?~$P@Ws?NuD_'
    'LH(J)2gV7=38!(nE%HGCsA%hTjOX~e`wfhPH*h;3H<W{lrUNf9-rvH%54>Qkj=Zo8l*d3`Scbe%3+m4WPVg^6Zm<SUV9ZBexCbh<1WqVSy^nqm^1@u?g?kUs'
    '_W@oInF22)s{kix`-A#l0WCBJPB5MU8g&V|!2)?<4XFQ5Q0*w>h8w88fETpafLba5CzOl>%{_?xpbmWC#Q`-wh|7w=1HwVb4e`&BS3$k*f>tRZA9MxP9s^pn'
    '1k|_y)c+M|vIB5}*U=Z)|9}^&YC)5Cyu$qooM4#;8sz|-z{>@#G6hcXKLc7A3%p>dh<xA=>LmmfJqInZ1Wrhv@eX+xG|C=0p<*p))n`z9Ti^rZ*`S3FaJd(7'
    'Le(bFVm06cvjo(qpjE!W3EXR-#vZ^4xyTDuEZ~IfbkLGdph6yU1M)&a5~!#Ja6-XxPzD<~L8t?qP<j)TwGVh9c`|T<eJkLC@(|Fx{h)fa=tFt|CzM~pe6BBW'
    'Li{OEy)fVeVRPVvi^o9)LBI*g??D-S;Dm2ZzzNk#zz2-&pq5#{2@BQ(Corx6Cm4SN|1vudIH8;iex+#{aKeuf;7i7;0v~V({!Kp6SOf82H6E13MozH#U&;r6'
    'O0<tdpMZn$Jq+Vl6qOtNksAsd;D$ok@(lcYb3oJ4e@JS;f4G7EgE1d;Gx7r$`Jn{)K@a)C0{Ni|`5_qjAse~D9JwJ7xxpK`p%}U0k{mZAA}{!$-*5=IVI=ax'
    'Qse~@`VC!?7iJ<aoJOBv5%R)f<b~PD3n9o0R>%oz=r5=OC-78|6JCJwKBK?z7J1<(@<LbO1jbP0gvr1OJU8Tofdu*$=qv0&Kfwt71Of7bC2~S8D9aT2U@<65'
    '3HYF}964dv8QhM@3GYx>0w?eW0w)w|Atziz^+#Ulft>INlrax^VH9$L8uGz@<b@F61V#(=6}ZR?2SG)(puz#j2@61tPa!{i1?5=+ANWUtvNoXKa3A@h)peXd'
    'BR8x@ZYTpS?2mOPE75Pzr1}kvF2D=C0#IQ;<c8z7krRO%gnLWTM*v<h&P9IE0$vbK1r@zP9Rs|;Lcf8fjlAH8enUQ}adY4WM#2O1bC4f`(05RJgggrB)f>2h'
    'u^Rc|Bl-}-kt4Q&mKXsqSk3{>eSphjfE%jzg7OTJACf_hzkv#U(QnuVYN?9c;0x-Hz5@e&hbmj(1jbbK8FHRu90g^yKyK&*%8Ni=SX+VP2QGI5UdX)&nmhq`'
    'f%gJb#6zE<8q{(baDzw{eTG<2kq+`g4yb4_a6_&F`V135jdz0bYCwf7-~{1PP~IEl1{dIjq#ei$$PM}qzzN>VLCxNQmW%~nu)hkLJQ_Hm_!y|M19HMv<c4;@'
    '2}v2C-dey1qDWBhceUs%0xxi_fD?@Oe?dP1IKlr7Y99IvT;POa1K@-bE#QRo9iS!cfD6K7L1S-&7T5tFcnd*ingJi&*bD090-Rv`0<>x+aDqM)_&|6Qw0a(J'
    'LijM?guG_J2lqwjD?A2HC}98}RB?e1^nYM}QxW)J{V?EzZx_HvXw?kzlr<&#e~%AJ(;)sA&jc+DL_X++T%i4bix1=<JU5J=fyfES3nf#iydeBTzJf6w=b^Ka'
    '7tm)&MxVhSeTMi%DlZtL&ya+?P%Xy`Lga-Tn6L0cUNFWy#W>`JXXrQNVxD3Y<|*tjPtgH+AryHb3VFd1dEpak2=YQA`V6O$8w`;b`l8R!6nP;OdBGfcfsJ_y'
    'zr7g0krzCX7hI4R?9f*jgS_B`zQPdXg#MU^;G(av40&NW^1>J@FYv~quV4h6z{^HeK|f(3stRyIVLbW@_fbb9FBG8;MP7J}`G{4Rj~Iu%pas0ZAW$b`KB5ig'
    'A-12xd;$6k>c9)U-N+C9fEyTJFh6ks_<`qzd5Kr3`RFrrMP8VWdIS}H2B91BLK6B5g~$yxs9ljCe32iP<MLTh-Uoc$1UX_P<}K!<<|9WuMIT}S`Vd<|dB(^Q'
    '*_gj*eGAtIdEyH$FF~$oiFu19psWwLJP~+;@dmkKEO3PJS@a`1qyG>ET6hHg2?g{c27(HcQHwxDYQPh`@u)eV{=LzkD95};Hu41%c*1KVs8IPaelMu#1^N@N'
    'Ptd0UHC6<k5G8>6-z~%A26&>X05rJ=@I=*IP~Icti~hh7CF?-7KOtv$06#>n1T8s-%L>2?A`9e)Xyk_j__`kY4y!>+%wA$10)2>R;D=;Y;0MMG^da7Z8qY_b'
    'XoL9*#kaVhL95Jw7fO@RhqwlsWC*;V8uAYF1)$#Dffrbbpq4Mue;5k9z+DEa`Uq6JA8<p-aZqC~;0DoUP*pqNhWNvvwVi+)k~e~uYNOw<7PPbjaziO-?lRzp'
    'Y<Kh-UVTNr0d9!$1#WQB2X0{81GPK|+>mX}Kt2Gy_!HDU9C$(h5oq`f;DvBC;Dr0RpuEw*3-%vCH=hMwFq;m%z_4wKz6B`n1@MCZNbnObJ^^0H)&(EIa{zEc'
    '_BG%H<wxKnOy#uxdwg(m9mN0B^W^Ff9C6wB|27{q^k5B=;|9!ASfbBhCg(GhOr`f3;?ZZYN1ve_`JoW`Artw*Nsb>Dpx?m5dWX$e@4#At^Ah9;FS&Y$D&z=0'
    'a>Qxmh_mQB7^3fxgdDL3IifXkgduW7bL0pE<cB=e=jcB;B1bGkjyR0|!&2l3b>xUc$Pr%15d!ob(vc%__Tl<qeM10p#A@^%QjsIJB1b$$j!;K_5TTY}e&RE7'
    'gevlb0{RU<Kv`2TFL4{y8~A~D82twa<c1CCJA6fr!MX+lIl>0{As_P<qmd)pV?Dze<cS>gAG|P6aUJU#GLa{|(SOiJez<_jxrp<B;0MMj<Om-653SIDC`Rpv'
    'bq#AVPf>x(-O+zYKwj8}9Ptp8)f|0>!N>_S(0>r3o<p8^g?SA0Bm5n(?jZ;{VhQp^9xgvbuCM`)5Mf?}7li)AcjO8#<~81-E=OPDCuref<cQY54Xins-?#`W'
    '>V<iY4D=;bfgAioupVL$@<bK#g+2Nb6OkX%K^fPPGYpU;#-d-5j65NFfIbOwMF!?Q`T$3iWP`HW0zXuxVZP%ts3rFa=B1vX4+H#=y9Tv6@PlX}XkjhpJHnr0'
    'J`6O9h4m4s=u4nK!E**~5DCzqxB{x(9C*Rf5Azr~6<7y=^${`14+_8yg$J-6*b{w-Dy)A<1a9zBdV|L!);qWYKky8I8=@{ENA$-$MG9!aOY|jr121@Gf>t#}'
    'PKW^YJ_lOZ9XLTa4gH4epjAHAm><A=#R%X9{gcQIeUTgXg7U^7FWdsP=b+zk2GlqZc)`UCc)|WBsQ!N7g_23g3!cCWNnb!qz5y?2Uk6@@?~lI1M$lwK<b(O3'
    '-=2ZSM_@jp6L3QOYtY<nzzM7X-~>TW;Do%-pe|Q{6AIQ~{y`Hs;dL2sLT!KGgs24Igym7-BMf}i^6&A%$_<!TSOn@7fqXC=xuDxWlMnuKzJg8V2yOHqET=Rs'
    'Ur~&84#qR7`3f`S2yNsDC*%l*97mKQM^qq3%#`DZnaB~f=tp!$jxa%v$i)0bO<Mi+57o#M4>6CCiFu5#$P-m^d5mSq5oeJj#v?~0AV)k!j&MMZc!zn6PskB3'
    'kt3d?AMpb@q6Gbj?dV7FvCd%xa)c-5Eu1lL5ruV;*U^t?jy}Y9<OoIdA)a8J!$ahW*62UXMpZ=rAptp}200=I{fK**$8g6whc(C#pD>@1f_{W9^22SccSylH'
    '2PXOupO7QgAxBKW^E(Eh577a+;Tif7Ld;({V*cU{Y9jg%VOMbejJz-g>l}7qJ>+YA-4cC>naB~CzhL}8p6G`@Lkw~P<}X;t4UB5!2;>G~f8>V<%x5Iy>&KBF'
    'YLOczpbt@jIum)JCDu=F!1~GV=sO&`joYIXeQ(TXXrS+a^^+_`<b`nbA1>ZSe+2UwGmsaGsdbdbqcFdbg3D*nhfu-#2pi;sS(x8Ahp!KNi1TdVg~A`8x$z>*'
    '=VPA37kS|p`VlOwhuDH@g!v3UsIU}$2zTU$o1i=g<OHmXsA`LOjRojGlww^(FRYtfg!K=Hv2OAUF1w)bAVhw^x(8!-%wIeKHOBmfNCh}SxC!eXywGoW3d)NI'
    'UP#u(y2#I<tcSo0_9KB0SUI5a+Q0`%aj1_#MSj2s#n&-EVG3N}H63~3GN^r@YV;343vH1T)?*!HTi}C|3{ZVz;Df@gpvIl3oKWKP8OJkF%e}w}g>2viZCBs~'
    'Z$sb%uL`VpECNmt62J+&h5#QN-2s}}4!D4ac?dHr-~;zu(4%p{2eI9N4@CDtOHKkOh-L#HxO0FHgvP)J1@4e{C~bj0LU+hF9Q_V;i`fUkM|j=(Z}P!^AOGQp'
    'HpX)+R3GF3S^YyeD%MTbB2T!)(fbjW=tsC9N0?&WLlo9M6w1{-urQC2E5{A;nv0r*JfVs_;f-31>Vkenp`3O@?pT9bj%q8Q=0`@M3Q?7jPxz=JR6FFATvYAf'
    'udh(A#;&INCT7SxJ5U)JRPC2R<)KlSr%6V=i0X=*bQ)E5-3OVjLA6AV%0(T8{){qm73)8&!Km4&CdhHJ>sCado<Oxkz8i>o615i9cO#YmdLaKL$!Qv@9&%wg'
    'Y75L;O+*!;@{t?Ap+;@{*E9+BDyltlXcj61^JFtnRgh1UQQeSJy-_EkYNHP}7xg=8F7j+V*6}j&{2x{WIvG_N{WC7=0o2yW!2&t0#rmFj^x1UxQ~b8Stjkch'
    'qJBjULci_?^7c&R?NgWslyUbD^zot&)~md1T<(Kh{uO<`VC3`1sLAN}c^{!DF9?^7(f_MP%|m{_hV@4*S^W{_6(684=z~1}5PiXwSckM9xjq<qUJJQBseWbg'
    'arrPR8~NT7>yy$^_oF`;i+mr4dB!yK1+B0SsXyiu-O>NEMgPwT{XG-pam@QMl8~oefSdjA11J0MK@N5YF3mN?I-4ZmNLDPK5Ay&xP2UbUiuds=>%V;Pzu}>W'
    '`2q(VuVnrM<}Vl#xExLO9cmlMUsTDs0eK+_c_9;d!5?`cT#gqC(QokP)AK?@nv6ODIpRw)l_O+zmfesWvK!zDQ{;zCSzhE1Jh4$$e}Vk4S&k=OqmDqHxQyEG'
    'FLK2K)N<7526(~=^&+aX98Ww%^+k@TmE#EkYB2J{E!3my|FhgM4waAk4RyF2KWszQ-azGtIjH5Rn#c>Ga{SP%0e;wp+6K9Sjrt7LM~)v>AUE{>7r7xCm5+K('
    'jvo@`_+cX|LyjK?%kcvb)fjo<FzQ6)hL*?;_xI3qgDdJe^vz7sH<Q)>zQ^SV<b@2>Z^#YHksD0@CO0fWU5k1Z^L!4-3wfyS$PGJ?8wMaZ9Hi$4#!2J{Ch~#`'
    'asmgn1M<QXIc{L1exc?KdDGGNTTJBzo;QJc3FL$GsIq$HGW7kHpQPo3!WFnI%Oh?_KA4H;LwCabp$l?BKAz{~ja=Y?=Rq&LK;Z($CRABn@^h?9Hp2YgMC5|?'
    'n4cSle%fL1&HS%o{>&5eU1PBhMh){%yMYhP!oi=3$^{>#Fq-jS|KR`BLw21D8yt6sq59$Y8I12^K7$cS^&wp3@*1^&@FQgPl~J>(`3;#rA;RlJSR!{6qE=zv'
    'qZBzL7xN!x$Ri@mgT%{x4CE3a`WdqGIFixVs9OH_c%>orM$Ja;j2yED)e<=-9JL(Pw*j6}Mt+G#tyx9o7@6)s4UyxQ&d4kLM*1O}QB9Cn4$E=NI2pglaY`}j'
    'I5|GCL^VS$DM79L5A#Vl>J8cJ$RP=+3^^Vtk@GPUkv~c{*3Ta@ZGpTo7gZm5V;!nKa>tv0(YJ^}6`+1b9U;dXbLFl-a$pCQGiJ(hMunU|vHTBw;f`8~d=ZFT'
    'aS^$~hw4Z0+96MDLJj?!T#<|%k%P-x=tCqTPjp0{;2}>qAxD%oz!M9RBW#f)uFClj!yE7&9-&@Dzab0#hMjVL!%fsR$PIDG4QDWaxR#z9cqyocc;1r%`Un-M'
    '!;u%7ppSspe=B^6^+l>wAA$7|d<6eBnCEk$`Ui}osL99)v#^dg6YFJsuzvP3_y<vuSf8p2e2|Rijj9g&@n8SoANA13`HLNnD_$6fFn^I8i1AkLJkBcgC+g16'
    'kh>m)j3-3M6Ajf%FtA?247tJ{>n5^iQ~5%s4CD)atfw%O;|w$83}fUB5h@pXLxfs|nvDFBD_4iXK>jF3t&-!ALfLsD$Rmt@6NmgslTh`LOCn_TAab0t0M)Jm'
    'J{gESa#4;;UZP$@4v9wou>Xtv(F%FvHL4}@#*YjtZ?tJ3ztS1`qI%tbmM?r!mF4`7!*Y3&wWyy__aaYNG{6&kP`!{RZlm&$Cw`(H{5R!6=Ad3i9WBQbdr^as'
    'Bl^kt7W+^+$Pw<y5id|p{*dSJMQw&0F&Q}m>wW6-8n;k=vF@ko-{c5c-EkT&zs6-QazuONheRqz@bnwVV{AuVAjc0~>*oi?8B}-l9mdM}4$tMd0sRI>GvEc5'
    '0)0M$myOHwuugdn))%P(FR&Exyy*2<PvkA<D`WvD2oK|VNm^LHvj)`w>vk_7AKXJe82N~jUntp#^{E{pzYy<;bsC~C|K)>!^h2APw_uG$MNSaP`3*ee1byTK'
    '2J(Ry@_{$<f%YHxK;}CXV}8TG0X`^2)s|gfN{$bFkq^9(3k;D1O60hp0J$K$0WR2#T7X=jj69Go=TnIO%|3;^x}z4#mXQMv%kh9as;q8QAjbo@P-Bq?bdUqa'
    'pth6qCt{HcI{$$O=An*}^C1f5c;Fjq3GzT^<N;N=`ihGGu<sz_f~qVk7aWn}0<N6jAj^AP_=DfjA9-LeYER^WXQ<i81(E+E7s&L0Tz+F8swHwkG3p}Z0~a|y'
    '_>6i*&S#kM2cJQvZ&0!Rfp?$E1-w(p1zyMn@Amy`T(A<COHsYhXE=-+gg(PX)Oh5B-VNj{ZlJED@&Rvf1AOpMR@Z>O!p~z=UqPl5@Omm%=qqHPudssNS75C~'
    '9YW1R6fVO0<8?(8KY<rX<%7Z@cwH3PdCT*uK7yz{o;R$7=kYMHuJ$TkPptbB;E4B-XVAWh=bt12A6Qm@{x2W=_d)Y8MlL{~fq{7no*XyG>LrV#==}xmpU&q*'
    'e$Ym4D4bP4H^?*@{RMC2hjcl9$VPsMM}CNs^A+ro8=~a6p%B#!xj_%P!G0;#S1?Czi2V<9g9~aEs_Z%!j1($2>_WAd<AoB`DC7k*<b-%xorWAQq^JEg{~!}}'
    'jeNa@T)x6t&NtYYN#%s;|6xvWLKUL=ARi>laYC>hC(K4ZC~trhgmV5t8EO>r!6(!+|0e%n1*#%)ftQ?rkcH}jd~j7R9}y>)k0?Q1BUdl^Cr)@l<pfzif-xRB'
    '0qY!i?U4@(kryWZi**jCksAh6>l=8E$Oi{e6EF|q+<=ep2$we^Cvee6kgsEi#{9!-%s>2SfDi1@Kk!2?xP^6%r~beP2e59T2>IYV<{P4r3+`gR;pTa&Z{PrR'
    '2s}l+eo1RQUl`B#Vc_{Zh2eO<&JR34cPaP=yh-32SZ3q(I<MjNESlkUu=C(NjeZ@K98>#ZDB$pQ;m$zph{y=G>_$5c-xDyU4jVMc_s{W^*ei1X)bZcXM*d43'
    'C}S6UZlAb7zMk~1o|3zPO7FB326u*&TSEmmg+;%kjjC+@F3qbwq}j2WNIfOHUu84dAB&8Z-c{63z_<R(_gboJD&T95iso><>Xv`q0t;@~C%2kLq&p-jjp=1a'
    'xkZw=5!^+TuM!j)7ZEGqH<R6JEi5=*{8bTglYiaoNWj;T?XJ*Bc5Ea&oD&i&xviE|RHCJ~x#KH`282m&eF!%U<Tnk&+XPDPpGLZIl7L&tR}Tz}jp4*fzbjU<'
    '&J&Hq8;fh{##khn*iP~^#Z6WGw7ukMfSZQ+sWE&&PyDol<Y}>+q4=o@eAY<(v?Dz2Abx5JpY0@m+6i9mA%5B!npugTc7dnX;{8}CRvYn4X7EAVMS?gtd+|Qj'
    'Cf_&e_xqDbu3i7@o_X-7WhzKs(3fybC(BE36>!j20;}}yldfCu-yl8=5D&ld?IbVgN)DJa2fX^v;jt8cwJTHUX70{;K&13xFto_cB&?x?9p83J%#Aj~I4tQO'
    '9st&;*8Jp6lNK|*HRab+#8glgOH;`~;PT_tnZC{}=~uy8HD&7FaCA6HbP)etWokP@K}C|olngwhC#S^q8px7<0RH-e*-M9%NAronsx&xw$Umn|<`5f}N?kKX'
    'WrexI*u|@v_Z4W`wN2Qy=$Wu5x!s#~dG}ovMa<sfb#E^BaRtdbs7d3C2J6viI~M7Od|ck5?<n(4%wdW&-`BGHU4{IdcI5BdcOWV9yYmd9f)00^Zd!QCZ=xQ#'
    '-SLA?Cc9W=L?3hc?J3^xlEF(`!e_I-%+u>J2KEE&cQ-|wP~EHLgmO3PS4i&xrXhZ!DT%+T=USgX<_f>BDWo8#ioTaWT1i+pqg&AquTL*3qx?RfvO#a|wXPtp'
    'wUhlNd`RxKBI$UNja+#2Ay=uWue2S*7OVk>FCndi1cbRV&DVAAWqWhIz9p;4ukU*tp5DP>-)3TK8=7{V+GbolmuW+8=bP1TH__}SvzMaW@gS*LZ6WcpBtF!O'
    'L$+lxy<od6fKZ>}JoNj6-OR*4Pg<%7(sucBxhXID!$jsDg~qkJzN_uplNs-rf#TOxs2^i0=!vDZWdEIhb@796ARM>QA;+F?=|AG=OXeXZ>K~WfTTP**B<Cjn'
    'H~68go}KvklYXWxAUOBV1^2GonZZglYk2-@W!KkDVY}dZzA`#lHmBnTVl>?^`!?J0@LcmROj0{jPrg;$u6*Wpy6w`hQ*es+s2<Yw#Lv|>z5mwQtqIGosjR2}'
    '8ebR&5|36gzO-q7JV@CU#zAOEEG#l%nN%|qFy_}iB<JwA!OhTjZHc)D+a9*3DRhhPt4<K^g-q+evfoMT88f@jqzSbt%pr|zx95`S-Gy^cGRvFLv?%<z=uFq<'
    '*3$LFcBy!WD+a%{<o=4BU}=<<+!v7z(ek$l&yd8(-(X4peuJs~WPevLI$+x0{G+7%d!{<z--`<j@Dohb6|X-0fr+gYelvmUAY`JU<jubwPZQGDEgrLUBRQT9'
    'a<*IWROJuxR899zcmKJw$no^_e2tqEJ?}RhPbFjP>^-3|Kd%On<0<D*)p(uqL}@&wjE|CgZ^`1Ro}1C*u1qI5E?Y_8?wGKq*@!K-nH!bm;#8p@t1g~4V`b0m'
    'P}vv8D?91ifyMJJKe5&^7t$Ti>XCsrvaXnt$8n7t+=_fYpD?03E=~2z=il4ik=*V`YP`!=!y@8<Dum8o0Vj%87f(B%WppRB2g7=HfT_Yhzws}+tRaHw#{Xz%'
    'L9lDrPUP>qbo|`D6$?)>N6594-rG_g4{|TJjTZ!=EQmJ%7B`Cb9^LSE12cv0_Z8@VO3UflksLQxKRnidVIq$>K^OOH_PAI6xM)gl_i^;cKKC-8)E~b!G2RIE'
    '6j!tOqsZ}FOTQEQb>{}+w=AAAHs1414jW32-$CWNKl``;L;U7t`*zvgt37#pUb}M1XhZOGY5b-jnB?9k62_3kMq}UYT`!Eo_znH+?uvbGt7okwM0EX3H6r$z'
    '&ruWdb}2K+8hyUyX5u|v{O(=8bLECqD{{LzMOq)McCV1e?*<=9J>BznR9chc)TY&0Z3g$MlE%{}4aHOK>pvfUzrZBN)AZhBPWim6U|x~?eUf{Y*2U9dYh3O7'
    '4(SK`4}MTZRDSk$w{6TQy6y6orC~lV&XT;IdR+Te<Kof@HeJNfU()e|qw_a%JZ&+@$lzhCJmxC8ejT}ZXi{OHHsp9p)W&FpWGPAGDP>%e+_TRBe@>xM(BjHw'
    '&M*#|NZ(%AvFyjLBkP!h{z`l!#i?1}h9C6ks?2KBw4VLY7UMVX`1)SqWxdGzq2Sc$2`4g+F$XI*E}jMzrq>9KPcm;RHm+T&?6{lQ=l8f&f0R>}#<#k5{r_q^'
    'ia+r`9v>z9cl(uW(X9hn<oLL!L&AW9-q#zBkCMKxZRlEK9y6UBA6w4$K9|F~M?c?mReRRYAv};AAA4x9UB7rQYIwdWi60gvn`Zh13?$Dt`SxXD#ZL+wo^MJ9'
    'vrn7>VQaSzAkQ}!B-LDuURENVZ&LdGuGK7sCODk>xSZt8?qdz(T~8<hsP-AFc`AfhOxF+lv`_r8SI3MzZngGHizs5OA&%0G2j1Z+Y35z^$?g2}&WGIUoJv2x'
    '^+*XA6uY%Mc|70WqvzyDZ_}jn+Xf$A9c_&LT&>9CPj7Ag>r=izZg{>aiC6Eug38{gk0i&dkaZ6=4Cj{7&o@OXSH_$*bs@*ARqN-wmIbHK$1CAEo96u%QQ}qm'
    '<fZHb`OoO*oBrKujt{r#L!NIw?z(2iTi#0g`R3!>rMGYUQ|6l{okJC~W>!e&o4<`)viNAW=}zH8zOA&K?5H7pjUHNVCLS8l{XPS=$9Ft>vYP(-zKnN0N=@es'
    'W){<pSYG9~{QcjwC%4nybGn7*mTSMwZzbpYDM*-B66?B&p6z>e2I3>*5Lmlbt;}oM3Yl!Wex{>b<a;>6n7ki^#a$0COj}CaqFc|46&lZ-hcqR(o2Iy9jCRc%'
    'X*~Wdu1aF6WP3##ws+Qf(-+1Gh^y<poIQ?DsbYRou4i7{8hPKl`}Zpzv#iMD#HzVYBOgX?VFoDug*ZSu9uB2ES{P!pkC~>>xOR`%Z+{Y^mqbjWYq4*WUFf?q'
    'RdPIbjv+=WFHxvJ9@oXkA4?p%jCCZ(W4-amTs%tFHa;GEn3?Q5ca+(tDNPH}ll<YccUw~CoBx#fDCtj5Ur*k?J83OBKHl>TPnbL9TEp>CG9H{>-8{<S<TP@8'
    'oU-EZw>1uTrSXw6pOc)sCX0_zc?&Of@(zH0KMdYhTx9pWY6s!eLatv(?y)SJKVEIqIq$u%C-i%mU)}As)OF9C>&(SX>hU>c^T#bgiTc@-tjRv-iGEo<hum4s'
    'Orskolb$6$wA)6Rmzic>XRWQ8LVTb*UIhArj1IFY<N2Dz@1O0iUnP#HQ$NZ|ZQ@ure|)(%(<Jd7`ms8I!g@`sp0AH54Cwmr0l`Km@Aee`_vW@4_a2R3Pi&+c'
    '4|qnzH~s!?$?bG{wQslC<_Y0pR?qk&^J7CcTs(j2=ScGSbFI8+)OTSSajdi4@7u57D2<Pjo#S=mt{?N(5cqvCdTR#SIiFI!N3?12o8JZVN6E*=Uw2S#al{tl'
    '9DHA&Wlo(rXI2yD+BCn<(qidJ%bVTFaZY>P$&=&HE@4*EjUcnrqxz{YXh)gP<;)pza?>VaQ%jn5@jF-V<40PMw`cmTtO=L8XE2lL`pZ*;7M)5jQRYd_quv;<'
    'DtaZ2ACx$E@Bm9e2j_ErhmYNT^G<I#j?RH4d^comru$oFVSVvK7w2;>N{^2YDYql<zunAygDAHOrjuNMVVNpBO6QM~-yLybaDj)T4eUSI@Y#D3n)kiGjk%s~'
    '{$cAnX`kaZN?eV<dLaJoEkEW~y8U2iv8-nP6diKABFEc73ZeU$GwHUcsM0jq-o%i+J&U&njy%0)6muip_uY5lb>^L;%pd*MxsCHX^pi>J$MQ~>KPc}#fb7Rs'
    'J=VX;8u41{$5IYmy$vh{nI9Xj&c1Q|rWNeJUJ#FsU$skFyO8P8NMCjE%W3_F<Q-y~H?rN!8R_%Sx*VqWV_9w=oCn?P^3RHoB8ZQYW9@Cvh&Sooum}e_`k1#b'
    'G~Z3VN3^o5XWWv-$GfTX+8a5HhkgO=vP@nMTi@vs@zffo40R7lt&S`{YHVLx@p{q-=ojHI#D|U!8RdV4SgrHhJn+~2v0LJW%X#UP<65i#l1Xn2t`Vo@<LcVO'
    'zx>r}4lLc7yD%RF@%X@z+wXS|PasCC(e#UqytaNrMq84P&of++7I()3B2q5T$+P;zQfP|vRrBLbqF%Sa@f^Ou$#trx+r(uAM}^L}9D5`Er&0&<xI8k3XYOpg'
    'nNX*Tb62L!a%lcUjofa5Q@UosfHlnPbVrKg@rcMJgbsNeDI-{3H+z;7p2qdWxn}5(=Z5gg?nkj;KfvhyXzkheV-KehHkx$tSZUxqW3R5{?+gF<)^A{B1<_WQ'
    'y8k5iAeY8R$uL}A+sb|0K$zz;MAG+NPc_|Oxa$?6tqEPK?jgB{k!=20GOS%&wNxwE4<o_fc;&iKZS;EL$6uL8khbUZS;w^ov9=RBYK?1mQgMC$g==exW_06E'
    'Y@hvGvLY#*Qth?8WlEWVX+^i5$!d<aZ`WIp$E|k?{dWH>de7{uT@MfI;xz0%%kJz{n^-c}o-<eI{rSja<^xsv<GmLARY@#}&2qV4%NYvCEqp3me|lD*xi^@O'
    'P2`UEZ(CVXe^tWYhTRf-POR<={Ut^Ewv9ov!ishGnA25cBq;Mxj<BTjM@jrQj*C9!r-;YxeF$M^J`A>9ox$uucO3ordG)|U7KLkLd8+yEzHK7@Qrsc66IpF-'
    'ary2VqKK}=?t#uN6cn_`{aQWmjDELOJDHhu+)$gcasQ~6#^iXM`ef>w>($@tkH@n4V_|rjmO%;TIicaS37;ptX}v`nj~je^|1c}>%qdIq_wgQ^@s(;0{vkf*'
    '9zLqotqV)(9~vJ|uFTmuV}2?*K28-Xm6k@|Z8$zk#vjiY;|g6rd6VPg_e(t$l$t$kI6g|^$F5zQLU=yI$?<VZm7Dbowae1@_#5}i=8xy&mK|KZw--4+j$N5o'
    'V&Cp6@wSEB{83UnEsKvwZGLRnxxyBXGw^fYM=tDHsm3La)Aft&!Zm8W+flezv}@S9lYRoiip~$@4}US-?JT7o(Wz;{%Jx~zY`Xrdvpd9a#ClEgan{br!?tpH'
    'S^e=!4S7Sm;KIeKnM`uL;ymy>w)sgaeZ2BAzWQ!fHYHxIy06mt_}S<6$E#-e=MDEn-A&wWNsd=)cRJP>C^ZnTWd5q<_1>%cno;7Fd5d0w{e0HZ$17EpUY_@I'
    'DEUsOmZO|59nYqZSN-O0*b?xS;#Ws!ShZI#TSFhODh^nBZhuONR}P_}ty?6OO5+t}$|||GfXrXbTXu%w<_YnN0fnI^AAStKIA$yJeIvR4(GwMus6BI;%tq#g'
    'G^WIx+C;pgkB_QW*{o?6Q^@hL&ED&i&t;0}<74&&<t}BS!Q}WDv~Kl`jl-_f$H&VjroLWcZBLGmRdKD(`@FqFA0O3L?`o=1VNH&Y8L9!D_kCYYA0LbNbU8Nm'
    '3#D#g_og9-b;24SA5T}OZ2eM^K_4IWVsv+RN}$BY?t8lsuNG~mkB`C5U0xGm?a6WHOSAXAZfAXA{^qYre6+gwIBZK+peFhT;G-22yWeXqOJO=TvY*+;jJ-VT'
    'W<Jx9&YxmkPG0rtjrl(^K1z=5v)@D(w!K!7<Kx2T0fsB5-l30==Fbh>f}e~b$H%MLQ%8RPbe}#x@^j{`8Z_FA93Kz9z1GH&xJVx#868K@8neiT93S(h9c$~$'
    'DWQ*#j4xBCj{Iy*j*re8$D6o{Lg?e8+L-bYeL7L%oYLJD-Djt+qK}Wo<MzI4`Hj-9^=AGB1>JS@@i8a4Tkx%}l=#T*aU}8hyC?MVF?Y(X{ClPY$nnwolFE1O'
    '7JrD3+D5N-&HhP=j}DdIJBvE}AwE7)^5qR`M~RP1SfghY`Mso%k0Jw0gZ71%<oKvLVy@eOpiKJs7`!6)iJ1nauE!x^{);`VKg7qKNoqsFE*Oy8X-=CEr}8<2'
    'K0a0rR~TSE-jI^-9Dl0laLh{j`P}To53>?-&B^n*7s64QWBXOrAFo<q{7A~bywp5(GC5uq&N!sArrV?X<CV-`t<-9p@AQ5kIbLae(2Zo-T&q7`$^6yI$ZH#w'
    'kM|<SE6blx4qo*ytv_DL{MC&Q4ywLXwIau>+*dm~o}ZsaAFsMt?0aK*m=dq%_0~vvTDq~(@oLoGgQaQT)9K@t>f9ropejndO165Ukh*^xeY~1j<2U5-$#&#8'
    '@I=+FcDAtg_jskZfmP?P`fs2A>G~lXvQIkejQPs(jxU*881>8pW&Ub%$n_P06%cpu3%4Y#j$e5_=CApyq~AAQt((pJ^Lv@zjcm8OYN4X%!aQaHZN00<%+zJp'
    'X8+jvV{79O(Kp#ime7HgrT(yfI=56BAC>EikCKD$?y`G5i{eI;<D+0~l+nw=dxW`-Tz!t@+)0_gS{r6b=#-<s%8|a?H)-3K_uq<$c{=j>PLjWRM&+K3`%njR'
    'eB7{b>JGz07l{`w<okN*ulT4kF4-Vj-2skkXs5mOOpV>TrOcsp$BUQqz?jwxyO59T6P&K!UpK!_2%F0tFOvErnZN2B-Nh|&64v!}lD<7ugVA1j`)I<NF5mfi'
    '**EW^y~gD6d}ofLvh`X45kVJ^3l`KAg=kQG+3TKF7qikgF*9h#k=Ip@(>+bd<H!W979H~UJRx=%*HdRB^H;r2SUAnzY76@Twt{eXKtC0i6k<8uakg1&p8B)>'
    'dgSkOGuKyau}Mf6($#5RI-cD7XALEezjN+5;Th`=^JRVKS!m&WZoRSg>mB;Y*>Il4v;z@+UzNRL_EPwrrxcnaFHY>W!8*tme;@EvzteY)oSU$USwOcvjqYvt'
    '?T}?m*`6)0D4ROwF=x@W8{Mi$datQ%$?Zl}W_Q=z>BG#Y%d?&6t+pdeRi8X=9iQz}Ghl2b{rpkmvzp_ZBVputoZQWe6F>L?{rpjT**lYUGg#z!yraVOMTc%z'
    'e~-tf*OL5I$GBDFwj$St!hYCLc4Y9y+svVK^T*>)UWg)o_8`aOrMK@?zj%<!{Gr^qxSGDbxT@*L4TM`G<FN~W+@$5BR}r;zEi8^)eV|0>lJ{S={^zvT+FR-4'
    '@qi|CoVQw0;&FD>v4a**Yv|{XqWSjK#;2^v^T*AP?x?m@Sk~zIV|(wWr;nW8_lNmo>El_ZC%XS*<D=^0$CkTqE+@ywe2cxEUkYwZ<Ku5}P8J`{I(lujzA+Mx'
    'Gl+9dwFh3zh$4s>-Fo7xEI#_#o3$!mH<(<X{Hdx_?2XE^^zm_Afc*iZN%rLUIN@^ddyz9P(8tHwBlmY1lx|OskFAZmY;D(M34MILTy^taj2|UFj?W$W?3MF1'
    '`uI3vr(eHph7~zJo($C?N>`7fkB^&b@>`kYQ{p4TZ?MU?tE=eaqxay4uS%UM@$uE#EL)b-M*8@uytUh^<DWW^kL%~oq4W9OUNYY?<jzC6vG`YfT=1o|$<d{7'
    'yf7@JZ~I=+SIr!}l=w^Md6B;F^?Bi-Mc;NZiAJ_txcpwzCFvq!lSw`OOcozE3cXUgM>&!E+2>Vrzhty(ApXnZqgNpJ#J+7d<oKUcedE#7v}F4DzcTsCmUYi4'
    '@&D1qQB%H~{vrNruvWY7S=f!-uA66t`;DP9nV;$6WAFFM8FNZHkjJg)Dc&DU!(TTXA0_ds_Fm$~dnqC0_{bTntCO(so-{r-cz(yN6Uu#$<P0Eh&+5en7MUK`'
    'nIoIZ@y3X3R^9m>;d;s|3zlL%=}thN=_8(vzawB)(9Ju?s~ETk<#r}-mrL!cd$;wSOH86Y|7c`@nt5IuN?v{07{8zv+nGz~+O0TN5lB4jOo>-c#&gOhe59|>'
    '@i$yK=~D|!az3!*`qxeTT%ORMFS#rKXs5I)I6o85m%LD{#96y!AG5K?k;RF9dJ$z)h#`$^w<F}znKAqK(&ry}LrorDQt$kajE|DO-!XWRn&-Zy<oGBgcE9hM'
    'bF1O_DB0=Zy<Tm9G;0JoKE4^$<=lIf<I?!}n@=i>kGt+H8&)%F5cxP8!<^O5^2`}xp-w$<PO?8Ge@uDT*+s?Hl>B4MnhwXl9b3!{qKk9pDF=pocIrVsUM|n!'
    'cKBp=nK03oi*tPOW@v)J;a10HM;C6xdc7{vw*&63DVTXHgSlP#w>VDJ)#uzT86MqX0>(%9!qJ0IxVp{=AY$m^<DIMT``cfoo>zF|(mPRVF4OL>^m|gfz{?v}'
    '&UutVB+$iG!^#mCQ^F|mzeMxtr)IjF>FYZyc^(m=)s*8!$MtM^6ZHn-l_aK0#^!MIFIl@)DEmD&`9S`;xsQL3S5}``3|aju??J)f9!1u$-yv#`8`k#@Uu7ln'
    'l<s;ECdX<{Y*(SokJVz=^NgC56K7ge_j}Z^Z!mw9^uMZZE6nxJTaf218+v{>RJ!quSS{aQ%69+CKb9z~tUniI5BnWRzD1&2Rg_Z-kwoXqZuNXxTM<dw?;7t$'
    'Or7&%Gtr#RA7`9Buzz5y*5r2UBE8x#I+RLS(9H*Ox3_iPe2?P)zc<|CGU3d0`gm-uaGqI2P~x#(>CHQydtTNbk6Yq=mAfcEbpD4Ray%YruxGDb`5kFI{;fV<'
    'c78|6MDLzlo&$Nl8q(zD^-cRuGFK?qQ?D*Nzr)>i%h2;<@O(>nyxEtyq*MPi=3csS;`EOTy9`|o$o(aBR%zR|y^@KCblc_RWL@<MU0ajewYc_K>*{hL)06Ia'
    '&rRzijL0!1Z_gi_oadcVu9n8*-^Neb{85;5^5GRH3-WQ4*WTMSO=s6{=R5yt{y3@X>zUT)_c5C(HLl%1YW|qEXZYADFOtadF>v16HBluu>GO|{m!EibiXKk('
    'S65#Oc->3;PQ&>}$+2wabYxzOy{_c^<0-2rn&*}mN%N1|@_9sAd<@VH=FhQ44jl~Z86Lcz^Tmqb(~SqKZ)fJb?P^NyzuT@au-o-u1<{NyevEmO^QfXfWqdW9'
    'e&1L9gsA>}U{j3${)elDD>gZh^MO51Ypf*fZvCDQ{QQ)~ke%O=n|HL;@cvz3T!xNbq9UpYpE8blnC|#&&F>%HEKHv~-Z5$qyKcWLppTE0`xVBIWOXLDdvxy0'
    'wG$u9>GO|;!`n4s_NL??_0z+S75G-r=N}8#zhH;2pyVItEyzuH`yib@|7g3XvP;|@O8m%tvC!MUWGj9C@ng@UDH_AtlG`<JTYa`^n>70T<Bh<16_*1j*Y^pD'
    '@?}Tp-1t5JDB(9*eEc${XCI~4oyp^ty@P8`fA4L~#f|j6`c8LH{n>FoGrN)XIb$RH`|fT1f`0xuaB$wI9|ZyAcoi@+ww*;_DgFGhu(E&GMKOKIzwg}_zZJSK'
    'O6cd0eXf7nQo6SXdHy&fX<+`*i7V;nkDd0f>pXmXC-U((ZL46^;K_fOKlZ6f`krB<Pj08VVg7Z?aa-x<kCkh;tGDk-;l9w+{7U6Z@0qmM2ViyZytijY57;hn'
    'KH{A%O$Mb#&HZcVBa*&vSmrM0)m|T%wATmlZ(AK}G}VkePniC73!`GvPWtNu#B;58G>@hD_~8cWroNi{=&uhTShaWbwh~Iae^h)dIJGRqBXSWrK58;wmG`T?'
    'A&rl}ag6Nzj*$t=cN+zHk>lfu)n6SgQg0KNY-rAZ6yD!g;g;e-KL4>RccV`mk3&SQ-1Tz`%dWEO{MA-%So}vJSSKDUeLuoOX=d&Dbxbq$dg8e3`T%yEr>y2B'
    'R%Cy5=;NkS<L?WJNosQW4{q|W>jNak4Qn+p(HzF}DCyh#92bnt7eq3f(8c4)9A}?ywv_RFakx^ih?6S`4qZR|^7&+&ZibY4%NJ+$Hce}>k$H^n2<sGndgqYm'
    'Ey?5gmTRZRUF5DNO6cOwYm;gG(}omY-apv?NzT59#LZ4}pFQ^Wm%my(aFz9uvO%yP;3(cdt^L9KD)Wdwbo064V-lDXd?=jqENXMgUY{pKrcpiPf$aPatxvC%'
    '9=KSO_usYo<`0`{l@YfLsr}`mmZqbE?EH=sPlY`fnw!D?s{u}#mw$1WD2p&^+qn6-lP5djmik=ckV50yE!16;nDuo5^DJG9y%CJgRvDCd{CT2H;+;!r^!|U^'
    '>CyTI{fx==K&#t6*neQtbNc+F=|z`8J#*OP_&7L&uleQkE&BXp@Utybzn$+*p3m`ocI~~9c#gh)HFv#3?bE!T<a!|UYM=QlFI=UsUv&-tx?^#VZsd7<n-Zh2'
    '&PS8!>w!$?9uE|RQ?6gd=9s2zaA|x!(7IKJ53KvVn!X-rs8OQn=z!+re*M~Zg~7n%8|d?oQ`uWqBz>ivZ)p%aBh)PaUH$byviQhYpfn+*XLoWvkkDSszqaFA'
    'X+2PbkC&A%0}p<p%vZ12tj(FSc{hE14)5iq+}N|7{;~dQ?EcF2M4tuZ_&D7?y7#rtH|YJ<!jUsOP1YDj_E+zI-d4kTQ9z%69G3E`yI&>cI#p*|^h#Uu@+5u!'
    'QSZ%?OT~ed`qenclY#Hr718G(Ez0J7F)+3vkFTopcAYvJpGZIcoJ@Ehu|JY>oV_$~8TlsYiZuU7nVMAze&rua`<L%uupO>v!+;%{_&#>|{f9BcF1ol{J}#}@'
    'a8Va>d}Q5tzc|lMKvdJUJGU*_<;r6ta=YhgQ<;Zch4lHyz3X1_?u|AgAJ?A`M;X02x<Q(MZ1AC7lQ(o=IpuiqY3;=GZ1#ZOU+wgwU=wE<i@e`QFdiiuzj*YU'
    'zshQ&=2&-q0Bz3eDkV+ySM{KNrHR$7VN(UfQeB$$%)2?RYWyL}{3By*%BCKl%8B-x)a%4qW{JN~=C9_r+uwR*2kbA^;Ljx#U5bA5ocO9PcYV3sUBBj!{!<T3'
    '@7N!%55Rz#LsKWW=`6jKM0>h^wrZ%_oDD2W-kE!EV8O5xTZsE~ykWfTLSnzvCggTi`{H}|>A8R~rTe}ohMbxn#T_Wu_bom&QTfUYdVh8If%8_0MU?aPMzi&l'
    'D?Ogk`>UB5>kSr9@+HS(B~RZ|D!Xpd`>Ptc&O6P*dXeL?%AE5`ISY=``>XRex7na{uQ%CW4fl^77+kZG-e2u;+`H$*X_R?qX3WK)A>IDqujV~><Xd-cMQ*oy'
    '!gF4xPA0v-`Xuk?s1P>AuRdO36w&lR75({-!aRG8Wu9Hh`OaFGmjQz^R?(mTSkHX*n$NaX))#it=O6#c{;H%eZb_VcuADiK93O|OjbZV9uS?_OZ*>f^_^6R~'
    'UL|0<2lRUwz@2w1%q+9tOzhO6iI2Gsle(7$IFQH5@i&<ZY>ply2FYK4)#W(&tCINm;rXa`gE}FHf<HHMq)*jYw{y&=it^{BEo9ZrA1h~AC||gXejr>4IYepv'
    ')`5B==01hrj!%Lin?LST^FDUra1R!VRlQg4UBGy;iLh*BKJe$B>Y>N0gv@!g*Hivd{q?g?XYzPAjWO%&8S5p44PCo--JQ)%9#O7Ko{;r^twzj3VuCtNJFV#N'
    'GcR4C)aRTQHaRpTDwD~j`@X9mm=^0|DdXfhFN5^NNq31!UF7!Pqiw9Z>*uKMU*z$lW(e$exN!0HvqjBEcR5BBs>zLatSnYtd>m0U-qm>{oY%~l4|(T;De?BV'
    'n9PrK$93yhKT1m!DfOg%;~aPT%vw!cpc^NTc5R~T)=Zn+Zpk#)R~i>Y#7?6It}iEv)1}_0*k5Nkk@;<H^63#Tot_Xra`RkP`e&Ac?0V+4C)M|N6heJ2<F@qu'
    '!{NuKInEUlAL!!0&}UR$-#p5B@K?sa@IKQ|K!ni6)xbVa^E_{uQ0lTzU)z?(xXB!$@Y^^+Fl6U<n0#oNl~!&}zE0zyr-qr{-Se5<8#ymKoO3JjVsrwt3tc~3'
    'llZwy=<+t?^&I7D!mVuawElS98si7I>)O~Mt9{7v_~)hlE04t7sNX-<$9kDAuG7y>?P^ONm)bA;*6Qn)3sV31w|TD2U#*$6!7RH5{Z5$YW=`2VyUF_{%nN@d'
    '?vVTE;O}MoJFFp+8d+zv-K1NWr8b-B{o~X(td#Ix6t4B_c)9oHA={<#xZ(L@R7LI!N3(zA^>g&3Fz(CBOyH5@qo0Od{Iv9Izt0~plImCW9Ro8CUw4Ck$d$hR'
    'w9?t}XutvDRzveg$yjMRBx6;Ref`M&uwSSAn3aAfh$r&(Ig)z=%JPpEy6Y~ioMuJl!lXF<o)@b&G8fYIq?S88)$O)Y#!0K6s&69BpC<ftXzFvCd}cC=rdp8e'
    'bL`G4t!zGYFY^Xnea@Dp-xViWSdi;;^wifLIh4|oSV_l|PHv%nr#+<1Bl`NhXFAyKVh(I%JZ`mlRmsDyCB$O6e5dq!VY2I!uU|f1?Lrvl1A#88t3w9VFsBo@'
    '=*HKB2RrW&a431bV;c&rzZ}XS_S5~o<dsuT<u%tOx2wI_L1|Q$kXcRl%Su)6y?R6Scg7Bzkk`rU9>Hp;epM1zI8VO!d&8k{7^^3TGeGqr;VM60t<ZF=JHKO>'
    'd-{@9&9I;8OW!v$G3k1zW)(4|70r4M_@H?(Hksldzk5_*qiFGzxT98&4=jt17OBN~Rz=8f-~;=(=Cfa{*+e+d?FY-t4|{)%rpyC%%MKq3|IQ=U(~Xl>T7q+>'
    '%9M7ic1Dg$cbUqpp=(#9b@`d#t|56pOmMBR_Iy>=@cdB{uQG?PG8O5MCC97s+4GHGcwVpH&uxYAs_&kI6Xxx(CeI%;D`%vQN;@s}bE%gW)MrUOS!r$P`;W1#'
    '19)CxZ{R+4?#rLSrA(#z^2ho(Uv<12RkNL-`1sZBzP+kFlgG4gWc>f=cka&TlYHg{#m2SsOJX*!?ZhDl(B(zO9G_sDJ&989th&DEq;9_J>HS>$3BA($T{9r>'
    'zsL`cXYRh)Mjs!Ir+thZ6Z_AMkCMK4ph8J~_^Das_~^WN;*<M#uhP#S&2KNzGBR@|&mXrfZRTP8D2smnnD=(h<;`Js<Z<iPl#@kgG!E0x9~X5=`=;`|8+ra1'
    '7WHQF2D4)N`Qt`YpX|$iJ;?LNy-7aHSt<{u^T*%vma_R{x0-L)RvMa<ulp$s+@JBb$qM>;plSIlZ@n&*^Vv6UB!ZN)^O%jDcaHsTJHPe3B>MT|Nri0+SNBlz'
    'k6m*zdL9d(Pd|Sgl>H)qW>-r6QT55IQyy>5pr1cZ`I&g5nFi&&trnvL*F|qCrJp~(7M(U3SU8wGe{|zkrc9r4h<^T<b8xf1>xdrY`Q!cZ8DE~A7)n2XbSiu}'
    '^g~HI^6y({=r!B7CY65vSf0^o?nej8{4wOK(dF$M9?{Pqzki%MbMk)5_49qyY%K)NkLl-+-@Xky+j*B6dHy)A+sXHC5n1%}$9l%sbPwgK(!7=Q^T&GHsb2NY'
    'vktsLKY#Ro7t;5`IWzM7F{tU#OJ;HV8$Ex#Imi9m{Of<1Kd$JzcVbmf%KUNS(y^*$cb?GCA8)8FvdnxoiX4wW6!HD0_q$fVpDUX`3KuU{?&D}hj>nZ>)3ZF@'
    '<VpQp`AhxQ%^&-Dr0(yt-<Hhn2QQQ^J7Sc~ETZdYFU`k$ver?~L$O@zGw;!+WTKdEyJ-D5ZFDG`a$U$NZAOnMZk<l==g#K#X+sRA_&%XpCyiKpw$jH(*1U_G'
    '#%BL>=Z{IxdfWItn@*08Q#*&5Wp=#U@cdB{A2(N8<<39vLY_aqd9kqbqlo>|`D26UcbxWn|M|#gdvbj2u<!A}wbzal&*bY@4{rTcpCj~%x*dBBxe%^wJo{cS'
    'am6Z^P?oE^=josNb^TS1g;%dQ^uT%Gc<K9(?`Q><PYh#v(A7US$?+H<ETo+O@cgQ7aC@~9;-K8PEV*w~-Tcw$>h8$2k3Grvjp{knsB5#KQA8+R|2>-2t9RIa'
    '%Jp744KaFV5S&GL(H&>U7%6GF;hN-jDoF=QyhpAkp3t?M6*$Q}@<L1Uab~^Ve1xa6kXc1H(#`fQwvN9+$@33dVXZiC(H-f0j{GQ`&iHkGa{Euws|H75{Uc0l'
    '`?l-QU#N45n5J0|Pu2OW_I&$IODrk%deL(?ggGrIx@gd>r*{Q+^Py#w_?XE(l)ZP!5~dg3_><<nvkP&DavtOM=BgLeu84?pder=5c?iR?E<W}sR}9(l75)Em'
    '>H80vR?ORFWkkp3_2eyO@sXj~cHq-~c>guXTP~f{?p23V3kYYrdBmZ1rmUR=rLJe@u6upo_1;80psQOpRyb4@$Z18%Kb~-kvq|F<Tj}av=D5roe<_&4QymQZ'
    'Y7uuHH#~oo#2$OI1z%cz7(tF#wyVB|{&cunzuzgV&)LHp-Q=FLC3&2f*v#S3CB1o4zq7$dWx~--1J+XfhBqERb}!s1r1u|tX6l^ivVhXB=j%VM`1$N*{pVZA'
    '&VNkSy8UI&12gh^mc4Uo7rfw^^n8oo>TG1^KUT3;jU(Q|bxpC(=If=Qy`HHAF~b|_uj)JctRH+}6Z2Uk?@z)TGT(W6!VdcLJ93*5mwebJ|JeGRk^6eLcNrN^'
    'j*mZ-W3P;iE@}Au4#{|+*?f7wExAL;^{XZx?xiohuS?JGpp?v1tY_8vtHuutMkK7o^WdQJ-Mwf^)&`w3#Cn5zju&N&$IpdRGVS9j_pQ-O4|*`W<Q(xv?tHew'
    '<U**=kqk;D8%{fn`-1+}7|81#oi<~c>Tcq^y8OIl@vr*TCs%Idn#}74an2L+5oQ)s!X`yB6Y1tHp<PNRwLM9hKVHh*I^p1{WyB!5_&7Cu>GSS26n}LZr%3zB'
    'keftDd7pH@f}?`$`f`c|OS5Kvh5J4u&2XpGUYT=uAu)w6{@?W|{&IRS<-UVsYPDY5Zr(ubqdQ(cn{Zwzq-j%p;9+N4wcVP`+)Fq9<eWHJ+xH=*{=qjrwJ0|J'
    '0Wrj&9-l%MA9GC&J9zasC!eqH&_>T$PmM>6r(4g69!1j=S5W4G!@nkOj+rMSUbn1gJ}0Zsd9Qq~V412Vc|OPdbhTjl_cCIhmRy_*+)FzD(W&>l_n~&MAK*;w'
    '_Fp@8Z>zI}m`t~xXK$bQv`W~Lygdh3MmNnJv59C&7v~PmT<$ZNNjd+qT4U_JsYxq{iL`#%Q58epa!UU3WAcxSM`!Zsub-FuBDG+;4W(Yz#Yo{((kl^teU5k1'
    'so3xRhm+%Dmt(ax0j?$W_^aais;oYz;PuE)U%&St*P(1POyXVC+R5y#3`0rXV`eg|u0AKd?V^|OW3i5}Kg_R03kUUTwwPH>$BVO7Zdlw^qs-^lIc@vYVN)70'
    'h^}A1x6rjXXhY#J<IvsDZx2poZlyckkMldv)=H*afA#RKR+jE5FX*qI!`uAo@Q8U`$k)&5zkJ?T+i_7uW3QiMmTaQq&~pp@^>hB|^*Q~1`feSmFqIr1jf>So'
    '6P923eg61tGpp|WM|FiAIT~*VL%)Z*Z9PHyrRFz}5f=<;@{ip%8?0q@qg;oBckkHrQ)^BW-fbJGUzLm#uM87P-vm(VS9KJR*$;kDL~L(UPu@~Ce-!jh7&h-`'
    'PxAe(JbEsP-JF%hTt?R;6t9a;-BvUrk0X~8jLL3?otIt@rNR1D{)|oW?^#{R^{Xoe=(+P+a)};v@hb1uF}KdgDDmo6VvED?u3aPE%9%UT`-RBj<L1!5(XSXi'
    ';l6|L1%}m%2Of!d#D2Q(8+oU=w97<8^8EvAdOMHJH^?CR(H$>_S&917O*)d>G5Y`5NyM&W=Fn}=sh;BtnmwS5KP`hcyLldZ#Eg-@-kXT`%U?|%_@;Dcpc(nR'
    '*Yb*;LBZSRGAro5kME-Fe<YPsSNMYwX)Rp7ow<~*UBAr?^Ze(Oy55>8F@cBP-X+`&8t_*ovEX4+rT&hWl)S;2u)go7PkTVv=u^K!w4){DA0>RsyJ9yed>fp1'
    '$LJ`1zgdTV+gtp&LzL-K<MEFXx{lK8uS)ndXGu}5Ko8Gb`U>k=DadP`oJt&Oq%Yf3TXAMftG&!_jr2(wVdwG=XKp0C=;DA-E7#!ELCSo5h25gCjr;gaeY&`_'
    'dXlzMt9~sh^|G7u=MK~mHSDiS#sOcu5ch%HVdQumc+>cD_La*G*XKy$@m<aAahaF9llR}U4ZX))DVjq+Up3fv<!agj%6zq(!j<i#2Cb)`uU>e5@AmOll=-TK'
    'Ue1pXUKP^#*x>n(C6}Mf9q^NKo^9OLc>T2ALgqBO{h)0YbTLhv@_l;4N8IW(Ih*;6_B!hOUtU`hX7G=_zTDM~_wRhHh#|+vlV96>zBTA_!}&+aF*2rgEYWY9'
    '6FL8Qe`&bp^J&S_{G(ER@lmq>OdM8z?{Jgin=qg69o;4^iZG|UUWm!Kl$X4F9m)GCR%OQ33HhnaIkfq}Vsr1P5tKUW9bxBg)db$GKVK%R&#~P9Kxu1|BROBD'
    'd_=Pa<I?{6^JTL5n5%gA=4OvB<a}AltM(;_E{~+omzl7-={m6~=i#<^xTke(#47rHS>`t1@-!~xddyt|$9|n$bFcn<nL5sQRyB>fGUmbnGH>)w-qJGd!b54k'
    'jGVgMd{)J=?)m@^RBo==`5mug0C%Q%Ua*MCP8Iz>_P#uzrtW*cIhtq9nkQ086mg1ZFl8>H6dDYT$XJ?Gng@;2KoU&~DcTjyB_hovR5Fh#;rG4w+^c=A+kW-$'
    '$FD!%Yrps0bM{%!e%AA>wLl#W4=m5)GGwIo2e8@E(D+h~l6Rg){=7Io<d<YXENDi}ab%gJWYKYqz^jv5HN2mjrh`$G&gQ)<6j(kMBJktV&xM~7%RmIp{p|fA'
    'IquKld7q<QDX0X`#l9KxV)%ip4W@{C4xhq;1{_rhe2&LBO>1Ob7J5#u_d}t+j<6vNT4$1R(92?N7+DXT@b9x=kFi=W!4Jg0r``3I`(t!}fSYw*DkG^YVEk~C'
    'tI%Jj*feqdBk6Iq-;gOdYM@4_e+=X1&|i3S;`&F@`Fpg_a+mTmWkNsnH|9QrtUl!W#|i&hWZXCNwiABe>leF2uZjOhfARwlj>qh8A-oU1O!{^<ALG#em7D7e'
    'zC3a#^s{}XcViHfADFm5ISG%w`Ie1HM=8Sfi1#8Jy=@)@pz9IO8!JjKk0QjS=CMX2i8F)bdPKta1!j!%$M@SqxduFOJZ8Y1LwT20{i;qb!WK+2FI&*L>4N8*'
    'bnMt9eYXD;f0R&liao*V2IFI#q~4qs=B7XSWAAx`-~9nZpS6rEwq1h1XVK)3nS%r#AIC}oCN%+nB>m4`PqTkBvX0P)<Cs>!;pk_TpjzTj9g%^`ABzQla2c+q'
    '<X3SWagqAd=-`(@z?bXKzB|ADIR@ch8@kyjzI2>_6b|n6xZ+faIkM5=oz(p5`XAW``~4K*`i_TpQgiu=F8*DAeJI{7=VZtoyzh=lqV^%1hS9I#UxfYuHTEHb'
    'ou#!voQ)RezJcHKJIor7ix}o9z}I=~Ikr>EG!>|!;;Z?P!(6?aguLG2m6C-`jc(vQ>Kv+8btPogAws{`g$s@qTUp-+OK6JHyamS$e&eI=YMC8}N+|Q6arS*x'
    '%g1irkz?R6%13n6ac3{r6EVU(6m2bT@1_(`hwRrkRWR40jNsF>w%c}ZZc;aPh#p_1;$vOa?HG3kC77@JM?Jo}wdfeS@1D4Yt8<$&fv-kiliI8megVx_Yp2e-'
    '&$NxeSM3vu1MIeRfHqM&=NkbEUp3#|e`02xHldHP?{ek(`(5D9G@6vEv4|lVA4ymzle5a?j-UeE_p{BWJD<BR8Y~f-T%0uSzsubdnGQ}&Qupf3DPQBJ6pRU?'
    '_UChHmd%#C3H&jvDb_?~sGE%cf8=GU{4p?aUty%I89cApBzd;Xf$&T$pI+T0l|Np6eeTktati;aSiCG1tG^F~U@gf0GOnqoNB9Z8-lvDZ+|3M(Lf1{Es_}n1'
    '`ikIlyE}Dfvu8&i7R>qQ`Q|r1dM0$3C!5ZN&naLR(+Z6>QRsZ`+Uj!`lvN4e=WEch(;zt=UH{mtbR~Gxu=GF4A4z*-Uq?z}sJRP_kEQ*u`MsJO{=i4?uq!w|'
    'lHN}b(^)R2B8%~TcmuaYRJnZ9d|wV+6(_()A@Zhc68`Xs&>!IRM>pQU*lMs<p61-70&sjJ;ju}m@5!?lC_b_ExSpLTDw}I`8fbFU)F;V=;P^<o`1#g-zZ@=7'
    '{Hr#2T<&o6Op7bUM%d}(bJ-~I{K+x5wQSdvVgKs0$B9_vqipOLJ2Ag%u%Qa)k0dM*ox)IVrK^hPJDqU;rFv$iq)0KQ!cKc%{txG-P~alWCAqU!P1yjt0S>}L'
    'ib@*gU^Y$sk=e;Vu0M{YRnCikrEWmxQsf334BHY;JT&<g0oMJEzkKTWzJK?_pX;j3Y3vEZ<kmGxN!*Is5Q2S0#Ywh)Ye{!LW%6}W{d!-PI`0d}0vk}@WKg}R'
    'Fhg<(gYvo7(n>b-&u3sqQR|9Z539DM6MRdDi+I<rw`>OUSZVl9Ou;xlsy>J{eyKp|yHkkgfgSpRYVIMRj*35FoXdQZ+X;Od3O0Vr5&4+`I8bp?)YZbF;RQdu'
    'ZfGW_jS^Q2P#2+-SEuGzTg<dWADB|+(%^~oU0W*_oyhbA&B*zr>>45YqlEdEd-7A?&wZDG_B#s;O65PfO_)=)B`^3`glP;WgxY7)RfCUH;|Mw%%aTx+C0n||'
    'c^Y4xjKmbf-~Jp=Yq`BWMijn^cdX!La@hUDHW3V<@=OkGI~zYH6Y`zG7yI`-jdTNLsO#L+kt<~}ld$fYxsiI|fiC)doa)b!iO4H68Z{>5bJgbhPT{U6^YK4?'
    '15|&`G_CA?VF45!5s$J?a$<Lx=S5=QQ2XqRw0fY!2MPE(OTQ0S&*(75UZCE;R3B%nEk_Bz&+}g4=UdIGX#Z+NPyd090*e2ve^sGBf8=V=UKk&r7_I3Z&aD3<'
    'ze?^i#zf&)o3BMIa~3tk_XU0ciRe_*7CP;Y*>g{TkEDP0R{F)TUl76XrN6?#$u>3{NORHXdftZRzv~~T=A6B@*_Lvy$2WqHt#0>QjeJaXl6i%9W&Og~#XaB)'
    'vOlSZ$}Djif?sd9&Cj0OJu#RmD&7fr?oC)+Jq_N6OqT0g7v66Hc`9`J?osj4W=KY?O>+U<ckgC=pwGtoD_}MUjsHlfc3gkd>2MkP^@);ivBT-JS;DC$``2O?'
    '?324M>!&?TS)36CI+-T7t~h*2@|Mzea7>yG-=X%;xzxv@!f|B|oX?#vb1Cr7*G{lZijF^!o09+MH61JEU91MzrPc0s_0Z+*{-eJfxlb&0eg|8y$IXk`lssuK'
    '?t9r>zglhZ3AiXigOA@Jd~6qZ)SuI@j-PL?l0VK_(9G&08w0|45O?I;v+4V|9>~JF?DY-HbG(JpfdncKTrT)c-^*+|yl&cD-z>NK0HBUK*WZgh<ijOM(5q{+'
    '^n*S8PGc^pCztY3wA-MGP&e_wxK6P{FbJE6+Sl__hXq=U3Hq_RV1e{5;ZFK|j_S`@ksJEzhw(C)&#f#wf75+aDw)p_-*<b1-~DsE`XzF#)=>KA;0+I5HMd)7'
    'jNQjB)9%k{Px1Ok(s{psK&N^bQ-$M1s9yI)-S7fTW0G-VN}o%d+$ulp80vi(8-DVcV;7;m+UI1?bdI+XfCaTK#CBz>l0SiWj_oSdZx?@x=8yNJ`NFq9Ak>Av'
    'ou0?DFERqnA3v`6C~&%;fU5=eN1U#GPW_Mh9ld?ezn__I@UQxFSZ<pC&;U*_J`NsKUs-svZesr`Y41DOg$ymWF@XK6{dt9*t26FS>|Z6tmlX-~<)tp@!~WIh'
    'tL`n+VFeTWS4qL_@`k1S1+P`%`1(`ud5TKa?TP)Xr2m=2=0o>0R;j}N)r7@7sxxXgk^QR^{zd+>ykz&3;PdO%*7e$CbDzGSl*%7{uGb|V_1A#?q~0+JDUU6y'
    '$bM4zD_x=n3>c~X$vNgoI^O+=`(7EYkw0g*ihq*pcoLYS;%i^7(uGTc5^#JyaN0~wXI3jF%Se-VcJMb~p!NswNfuB)D@5U|I3Mp8x_Q(ru@KYxn>YgTK#L2!'
    'p3NQm&~+j7))gj2=xClr`(^LQjjS8)CGe}Ai>otwJ;Km_*-H6SejgJFe%Yz(Fb=1E?SJ@X8M=nX=UbJ}Gz*QjqntNz)c$P9ANgyAJ+MN?$up-Jg98aZM$ymq'
    'ry5ndz$=jn>K{qxQ-_(K$7X(;0sCdA+@Is;)X)pOc<K0Msr->|nej@)Pnxh_cG{7J+%4>WzzcOhbsxuH)7l{d-_I~lzQ$#1QUL?1UdCZzcHO300v<;j>K8Xf'
    '2Y@}ObJ*#-UhZd;2=5U~xtY<A_CT~>R(7f3WhHJ2n77QcW|T9lZ%5}>UGgmSoD_^;Jg$^-hz$Ogg3hm+9(r?8Axaz0uWn3VG@#-UgwC%naE~2)yN7TdcGP;w'
    '{qA|>f6V>Z8p`=|h7`Om=o03dv$z+HkLEk)9Q5}f)M*AsEncm)HXMzQXC6INtgR>Dqv+yO31Kt;10N%1+WYB$_;>L!^mSd@GCc<vA3e{mbDZN|`v-p{&#9pD'
    'M;FP)hF{w;d>`UCP-Z-|FJMMKSS>bzKZo=j=02CVG0oG0`QuF`W49}xZvu&Fba(`nKfZly&1bQf(6`>E0{arb^%!76^)*>PT3xjxUmlKIts5CNJw&20G1UIu'
    'v%7RoT@e$!Zol5+FA7`{*c{aR;@h7uP^C<$!(F+-r$?duA*hl3Q}-eVKvJyS!J5T1&zN#<z;&<nOWd+n9Xkik(CYQQ)&^uuCB?J-xjMVgh*Nkqej^$)ll%%E'
    'Ji<u6bYe>)8H3;bb2<-Qf1W=;IXB=w_=;W5+KbMmVlPn$m#r5vT*6I=cMWT6EcE+MgUzUQ`)(=v)mn{W(7J#*O`#==Z-dROw9ow;_+8)eUir&LvuGvQfBd9r'
    '<BPk6v1q(1R}*AN{LW42`+c|XJf~PL_LGT@K1anz{@oTaf=U$K8Q13q#I}~EYM#K-P~T^CEH6Xv+;kW})~bCGQY$<MwxaHfx!&@d$3GDKIXX{LE0XVa0W}dK'
    'J_<Q98~)ZGk26O)FIY&aw|qtZTqz}IN|!}1;1!@#Z%Nf3vt(w}U&OTFddtr>O(*2rPlB_k_$reY^5nra!gD`<LnKo|Diye(@}1XtSg#D068LIV?Uiz=P){I%'
    '`ab_bMnPqELjLB>s$XY3BHPgZ)pJ1~{S=NbhVz}QfgvwPn5)V8PQs-I78?A<#~nW;n9aT@;O`$k340r3!*cJL0T>JFeCiR!V0|uv;ImN-za(35E&_C*o_o6g'
    '^ueJB0zTUCy;(lCs1H5gYKq4xnRV3)1fR6h=MMkM)03WW_2aPjo$%Qi|9`#}D!=+;#3;%5-A))E*9_b4dFxO!vHnPkB|{$=XB<~q1nZAa%RaXhd?=Y%e<c0S'
    '+@kDX$@R^HpYQ9FS0y`qD<{?;Nmw9t=<{CtMon0MoW;LXhfBa?V*Qa6bHa*s4@t~cf%V7Q@YbZI_YRQt#|i(w|FF5NT1e1aI`u9`*S)O&Q@=WTb=<$2e2e$v'
    '!5ftLdYk;Qoc4<85h4e{F4X?jeaE;o)L9&kua4GH#m=A3f%~YPD|5H*m@OA8yv|f7?3E5fGkyJvijNNm4wqyuoD1t$wJEP%mv*?L^(*OHB2G4!2>R9Dpzt@V'
    '7<$P16<ksxv*7EvKj*3#SA*p;%6)l=%YK{tPV|K(f!x38_X7RCzzx1}DGMz@{v_vVw2I~zKdp&G>t*w`rj6(Z5aQNzU5}y_%RA`nWz_n|TV8=PneCKey-Y<T'
    'pkMT999l0kG5wTh%S+IO+8Qd?+MUV$Loc(bk{GY=s1E$trISbDx#~C`r<#`L?c9%H$hi@^)8De?5azbLJA9n6M*lqULfy$1XSb|~-1rHD*1a?dlzg@8(Zu;x'
    '6841Xxn8U|X$bQ<m$c4wsRdDFKKDmo8mj(ycF7J6eZSc-pYu3a=kqr7B!<lY^M0~9eBc!!Z>fUm@DIO_1dmYrrBRBBzpjP{UUxumEI#C151G&Xk=LU7SBu4l'
    '#(t`+!|}}Vw1t?*U?gUMiZ9<p4@yTKC)C}8r|z>q-h7Uk{n5XY%iuRYda@o$WqUUZ&ToWjWp%${PR9gM@g-izM?Kt*P-heN%<gq;Vhor!$-MJFgO6l?@<E%U'
    'Upwt!eB{-ArF6!j8tu<vIJdwr;pGAtAEVD0Pp`OfgN%=K`{$7GcvsE*7snS7>co^+yTA1{t^gVGbo4oDel@T9k;5fp${Yt=pF8u!*T-vbBqqm2XD${ig<lD#'
    'ZF#|AOwq6KxmbtPXKlFgv;-5MO2;olt?$^LF`H3Yi_o8ZMcovw$~6FtGSlEAgXK6r9#YF{N-xFda4^imU%i<>ga&imu{EgodfS>%%isn=U7_0}y=vec2QH%C'
    'YZ;cSDTa!S@H+NFEu;4I`<M)kf1X@-N5x0)$AT*wfg*mLc->v%ybTN4WFxT()O&VM-0{BB6~c82eaIF)UULRmquwy4%Pe1)u#Le!8!u(`?%>CvU<&H|dTY+('
    'fSMJ8FdnZK7_3P-*91;Xqm$>R&bON1aA}&ii4OkU@m+k-vWsg)|9-%W(h<!DZ(oiEgu2&;AS+#F^&a3x(?@kq9^-iboT&6A0+=*qZc9I&?{qM~Aica7e40XY'
    '?n^2e@1Ij1el#W~h4LKmeXXEaJ@W}uC<sB_7r*UN5-~;7;OF4iePjniV;ZnW)lH-ZI<P;f7>1w29fm~)2g2epBtC{u-4v3voKR<Dv}$w8tAuFmE^@A7LE@sl'
    'dqrRz7*t+a(Ph^@u|J1|Jy!zfe=~ce2%k5+^G9+Qb#?!7-XPC=q3VyHv+aV{9#@3ty?l8-r10WP7?7W=9)H?X-D^%NxHHN6hoy<!qDfor(SD#*$CTpjYJ_@3'
    ')1N9T+e};O^J6MLYN{J1aakL{{5W=Xu8w}_@jv`PM2+^q!&|Q^<luburWSJz`-h2Wf6ljpsn>ctWdB)zPKl+u(6`a;Fh0g!vbezQUNtd3lHTny0j%QSJv|s7'
    'FLUfaBpq2iF+P%FiS7ZFb^KEaee%z0%ao1zl#}t1@JE^-%tFcME}VB~ZbChUKjHygPuA69NT3{>%Sxl~thhhUA3ZL#p4s=BlAm+Kd5cwl#az~KObHc#SX3=7'
    'nS%-Wq<h1R&%D`u03T}KAC#D@;dGSH_v={8hqIs7)quUy6ZmsTcpSqv_wG|aO5O{PlWiwoZRLMz0)C+C{BO#As9d|8z~@$XANsOrUJGy$qJtk)d`#W(FxSan'
    '3D!xs^2n$t`JVtkQM&A5_q+u$>!jg0nJAs%`dp`*K7LU9%blCEd)oK6DmV_{__4g&>Eh$aC-m`ynqPfgdbdg8rUHHz$L|ZbrKIQSN2k#EK}YX=Y2V>$V)GyP'
    'L1&#*5#v+qq0?ymIQi;j!zCse7_T(XR0T&Kj6&nb&kqW&it`Ehks+SGjLWx;j302w>*fDG?q6N2USS)@LctGwZp5=s`i;vt6Mzot`5Cy1o~++R(339vO|yK@'
    '?F(E`IPme;t9LaA33<eo?X~%V`VZ0k@$-wt%VUo%g83u1Ww@e=?-+eOkjfvY52Zxkd#nlb#~rs1+kaZ%kJbYP?$6wi#!S!yYm=Rty=(qM4;=kfA|-i+pwCtD'
    'w9m_7>O%8J69&T_`XOp?JhPkPyZ4)51e!nEoO)zkyN$pfXYtC$?Vj-r%^wH-202anXTtn(?Y))F0)?q){^-!4Smho<;E!h-*sQwa{y%?2)_1IObvvnVYy;!t'
    'h?wQwEB<%L`1r@ZV500_^Y?o+Tm^Mue~v_`WZaXuTA-`_XFcNY`N(B!(s_gBbzz<D8?*ip%Y8lIO~V7d%WoL`)*m}QRjlC|qtqGW{odA}+^(i8dIWocdavtm'
    '$VM2vn-1>_$&E^%KP1(HB$_yB<0fRtNac@_*{4f(9>?cDQTjMVv+*t%epn6!S&2Ry^Yq?v{<ukP?-9^P!JTvD&+qdbmCRm$672gMeE`B8sn<MDTpa&n?kSyh'
    '{P)`126?j3b8>2h!&ZWz=dd1lJs=@rx$J#VtVBn5rp`x>xyrAzF@xemrJSBNbjGY)a1?~|Bfc*zRq5jL&xHO5%Da4~xn*<!RhoWpMsGRs`H!Snxp&=MzSJDb'
    '`Q{7x^BFb?`(zWl!6HH8bMW3;U_ibkQfwaFw2-GJlfwU(lRxLPGORn9R|~pm<MPObasMjI{G}qEr|>yr4B6z5g}on#ao-3<*LN)S+xIHhlHgwzW7-)Z`KlMo'
    'WuW=K{G4%rPN9>-nS=Z)cq}c#A59%kp+>`JYzqT1U+ul?$~ZpWW&UAsb^%^5!(fI#hEGv48!K-9oxcjI&*|Bz+E!Aw<L~<CK<ny1nEs5<3I*#>E7)v>mmZ%n'
    '9j@zno_7D-Pw!-4hvM<K2dWNe?IqONT7`?OJW;d^dxgTeYwHU3-TOWrJ}<6#7Mh*=AQ*d$!qp}T>je?vf-p}K$pXvE^dF$>J6_LrDw^>{ALeuC2HcvBPnOfy'
    'JE`>@X0AbVYop{~z4NIE+tVV?6KK6tZISsoeo?~t#x-urVV8B0XuWfEMuEoDuLS%bYdoBM`Eop(KQb(im(bRg{b%QQeCKXm;(cT*jE}bRHJXBHxBtVx8q)@9'
    'B<8{R7@C#FwQzbZ+P^BW$m#Bdk~y${_26g%i<ZYVw10KhEuPiyg%k+=SMRAddux{c;a_F=4*6Fd+0LqWF2!|zhE3#;AG<7HvOmcPyh5F`a~FrS3Y8GvvrMN4'
    '&KXvXfC=Sawc;QCd{tHz=4IQs8-jB~%E1<tfAydn(^m%{dDy>tvRFEiA;cHVN4+mk*Kvond?Mu84({0Qx+C=w+P}&(C9`Usk{s+`WqalxFr~C$(*9LDUEw?3'
    'T?Z%aU#-8L6!iK@61afkf$YlDuH9%5faB!Kyqe>7aUp2`>g7Xy#TSke`mZa$Gj~6u{E+NlrPZs){i_QS=WK|;&-E0&dj5mb82#1OU=Hd!t&Z%@&S8;+zwg$J'
    '{R`Y>+khaAe>L#p=r}&6cpkY?El=r3ghx0XYyVd(C3=89FHL-%Rxpl_b7t!>ToI<6!>-`mtNA(862(ZMkIFZ=^uB1*dqb$xEWI1^+FCyiFrdzv3YSiNxN1x2'
    'yDQOSKcs)q8?>R;vDY+c-^Pf28oH*6NXQ4E{i_j;2G?sWMG5{@-{i!s<t=FcYPVfjxoOgTn6GLH=(`(h+5x(HC)J<hA?|r)%~S$kU2Y|{uD8+`t#|Hm&N3aU'
    'A?Tf%V{vYymH(l4N-K@^g#~lN>lV04a&A1|PUfo<`g8J~x3){=66)hV_iZkEus8(!fqD)h9Cu!xY$E9DGT)p}IS40Xp{R4;>fOu=>a8;WmVfozIDb5LL2b{n'
    'hRrxWQtDf~I;vHjuY*sE==7WV%^#0G@SEyn1fQE+uMV?KzgP*B^l0Ae{4+RzB*n(7W+$Fo`BCP`<DKfI)~uYtd@39aPNTy+sr<2^+s7_<Ko#z5()#X6-jUe-'
    '*lA?{mD*EDU&RP@TLItqXP%KR2Wz?i_<iH^k<}Em&V^qh_$Iea^W^g3NC4ka``+S`!nC8doNzqYV6ew=wuKYegz~)%-N=x7l1AtglVzpHxUld(SR_j$UGNCw'
    '{Q*el8<&tAIk^mqE{hM)2yAk8>eg%oZzYKJW#1O^;Q3Y3`9|sOq9`SEN?pMy9s*y=Z{G5_31myq$#YZrqj~eL3{fUZ-vs>T9JD#KPdCaNt4H0JF-h^k)^@_&'
    '$OM_&r@XcAgI+P(etiJ%ze>Unvo>cz?k#v-0YeX-Csn`GxAR6L_#{l^S1KPqjn_YJ-o3E4szeIM|7JWy{Agr&RnP*~^3sVvRDAR{7s2*$%i_EYk3S8EWb%03'
    'BR~s^?}R-0)uv`GVF2+fXNREWO=}}DZ`6swwwZ^uvtJBeH!N;=dHuS}AdQXYzO2m}ukY{xMb`_eW$^2~gumw(>RYVlM`DNn#<v80-*rxI!M%r00A<$6t>dt<'
    '()!Vq1m2)<uJ?+H)bN=p@cE5<j5(+2a5iT2H|HtneYnN-JZnzQSzwO3&WDM|@9zIX=u5w2MoZ}V&OodQwXXl9@hO39LOqG+k*a62?Yimr3#aB+%~QG7|Ll^%'
    ';~@T`8nEu%6MQ1{@A`#9-<Q>O`1%d@$K-zDw4-o;*LUcu_Q)pOkb!ahvBQIJyZO?wR@8Iv*!GZTTZSNvbL}kQeheMam@(?U1b-^<&TbO@U&lwLiSY4|$+mf~'
    'yewdR?Efad?XE^Cc(s5Iu2S)F7lYf@bt)K)k9Y2xex1`(0V4G#$gh%OoNmw9!~LOpaDMev@>(lf$wVxPmnOf;@@PE2+8c2*LxKnIzsi6sLDmfiQVtfnVB5JS'
    'mp>o5#aR93Qy`c>$-Y>T8|P>*lxhTDX&Mrc=Qy!b^2aH!owLPyD0qdR5`11}8JvhH20{XKe7#iuIFtKPjw`PMTnFzEe$g!O{t@sD)t}TRE<_;1O9sYOar?a+'
    'PIQD}TB!FWjPX)yZ4;sXVd^&hOGaU>pl=$Tx_eFvUSU^ySHI$-_?>$3l#lq?74F<!Ac~((zLVM?;B^3J`Z|fZaK3Y<rp=R}sdqp!E1mv%RQ*xLUOQ8|fs*f}'
    '<X;PCD>+?H0XtE@Z!PERhu>>>;JAGLO#ba(mtz5G{sSRzIdXk`e#c7_$CLguasLs6IeuS!7>0bO6=8wQ#NWptdC}mve|4zvvqt}CN*^P9U#ncx^4jgV5on_J'
    '*=v`TdrwNr!1x%u#HzP1I~TJ+?acc2C@#xhS$JKaqC=#@p)`Q(6R$hVPGpBOFP!hJt^NKr$}t6eMV;T?$1P_4;Y#R7$QI>YS*UOmnA7kUnM5hW-~Ct9uk)A='
    '7*P5j;uBE8Ylr$itL*@ZSMPqBf)AmDev2VP@rA-SE&yND@4GkB{(Pg!09-F~Fi*y4?deEtGl~bkzaCnkVojJ&()5#q;n0Rh*iH_F{y3H%>MM0$9?qkwiz{3^'
    'E*ykyLh-q)#!Wh3><D<gWb?OdVZ~mI2c<twpD#Vdpqo<Pf!{!*OAKGE+>)^esC}kT5cf8)SOCVkxfb)?)N3NZU-fPMSNWrSu4-`3wv8}8-t9DwYVx~^=8v4W'
    'of&7y8^ZXQCbZ=a+w6*o`6DUbnX51_y|i2(=8x4nA3AS=5Hf#cohZM$@yfJ-L(`Pu{Hk^F%l^`@=Rk+ZpE^GS)xXN58)0tsULNLwB_)p>o-Vn8EoG<40|(y5'
    '=Ub8DgpRql6}OHQ>_@D#6#LNl9$;)V`FgT1l&U|jStq(!l3N<~g^q3*wB72k8&gHSFTt)m6LrD~^EB3%>F93deTwxk(A-Z=xAFct9znIrEz|J+D-3vM)V#2Y'
    '#rJYLc4d<J&N<uD8g~iCgBg=tH*I&K{$k4vfK5^-&6urgwA8^9#PA|kT>QT1z{}xjFpl5e?TrbK#$voEe2fx0?l#)T5A%ozeWh21-0som<Ghr3t~7k2&Q(zh'
    '=Ho9C2REeus-w@xsq;Hx6<D-ue<{O!e6(BRjmy~v`h5KNeB^C)A3TpJz<k`>WKQDwF)uV9f6ZTi>CrHOj}MwmF)JQzrq9Qz^^f)LC$bl?$ijSl;i^0XCLMn='
    'AD{5=&~%Qcrw<VNP@JpHt0_E{h~|$n@62<*uAd68d;QC+e}#S4r1_&vivB*9H7RKR$hY+T7S-c~=f0vY&v!#+0GdBeogzN;z)}*%t6vs1$@W~GX#VK(ivOiw'
    'i!{t1jpe@%#Jrn4e{>1XX;(?_Li0!Sl<fh_ouy&^_(3~t;m4j-G=FS&Xm?XzMc|Kj-%asfB^8C{kJ}8r)Oeo$oBWaVd|$M1L|J;Shw)K8NaXEaz8h%%xHTmp'
    'a^*oi7#|Z=gYwjNm;cEhssDtuXT8OF^}u%>`1!U3W6dv5?!wAZ=hqE<Iok&26LjhRz|34J*IIC2o%l~lb$-9+KPv5*WxV$^WiA%Z8jOoG+3$;70G7N&e{T7V'
    '^6~kPE7r(>m6CYhD26Wlv2IL~h!R{5_V7&Zy?(b*)geYQ8N8WfJh&^SA(V=>08e^-+o<_f;iZcm#ggS=-l?)o*Y<wT17Ida^Ij{ZkL!;vC+kn}9F)i7GTsQ{'
    'YL@?uA-5x777E8NXPk(7uPO+~<=nQIdonZIfg}y@v<Un%UjHcYMnJ#nvo;==@8Y{jJJ>ijJ_*F2>I1!fLq!@i2<L`&>-{-mm#RSER64wc>d(2Jp}%Hmgu(-H'
    '-ZE-6;&#ll3_PDg#JShcI1PXE$CF12vzLoe@~b1{&(B_{7+Psj14O6L;7(ZX`25F?ig>XabDRfK`qJ&15rVBqK7j2<@fOFE4WqLPL=89~ebs1g_|@`okb#O@'
    '*<ZMvUU^Q3*Ih7~V>R_<7uLu?6JP%iAAk1;5Me(Mn=wZXzfPS0Pwmm(+w5}{oAWpO6Z*c_{rusdN|$2ss1@f@&5xQ@@xj;mD&*M=&e03t9BN&JlbG8w^_LiQ'
    '4%_%cad%sWH#Q4}1NQ<47o^Jb!{@0R(~oT2(e{AMABpGo(*_Jw{c!`oC->DQl)NS05jd&EZ+>r#5AZ-;rw3EE#id&WUN%jYr$MD97VAe{r^(S7@<x5)@H$!X'
    '<2emZ56OIWLVV;}U0U*BK-K{Ie?PfqtJ_hPfJvjyDeZ#sCFM2KVElh>Af5UnAP_S~`ExY2v)8UZD*bQybH0uHbFK>;_wTM;3*%#O@94#j`of9%BPkwiTzoFD'
    'B9GwPQj@+B>~jU6^+$1Fb<@jA7v_)FZM9D`1A->zkEG|P5F6M0{i!_6AI*5$_--!An3zA3VzKxbkGY|%EX*J0TTf9g9Cjn~$BECqOVrNjGZN}1{BQVwQmj3I'
    '<~xu2&zsDR7KPU}n%$gnYjq5o?{wd`R6V+ZaPGb4GFzY|{_e#3BPljp7_SUc%FuxMPQewq=(S@t6YGzp>z~WB?Kab91(@$FI4JGh5?F`UA2)w4sW~Ms2lE~A'
    'Z3k41B9Ei_j>!JoR_ZSae5cBg_u|vSCNkfFj~u<LddBBJX7;=;ZXL$^(l89;@lNXy^M_ylKuiqreF5{<T5!t|_|93`%mYqXCVD=S!SwgG)k3e}`oPzEM&&aG'
    '&QF@}WN%enKFDzf&3Ej16gFmf5uST|Ovi&IH|xlJhj<jrHu&8?=kVTEtCm5W?@)Zw*K61$fB4?RBxt^mTpviqN0+@VOB$P`;rc))o9@(emJyi4-}tLx{}IDC'
    'R>n8Jppap5>(=bPx$vlkA3)|ejFR1W+h!B+<LdHTBfCGhk@*}U@3W!_pN~w!k6mZ&A4lJ$@HyNuc8gV|p<!JIrcQ586%`*D_=C-v!=&LkRpnVv9lhUKffUqr'
    '&ev2|=y)$gsPAN3DYkdG3k#?1do}Hx0Rz>)`hT3$CNP!aH`K$AN;zJKRW0+eMU%Xr-f8{98>(WkHOTout1aXn2N8T2X(7c^<My5gnv>+Y|CIiS4NJ%!6<$No'
    'ALn_8Bnn+8^GBA6_~YT0{w@9`Ixs#yYn|HluDuLg)SZAolAhlW&EZmkBZNMOyZ6N29?j3e=F;*<r}lAw4p-J0t2HU=Fg}KA)pd0|PsQ${o~(t}JGJ~>gt<ct'
    'uW=kOvYmxFpnN?Vm$x#n+f3*)vexs|kF+P%z@3J#lItJXDe-(}{KAaBZJKcXLpqPO_C23Mz(V6E@n~Z(B<qhPEO0rW@ybGz!UG4$pU2#NToEdK4YUc-#7S@)'
    '&##j3>dB{~`dR6?{>bp0{PFP|q0g>4`QV-41oIzB_t|3+x5J2s7>+ym{d}=xRkMAw8@Mj;7w-#%ANlo}tFGQVh3PU*ZXNf7M-Hc2&I06|V=>*O48A)EzR3%%'
    '%83b#d6@bn@ham0W6H_w7T_#OXTBvb1up!SnzK7NtHJXv!=*fq7IHoUxx#ej@=^Jt_u$!KOFuRIIpA~o<kUSLJxV!@uD4vwn!8Uth2YP1NRQeyQrQiR`6ife'
    'MZ&t#rF^#BeTwk&3pw@HGa)}7bfV60x?eQX_irTlo!d``s@TZKV0%$_vh)sfFVjz9c-_nFSn<noMOXzBO`Wt<%lQ09(Mqf1Cr(rLCtl|--qF&qF2f&NiQ?7S'
    '#vySSQGYzaDd#rDG7Zh=J|7Q{PwHTT*X4gbKey*jFt!-QuS{KEpLWX^fO%4o&i1+b7h2HzV{yf^y)W!#VEyq6L)xBvO+U2$_(1Mixoa=MpKHPAkyBOIh1MU#'
    '*0syHE6c<B<3^R^lHV)i(fVWi>z<l3x&)r<9%)y%JSXb^>yH!hN7DTk1-V?>qUJC@?uZcB$hz|inLpC4e<VGtwU>?<aY+*DAJueAtHpPgf>ZNp_}rUJgWvrD'
    '^glJsKcY|RFPDl_c<XC}Px;fpXL&k&m4(80-quWM^vhL-`||78#dvofE&%;9#CT_L^5b~_oUi=Gnfq2#;vMc|Otjtp=tE@@Fq5Iv&yHIE_^9kb<z5*|yu<4s'
    'c8dlZrre6bUh>f7KZJtE>mN1uEI4GSN%50x!}+7sC!r-f4qz21-@SzJfmoSpaX9|4Rp-n!?D7FBNWWgB@7zOA3GvRN&d96uNG;gT|EG`WH$GM_S+w?Mpca0e'
    'INNXav+liN<PMZjey997uPSTe33Yxicuh1*mnMP*sO!8V_x^Hvi~zjusmmgH(ZesXWCj}iKUy`;9}gT^<)_dk3+uoaexz;vbUzRCn55n**nDENUMC8uOjalD'
    'PFgagAAs?o;!jJb-3R;agnsC%2bapt;kgguL}>G!XL0?JgdasA$}caEQ0htWNs%9>-xxHjE&|GI41|{9z1ltF^&LF#2GW1#Q|gv7$)BgpkUw(zs~<>1)`>H3'
    'n!V~6LC-DwH1NDf@F5mJbH3@V<ufGDM<)B7y=Imk<fZfn!V?S}PkgSQiAcf>QRkxC;84FMHwgXE6`!4tGre^YES=<Y5ErQKUcKTGaG}?Chg#pUWYfsSTp@}t'
    '8^`0v{)`>j30<I_n@+!XDt}Ck71Ne7SBCq&uPhpH(Yn3}yhZ)K>Sl>IXV(+-2Ck~ZuPS&G!4OI>a;e$+SxTQ!-+8HXnVn7GCM+IxZn$O`$y!%HsF#uE<5(LO'
    '5{Tyiv-J`eML7}tM_!!Vj_X<?(e;n!8W&?DPe_pC{J)5gMp?XF?HVg#e0-O;a>uH$0x~`_(c_P#{q!~I#-ROO0)M=7(O?PRw^Fo!)lR0f%<--!>|c#Ku_U@E'
    'GX78hD%>AH>ATslh6~bge}L@LJawl65A4j;$>k%e3d+N^U5|jDtdm=}Qg(37oTzHDfAx=Z13QI}f01N*Uc7t;%yTsz=9Syq*MnkZIyx!Uzp9Y<%(`ip65J29'
    'YW2O0&TJQukHRZ8BjGqUZ9=@0jeO@TKcfXqk)Y#4rPg<>-P-SvJ}e8*f0~hW!CgS}5oi&j<3pwTS64Y{3lA|<a25BVYMJd4Ro;0Eh_TVBm!ay9@4xrxm})4('
    'Jw-mmy>Cce69<s>-ullMDe9LA!G6nS?`FoWYdeR{Vxft5+pgpF9VGlXoiSkkgi{gM5h?v7`mWh-e_w@>e9MF_;}9`k|G2G1#9FyS4%g>!zmr+jqRi(Vhk+gH'
    'I@e`0U#>mH3+rrJ2e%YUm3Cuow0+|HO7Qwe(trH?;ik1@qcnLPMd!ZTbZV&tb0n6D`o5Q)uD6V`_zXB9ea^EZT|>R-@9I~fbv8v^zF$5jfzPNFzIFMdk&%M1'
    'UzYQQ;q~@Q@!$qZk9YIEerv=_2tF4{HjYTOEx!up((}Di`{(p)Ni&!2!u#1#^yBWQ$+pqLYe5m}y&dS#{}CEYIA_iZNIBnQnF)MQzfa*|%p3imFX6oUlv%sC'
    '?porDEkfbdBdd;x58i}2Tj9D0RTkfB%$DXl$#Yw&`PC8D+gsjL;&u0wxvj~YB{3D{0E`Xw`)WfLcQ0%cgLz;CcWcLimx;g%_4_o$j~m!U{lMTn@WQHDVvPTR'
    'k8I|%SD#M#ckyvR>1xW-17<Kj9=f|or?fnujE{8t%aQi4#)ZwVb`;Km`^#xNyS8m#e+mezPtaeEwD;#*$|PyIYQpuc=Ui{yQ02%11FAH<)4mYLM^ZfhS#VgO'
    'S%uP9se=45WBV%hzE=@oh2jML<wzI5EMT=j)lp@*zuaNnuU3xVbFr&}G<$(GM~>>xxpmu0EtG90jvx4a$r3)GSFtGqi%0Q@Rd%+g7HyV<&xK}BnIGvFdjcsw'
    'I=qvbUyUkO?lEnUhk0kq5~twnH;XV14jO)CacMliYI`R|Y)K2nhjIbe-DS+n0zz}K#;HX8(YvH#yg#{(^w>LwR!aSxE&iT;{WMU&(;bukoAU+~Cv9IX{jy*v'
    '0pv1FZrxhtp(Ov1oWIjsAbjK$9b71o=mP{%`747<CZD963BI1G>%K0l)GG&Zw7l91%CGXM#a`MpMwyR{|HCtt(%s(&^FR|f5m(FY9*^f&`5iM1vo=%ali`k)'
    'r3Nv*<~}DuG%DVGJH>gg`YStpolPFvMXX8r7^yD=e9kOSsT=2yS&EWZ=XK&dp5ZO7KeoJBR3~nY@uK=kbPaAvh`mSXXY(@uTD;qVd~CA5<?deQXP4i{fqK;Y'
    'z|<A&#QliCA1^O0S3No!36fBJM>Ols{Qfb5zA=4M=0$Ot5NsCe`}#A2q))9O^y{k@3-I2M*$f`@)9Keo<&R!@{u)b?<>7vP@45naH^oH(5fl%!Xqhh<mmm#)'
    'AHTw(or@c;fLfaUN$xLC#mCi_66-Fz$-@2RIb1EbFK1l{BvJc1Ik4xI1219T<maTNJ`tH2;4<pT9oW_?b-(5#++Tiyj@4aty$Ccu{&dJsdR+29kB<}jR|{7t'
    'sg3!V!1%Z&{B5OoW8uX5BWXV!eHs+7%v}rSkHeh$tJ(WXCe|NGu{nz|LB!&lI;=k~GhN=XUMU-`KX!MHN#)NY%zun^+a$TBDg~`S&g|SZclIU1`Oa@ZHT=i?'
    'jb#0iK+U|@l>g4JTD@W0yQK(!KPkVj_SmX|GhI)y;|vq%kEEARUd`F{mx2V`KgUerg}^=i{J)#e3dNC!I~0??U)YDOp5(d>!plsjV;jH_3g<i|o`Y>`2>l3b'
    '5=NCeB<|4HS2-#C>fz70A4BI=VSQEon!Urok9_+2Dz*RWr&T|ccL>SC`l@Zf@oW1I71GyNsrl7fjW^3!*`;88^>Xi7Y(UkXtglY^CuM~x@XJnv<8o@A^-*up'
    '6tvzM#(^=P<tNOW+<ejbMUZgLr1j359@`^I6YkO1JE{3qHSg-rGg6gcz0+~<@S_J09?;i2srb03&3P7YfD){C>I-$SX1_du);pKR`>5_SCg`0V?8RLB3_9uS'
    'oz(p5)Q@#+msiQbdZ+2ezEzjgPowotPxX(tx||95|A26#J>?uxXuVU!?&J}PEd;%DPkUR#7nW;(=$++G4-9_yCm#%J-KMHX*)RD0^pp>nJ5=C@twZs^k|CXU'
    'z)g%$-wA@pvW(KeKGb#UD7`N6T|>}2%ZIycXPE?|^-h;;*^^g!_~3k}-0Rl)k1sT%{i|j-)pU*ZrD6YS@4i9P`8Kg=|Ek$rk;9y83I0`~*2LuZ%_q?QRh#Gg'
    '(h}N5|4IKU>3*j@*qE@baT$z{ktPe%%4X-0`6I!H`s4CAe++#au|NAg1s`$VqQm-Z+U$9*AXkOXdq$n#krsQC{f+~npKt2&lfqV4(t)!g9Um$Sh3_n=5_0Ji'
    'p!DU(|MXs;jv8H7jIE>T+eYS=)cOvdc9m83;uN1IMbBOp9eLxpJ0?V@uIYDuM^;CJ^~!Qe{R3_wHLIIbeCf(j>^*89PS0M-5X4KEQ{{Q9khfOhBv3%k0p2}d'
    '&0d&Le;wz&`QE`7kI3_ih$$?}{72G{j`sS#q2rt+Tz`FQUFx;Q^I4eLByp9w#yL0IH3<8H%Ih7JpT8vOGa<fetruiDFHsC6CF$s$RD4{0O>mb%t|I;%@Lk2o'
    'Q)^<>6atR$(fqze*T(VDeyvv}pE%|B)#BEMFSl7VSPU?vkC@)?%TD`o&oBWDO!B^DAKPvmJ=_G+IRETp_<MfGfsPba-y%vKBi_eQLvL#^w&W(bz((^PMP!fX'
    'SKY3}&9+{F_j_YV!}D|3u0CEpas+syc!X`dvW2=TA&;oDB&ShA?DF58!=QC1xn5p7suB-IQG71_vw%%nB|&Fvb+OO)+kX!u<?IO0&*l2KKWA2%ZPn-*%6nUn'
    '^IXHxHw(ogFd@|Y@cyoeow$r3e6IazXEwy?d;ah0bD?!_wZCt#*jo>l(#(4!`@lsh*t2f?)QvutGw`^C`@og;M;4p^tN@+-bbR1ce@;;7OSW?A{8iis-kKEK'
    'e>B<~M56i;ce8J{zrjrCM<{8XeEXMUCT2Rx-)EAN9<Yru8SF+qzbj^ZiWd`wVV+y8n`l}#Hw?`mpBr*HOJ)7f^T$8(omBrScVFGw`!-8)e8lm}>s0=&;Uky9'
    'Mhwv(K<e#I0htnQm_JsZ+rPjz>pFNxTi-E8@#m19-(&HHR0fk-_&$szf6Ojlc&+~6MsP(2@tks7Ek~x#B>26zJ6Dc%S~q}&Vs!e9QSmXkc3Hy{LCX7y`*W^d'
    '(OL6#-$TI1OLI=MI5D1IopSF!m(n}R+{m>!gn7@s_u~0-V2ygOT|ZTiD18wj^p|s=Rk^A(78IcJG%Ah{D}KBu)P<BaN;nR9<^y#OI_G6-{iDj`5a;_{@-Y9{'
    '7~rLS{OAd=U@D#dWE>QDGSGT-e$6V1{)qP{3p&jiba+Mw=Eq3PL)A|GGG5>Db&Bi;zCubK5x<`!+ox$WI9|oHCmEM@`i~q_4EM*@AnU{~WpU10E&}7gP4yra'
    'pU&&xIy>>X*NXkb`;(LL*f8ZfhZ3IWX6VEH9jAPD-ExQq$51?ip;T}F)*t-v_bHjI2z=rjkIiSLsVfM`#`CKrEZ7qEO~#;72JSN+*d<YUH}AzC^?_vH3$_0$'
    'Yv-o#?3*Y$DeilTk+mPLX!?M;(E66FOULu8il$oq>LrwVOMITDrd*9pbVUMY{5Q`5(n;s6S(EzcSShAD$#vG;j8k4bOaD8+6O>mMYIFE#>fi{{QSp4%o#P@i'
    '*Nek>VCSH`%_Gw3;3MjrBJ_+NEodHr`|j`buJU=L=8GYD_1v^5_o?lK{^V07wmq*s@c<iTqvaPY2EXxfy~3O>BX1l(DA~4Yx(-vzJ_lnqsPoNTlg2ua7sBxO'
    'C5FkL-QzZZB{I-{ALL&xpBXo8fw~IpU)?kJCO`N29L)Z2o&yv=wfmY)(>OeU_9WN+Q~VJVQe0dey#(fuGdSWNrhmUYvHnQf>$gXO8D=MI!208C<qbCn^-hxY'
    '$3OD9)cQxWtkF-gGbqm&|Gt4IX?0&yvOu)T1o|T>=1iN;)@#-(3+s=TiKYQoym8>IJYv4p{P+hvSBs@!{ZTIHg^YxZ4aO%pxqMlT^xfRn8wbH-zR9igX}N6C'
    '%ohgEqvAm_!|I$Hx`g_q*h6VLe04sU4{BZR5q<?BOTxLZ<h8%noYxKX^%iRVV`1j9M~_%#VZCMW>|X0y&Q`MCLhwl?mX712(WTdQYlZPX4V3<CcRIH-#5wK<'
    'XHn+{ous(t#XN-hI_Z4(W_7;q1_x=*ujPw441e=S-DCIT^K+!}=Z80V;fQ_i#qcu>s37m>^`tkYt-|nmgRLX4V&HB%SdHxWl5Up2X_yOMXWTH=*!cg>i*)rQ'
    'J_@Y8YtYZz%sm~}lOD4io_q3^3r5nD;OFpjtsst%r1)AQ;y>4^44*H<@EzBkJ-_#C)6D_-w0Nv}5znuZ&ZiG&2Ys};I34a!e)Wgno}RBA*cfe!j8Ooue<b$@'
    '>In!i*)an@H%Q?;ih0G9@Kpm?4FioXVvuL>JHIOY>b&f=QHsCnCeDwuI)7|meh@p1dJ|$=o_Nh!PViSTCx)x<ym$p$IZ3_KFJ_KheZfVbGs$z9cIa2Pi?{E9'
    'A{OF#*!y<nxc=CGWoNQ(86|)Gg8ccWsFUr17It75wO>R<(<CqO5_tTMtqR#G&ojU+)D;#*hV^aQ{tfm68Go_$YX6ds?Ln<GIm7C5u7;qCaGqEEqF$Gd)*t<6'
    'M{K{W`x(|BW2L3@Z%qq9^T+hHb3eV(`ZxI_2_IM6cUL+eSq$^Xs}DB*WQxz8m_L%@LHOJ~c@b5bFn@dwM&(ydxkl!Xy!7-((%x$<Iw@K)a|WzGx=77rm0Vwd'
    'S#i+GlT!6ZTf-gmyr0R#bI<ZNb(k)gItMIA`4NJ_k5#<+(r{dEcMWMgArg)8qj&`8X%(%{GK6(@cZBb_ekupi8WZS`q<H?3<&{hjAEExSqeyd23wH)sFGA!y'
    'HbrB&KZk@pG10s&TO<_VzNR+Wa&o6jE`tlgG<-a!V7$LvAjeEYtv8f<eY{e;Gw<<_b&@MUA*zniTK|as_gI47GABKU|I>pg@EOUk)&}oqVIt&#{T3ePV&!~}'
    '<uDMRU-7H&2EYBQHSeo97EYneHNp8@rogt=R;?mz@g(z|`B9%Mg3oQicB57dN1YR%zLN*O&g1T2*ZnKa023qPe<Ke!?$06X@$W)jT(qF@@vr#ro3rmGX3_%I'
    'OrtZWlIqX#$hTVE-bi^4xMD0EVY~Ld>M>x3+MlepH+N;8CiG3{!=Afu!a4vu%}Z3C$7M(k@T7RI@H0u&WQ0;jgb#{dlljzDrMM9sXD8+ZT~_@V?=Pn~^dYHx'
    'lrm2fub1hwdVV78a}+2?@vE4PD-Zd3@e}ym&)H7u7FAdotv`25fdK<Gzbexb|JICO0q4iJ$sY@NNf)l@xP)y$JwNl3w|k8qGr@jYsp{+BInA<xBO~!WN^NBw'
    'pO5^m=ILq{FdgUBcz*!#fFtIW_oKkbB>C~@U2YZqnrFy7elqL685gg+W|{(mP`KmLxIC*wk_*PwnH#*#m%TlLA?r!%1G$$hs3qWI&d&<}H*qJ>{8-ayeSZ{@'
    'A78zp{Fa$7l+2GO{9{R!lyw*o{b&8FbKKM|2HFf@{^;%AZ1&<y_8<I_VOO)kZ+txd;Q+to?3p+|z9WCU@Rql`5Pv#&H-7^Ck+gq%uL&<&V>1)hA8VyUO_Ck*'
    '!G*ar@7c3+IDaI)+n!@=heE%o;JA~4%Rj$r)XUZd0vCB=Jn$^|it|U(|4h>$zUJFaN}U+alg85aRoq$;3x?%raEDZfLd~z{Ut9fC@Fhim#5-@j+a_oF$jt*g'
    'f%3f+CT^L3zC?)NUp34sa8J7m4v781tHJO3j;KfH+LOXH;Xd4~@0b=<vYrJGQSXbgN2oN{bHaO;*0f=;ODGGZPou%{gPi02bAltW)V}3Pc$~zA8Os*={PSG1'
    'K%pQFe-yoq^GDKVa1?Zkm7YuKzqW<^dC!REY`tSum;)<q-_M|Nd>qId?m5y*shhy-EeZno+@<uQF;Aw+#h(S5aYsk(?ZEp<uG_FObiQNpKCm2Bj}UqP`UB53'
    'a&TOptv54$(D)H}$4tC0sb4zA`;#wLvOdCmL=L|%C&`~LHcx(`sNe%KP}h0;+unFRe{T5uLLPIsUU_*QtUx{Yh|$3iagNUzw9cjD-tCw3&S8g8cf?O4@LHOe'
    '5FB4iRxF)y<4G-;CrGERP>2G*!Q3DA-td=&>k0+y&Wq?TZU!ItXn3bw-njl)Vz1ffDn{`UmEv3W>i4C^%m=|o)VU!j0$eX%Nzgm_^-J?YpT~k<DBKz3{@J!E'
    'fZ(s{lvs3PPySJC8R|I<rmLK}I9&?9FS#w=XDt=$F%uTzxv%jT!?^x9R<Vq|ET59s8p3f^>&H~BW-p8dmEVx6NO~fDlyLs`aH$#T)cni2CeS+8YR~6ftfv9;'
    '`)uaRRdbpX{5g`JHM}I`gVFr)!&P}7{%7L<EPs5VSM#aE1cUkGw}i_Tp~+cf{z&*E^%-NL#JlzV*>ew7s1x$;zUJF>PgP)V*lBP_E)~}wNwHGCMmA!VCMB<T'
    '49_dfSLk5fok#AEM3Csb?Z^GA84^><xi3-np&}k%t&;guAH-ouJ>sYp_p!7O1f9RcsyA4s>lCnNCHhyrnJUKd@%wpc&A4ESuW%<WadTSny_ARm$a(OsY{!0$'
    'h7-<r=W=+G>TgzHku>qu;6~$k{bQ)t``qGMN}mx&{K&_5ZW&j49C?o8#B@&5;<El@0{@tqocnX$ix}WS>+3N}7_aZBFJ`)+vW`+;hAS5a`aADbW&~iHP~Uf6'
    'URqx@M-awiufQ30q3sbs47F~y@&SodZBcmLLiH{KE`c~K8MST&SLVeCcfx$VW$GKOI7+Sp$9c4Q3T<3}B;nP1lc{ru2Pkt6@bit+eUJY6i!-r$dcI|9{o{_Z'
    'I6aOKS-20yX8WaWwLa-+zojPIrZrdNIpKYlt#Zf6)YJ^yit5wEFhkPPjhI(g1k+-9PnUzUGymj|jMV(<=54;>9CDO?8Tdg+Z$wvcuV^92mnQBr&8y!He&<)e'
    '3V*8^vXqAV3coAkR%FjB1wP_5@9npO@%-v{^MlWiRZ(yb?=S1UwqP0Sb$g&DJURZK;lvTd|1=6%uug8Be{@>TZfP%a-Rs2opBZ|KrAQur|5Ok2j0_sKfOp(<'
    '@&;6Y&I^4u?XGZjIByV>oj)~uQ9EGbqLVkE;$!0(;rrX>DZqIH|4%EPo@hD&HleQbp#5w+&_=*<P3@9ZYTU`-5(=-1#lF=!@^TXTNwt6SNWSEay+OT4l0l)y'
    '*PIBxh<JXD6=G$<X#Qxse5}W4@ZaT+*W`ZwYTB?6=8tz6qHjLdy@=+IYtIfV9#Bz*`Qu70Un%Fl)QR~cX+K5EJ2U@0rwsGQ{KjyeU(UG`^G8zbVwvG!>U>QR'
    '=8r3mzk701KVV}1NV<5B8?M<a!(?Foc$L*C#OYx)nm@kpS+YtpKo;hY2_KuycdEFf`D49%MZE|&A^-SzPm9v@^+jm@xaa`C_L_iMFn<g_5_+VSB?ZkN16vl!'
    '?Z3he<4&9RTK9I_%!&CU>1LhIvJ~XnB@gq*wmMZ+AGfb%g(zK?Tl~Wvi*9K+&-Uw$gZrzohxGXsl|MFC?7wg)S{CM4rTH0epQd}G`BfSJvhL{3JTSlV485%r'
    '?fM`5%I1NzGs9H^zmf<IXxmbg{vZ5`(|eRxoWQTX6fC*nXH-L<U-41$ohz0(Ulm}Q4)d$QzEv)Fwl<OZ6^!HFmIVgC>pR9CzAa>qpu|^PvC?PHe15p76zrNx'
    '^FEkE^B=``RLuIoNa;&kK>qla7wemOZ(;zFC++I9sfav7=wr0~-QM+GZ$p3#tq$D8KHfiv-}QS#9TR0v48B3qpIAp8wmXWk{Y~FiXulX29eTEYQ!o%`p4_@L'
    'pU$JnC7s|DZU5EUBL)oA{OZ`dN`;3fl=1t5J3-@v8r*73QZZZ9H#uH355LGss3+-My)0v1N<2_S;nkF5LdW-+67J`PCj#o89}lDXs)kD7EORFUUtO7*Dc<$T'
    'hs;+e#>d=^`BU2n^_J<=#_TWrib3<otPL@y-k%8k@rLuUA&vckX#S`><D1`!67hc)A00pJU8EB;AI3+qMfJj*+*xRR4En&ly<9;V#>e+7LsO=UA@GqD4>13t'
    '+e{l2VSM!6mcbs(l7q&_d1EUy(i`Mqe9V3IB(ZL;KN%n4BYJJahw=V$*Ol4=lEWzH<tFmyOSdrFnXOL%&t+-iIqSXg`VN<iv1)EBDEU>q0!qTL?YU;-c5Dj@'
    'KRgFE-1YJ$=z1HI*xDTfDnJlTUa$NWli_du@u7+FfjCY|eie_V>rMsF&EJ&|VmOI<i%fpUczs8TYv64=R%JYHh2Xl-)7Pb@@gYDS^}bx|vvH7Q6@hW?;K$_h'
    ';*U<4E9!lDwRvb0+iHT}f~Co3r%YA~_=1WD?M12S*H#hcF^+a-Se03KU?KGSno<2Z<;%w8RkNhwzGi#2#x(HeCS#^3eDrK`9~ucF)Y;^n_&(ZxGZieEq~5vv'
    'WAV&~M%5GJBWXWqOnEr!w_Fm&$NaLr;R?!)^z)C@`VJ??g2fvX<>CC}oOy$zJI=VH^N*@8d(4hJCh(7y`^|clSH+<7j|M#j7ZXJY>k`BJXDsUg<akay_v1G{'
    'zFB+cWw)^u+;3@fS+{bASS)73IJx_ICcs;gNg@dRoaA%(A%3>K*tME|zLVNtZdV=K@QdrxaK01U!^yiYvYedn{G$$q>d(2elKpU)JS9%xbtnTZd%b3VI)PQ9'
    'aJ=fCcHED20xx^V^kED0?Nsnrx;nJ(blKPpi}DMT&adiJJ-?&0^gr;?vuoIwWdos(@#>{GaCXf}G(KiUUt!zO{_o~j7f0TI!Tn1Y&aa;9zicz2n2Gl1^xE*%'
    'v+Pra{W-CQ>Wu@{S0?u7kk8tew&na-ssQ_Q?r01YF)z%W*q=j+m9bT?E38c9V1Le;uH{PcdVyqr4t$>M^=%o)$CH9f4ODq3`~DM-A6qRrEc$&hewz5xt0!pq'
    '8z0|)DoD6(jps=jrsIz#@%e5o#~q*qm5;dZINe!$CxJg=yzEKEb03oPmb4-7H$HB+HE~_fNa>%$N`cyAo4r!{u!jsZ`9RV-Dn3Six0QL^iqEN_tb2b;M|@@O'
    'IV|OG-ZKd2B7X^Gl?M5OxszP?<Pf-4BOZaCTf188TydN?BaDwfpVuBxcDM-UPcn{hy1x1*=KhH6uOi$RxtQ_#jvr&&V#*mPzGZxOPdVk*e8?>wt3-X%;h)_z'
    'qnio*@nfpei&-{@u+_-<dTZ>@?64=SJHGe*<5lx+{o&^#&(o&j<NMrl#@)`!@I38jtQ>>v@tc4IDsOq*H}l|<(}aGnVIy3L6`l3KQ=Ew7foCcH91?!aHuo5N'
    '{ehBCd5qsr&MPCS%q`%UAkEz9mO30CN%)be_~HN$^DO**a3Fs^J$#u-<AzFL&GE<i8~pa?_z4`;(esgn^CG6q-7D=HVnHJ6JXKVtaa<Y@`U*+Pe$o2%B?nZ|'
    '#B*{TBb7h$9C|<GR)hOVDRqoOXETO`=WWD(q4K$wwz)|~S<~QiU-YE`ABTf6pc?huXP#@z_G%G^*YOtJG3x!90FZsCil_dpJbQwmuiBQSv!4c5;5#b5WUcq9'
    '+?Yz}C+SzTaMK(?FSI}BOQYI=K^wuJ)64!$n(5t1bbi$-dfUcJ0sn4(mFN2I0p3q@VgBgInHcqcS0<T1($ybH|7>Hk|HI4bim?6|w6MBkyXOIn8+GoL>>Ews'
    'wI|FWy?pv;l#5Rx=$}iI=eG&O^+(cw9Fje|LGdo7{}Zl1>Zo5}*({xl9hpkAj={0~_k633>=j)qFDQ9g{0na}v=<kxO$FYlI9Yq~_FCTBsRaG}sDQh9vpe=3'
    '_4~4KN0{E|Bj`!n*!4M;b;`hQbvk;|6iPh5Y;tfPzpWCiC&>T}qpu7Fpl~|z+-q~c6Zfx@{$quN>zt`f(y(q)zgH}-&g?Q6k)EKwgA|5SXFt+DVou>@c;5Ne'
    'g5-1Cwq64t#Q&Ug$Vjd4Sn!$KLtl)dTjDwtyLtwQtJw|~{*4X=#k<tC7RRP64nz0Tq_Zw#h^3)7zz3UxIxp)99=O=Gi7@|R!BVA~^3-%tg8C+1o=;i+1z+L%'
    'xi{R|%%Vg0K`}l3idz3Dr|->ws+8b+<x8j=)Ddh0t(<i9D{B4Yw>f6HqO8)eel=FJbVevs1-Z_auukmyc>SYDxBt}CMbqIvxK(VTVc+ii12GieIsQ&4_nv?l'
    'j610_BFDC=Uci<yOzt@>yI(6HR~HWspx%f4NLQhILIk~R2VZ+l^t2n;Ge$b|$wesV@15b@b4ymqz<ufGZe?_d7Hh{e8R^XLr1HnZ4Ow?pSt#?#amVV#-Lj#s'
    '635BsBEr7*zdqid+=M?ibzK3aUtbHp?YC}T6!q{tW-`h1VpT=+PxqcUfUJ8ld%HZhC;mG;-!jB{#pr0s1q|7*$9vbjz?V)^@beoiNSfKODi%FwEO0^p&J`Y0'
    ';OB7Q<$!;~4SzI$tYw{|wf)!sJbz@P#~(@WXUg8)hqgS`f%&8QEq0}wPtE}*jE-K<M4>5*INGe=2vYD7kCSb^`$cs8?7{Mxba*GVKY-Hr^M_-ZWZ?S8n>`V&'
    ')1xc_FRIWue`V3p{6~a3zp>X&OegtIVEU*yS##@gAG7ilc-^VG-u?bJtH^VlV7~fZ<vpHXCB+}^JNKe`lcn(M#QU~9<V^RwQP~1IX>}pbLk8qPM*5F(e!03O'
    'Ih6iuc-;DN&mdt-a6Yz^386oh)PU7C1>*3W%5yv7Wkvnc!4}lMkKo?!Z`nEw$MdKukrERAH5dmIjUTA|E{>0+p!9}kPT(#dN?#hh;;P8FC3I*0V{95DO+Jv6'
    'hobhEtA1bQYc!t{ms@c?Y09(&7O{S;mw_gpJ4KEASNr``Z+P=i>I(1<DDRBjoL&010pm%=pBYTbl`o?|VhObIu6M^cf8^X9oBh>#2CjF`!|^P)Gf(W>E$px6'
    '-9fq~%a_`Fx?G8vBT~Qmo|d<$OBmKk!6pU%x~*rxyh-9trmr+dwt_$Sgj!K^#>UB0ih%#=^Q?|b1fK@`k@;0S=ETs)1i#wRgay|Qn%RLK6wX;y6-D3UAk2~P'
    '*xsOba<Bng#ZG4)lQ8A{oj!Npy~r2S@i>7Wgs$#s3ORVN853lub8Z)*<~t2S3hYe9;5c#N#)0zknhMOEHcvWQI^JLIxRF$)cQf9<o3cM8pC&rW<oRNc|3>eG'
    '?u(N(XYSSd7~nR^eCO70-J6u&c3`)eX!d7gzQOPQs~M*kY<k-ufnO(HPs%;(9eZs`0%nfl)embtjxg>MfS=#}veeg=D_wy*>I!F$S*fd3a>47Yo8y1f{z&}0'
    '`Bwkq{pBXiuXgmP3340Ef$?#PdZEEbmUC$T>Z~JJL&aVKf2@hmJZdq$4DDYPx_V}3-!Ey{zshrdb~lsx1+;%vy{2x(PZk;2zq)InW8KcdShRojS(<uuQ70iD'
    'Jk2x=zw|y4?O%Ny;Kh1STLJd37IAM4tse_V`&Z{=^^{9j68x*17fXIzAyS6+uNFUSVP$ocg8i#@DUE${yJOM*)z`CH%g==n{Hw3{j9a(4rlS3;CkJ&q0}Q2L'
    '|7w8U{sZ3x{m}l^L#I|XJXlPa$I!6Nvu=0(fB08JZ}ki>h$Z+}-wSV2RG1x$_OHHo(z~p2R0596x?N8;YRFxo<6ot4Y7+jdtPHmGxF-eY@e6|l20m^$N5Ahb'
    ')xX-|EaFhAG#&P@u3f*z<l)IQG(I}Su5mdtn-$(?`=@Vr?u=iG{X*s00xk>mpRk!mh+AxOB^*_yfRSFlf$Cqi`msL6@Q@6gZ@6wX=3MYC54@KCGat`Dt$)<|'
    'wtkgEm?Ye9v3jgi_7HOvSdEIylE3C(EHoj^3xEIU&BM1w8Q_d0&3kL(F<$@ZT(l<J(1hZ9#Sg%B0bt#>SEsRAlhiGz7w}%G&vgXO5|dk}Zmrl;%GL~oc@X~9'
    'LeGf#x7R4c{J&#Si=sPcD^NuFSL<gjk$>YL3-f<bz0wtmB}aiH>fA29Z_c6VeS|tNdEv7~Y(@#-CX&yY<TlQ_#S5=<i#^3WPtyx~k9=>l;xnGw2*A&8hU4nI'
    'U2*>C{A$+x1?`bcqW@%nauPlUl$aaU@M**O)r*C-GDYhyPTZfIw4Y*Pj`-9j%E0<#)$Gu(^D7D_?oUp_g4vfhUKlf;4#%xVzqdxa*)!4o$-iWo8d%E7!Trh4'
    'c(16KdnEzgpPVyfw$f-T0Usasn9B2R8!02jNkV+J>>c;#l--;DS!NYQzxqc0xc`*e(HrVZ0fzGHd+xM;)N-28x8+j)mhhf61%KDS3tGovBg|XgP)hC>{>R*?'
    'sT96iaWCsq&NUghFYQ;st+rfIkAMpc9lr&YKRPT{Q*^r^2KOV-*qc-4n12sM(9VBqN*m9whG|y}xBFA_Ui!F{(#*DN@A+7OJooNq-OYJlkqdsWxABY4&$w3q'
    '|JeJ|K&ra1?PDg9dCrtjQISw$Q<`X$N@NI0B~3Dvii%2^XC?DY=9#v!jD?gjp-GZxkmm0_XYJ$ceLC&W`TIWa`SJX4p4HxG@3q#wuIs+;>xPJP2(u-~-_a$I'
    '^*J0TJ+HG}Li#uZCoC%_t)C|GNstxZll*9G@lEUmCLTZG)BK{jqZS?{Q1$QH@jtkl93R~({q%M*B0L*F4sIpCZAuFw;37J4ZgLkhPq0T0w3yCuwrZ*s%~C1w'
    'Al`_<7j}`aRo|m|gIMk-i_&BYVWvei{0PbQ9UGb0rYqRR(0+vTiZ?$|Sd<HU%2VelWUB~deGaSAyod#^;^=-h-K(BIe|{hto)s^PdheT#@*a0TXb)@2<BX6F'
    'G^zWr3KKs%o<rVOhWg-3jM#WwWzq!ntHgLU_r(1??wd%wItloVLnARVOkL1*0(L#<7S&SpIY*z=G_tQkc-gn$-&L;Q&ucpi<IXd=Uu_OLWf(ug^oq0#Y1Kuu'
    'TkoiGhtn_Uj<>|Z-|*txpugUiwF#K`C;(mVn)llS-w(Z-&?NHu7&|(zB{J^(h0t-4<cI#xkHa0ljPZH@&HN*AZpcXLwXc4xf{u?)WxLnKH7ERAze=2ofwz7X'
    '?ED~&)~_BBG^m-<jw0z-F?j_EFVO#yxF+v=d%p9$0P>H7U0|Q&kJhNEwT5^F@bbj__*oC=V*FR_{C4sxFUp4p<Y}C<$^HP$n*s{tyrj_Q?1jo!wI-Z#u$u%n'
    '4s;K30DTVe=9P|CG=F48;^P<6_ep~smhZMF!aF5s)P<1!bFQ4Ni#ES0j;;$CicWM;yi^03F;T@U+c-*pE8g!5bAK#Ebm_N1el;_Y?da9VkPRc1t|#XhMZap('
    '^)Y^(8RGj1{KCv^`h)j>Jp~Wrt<15e9jf^!Fn%Ztg5DeO?j3+Usp3`Dcs<2mZeyxK<F|Biz*~Zy!_8JkOsY8<^2WPfPdBH{yfh5|Si=4+StITy^aF3)Rk2N$'
    '#?!*+aSlm2j8TVt;c2{a=?zt4{Z$yfti?r-wc(oz;3*O8`E78)drExt@iK28yNl>az@4qI9L?Q6a1yQ-!MmO@PvxA=nEHUa4|~3j9m#;`^+VH0DC3`M%7PT>'
    'JWqAJ_^>j)traHjcYo#Q$oTQuO@Wpc9Ymi4@^kFB=cptu@`J<haBhn#%ia-7Ui7)IKf5jD>|h7<i17D0Yz9AWXZdr!XG{UDdyRfS9J0J1cXrPO6$g&wbPV3<'
    'af74qMr1CH`SAKN3p#$xTl>pmcUd65zB9J>O2zOGO#Oqu<J>4^t_D(m4s&i0h@;>~<BR&rzP~|!jsSj9%q>-Y^V?o<65f5T64QDWD$RwyUVmuV=R@mSp%PXa'
    'eOt)>s|-!&xz|2H>c>If7IRJB2m8}9pjq*xX#Z-<5`!&)SD^E(^o{$^<R6LecW;m9n%|kr(fsPxhwL?u+v4%_k9V$n=R6meLFXSstdg!juQDU$AF)$$kdl84'
    'xofBM?W730zT@=0H%2;#jbK5%1D8;wi{Yz_nEYczf=v0dpPrC9-nl7WTG;YYE<bvld&3>Iq^Md_{t=BI*|+}&_>nmGJ70tcUUd`)_xTIx1Gpt4cSzL3UVK#b'
    '78w(|r2fdn{lN3a=fO@(L|<MHbPO%?xd<yB!<%U3E!mOtxzADGEBo6JJ{ROIcL#ZXx^^^%)V~8$hoaUE^f{z{#VYRZEQ$y(3lyZUJ5S^`_clXu)IMCf-2gul'
    '_sTgjR(D?pAUYc#u#YgF%nu7LhFqxh4<6qt0DdIid{K>+u<Q+45RXA-wBpDqb{mgw=nsKLewCbm^omgsbksroWxz*}>(q1Qpg4F9o%z5?ql_B;^p{XAfr{7D'
    '^#}cti7&Oa{A*pUw+!&>0U1C?PrWloA}{0ng<aQ~SCmpIjP7&nqnm3Px^xQKMWEtE{%7Bs@y}5VEL%Exa0Tdd90Sf<jv`Yo{MEDTtBC4X>~t{l?J<G#v-AB%'
    'jeFlWYV*3}4(y6ILU!IA{k5n5pmibV*EzI*yp%sXoQsO%ia{^GZMMA)<LacZ-`-uwtHz3sS0lWM!u7YqU=BPUk;nUdNcMFebUY3SzHzFbB^B1DisOU7Bq;jT'
    'FOOMEK7RvzH30~R*WTA|6gr~=d$7}Y-gssZ_#!|%4&O(W#yF9Ww(q%xG9Xnt<JIP*v~{bGH_*=KlH;RNrpTab;(T;Ix8&`^s*l^eN%`D=f9(DkoWT0$sNM+s'
    '(7<@>|MHJ``c-wSqs?Pn%h33-T|0$+@<ROI_|dP1f*<D<#CwUHLi&(B2e3wTUfAdOK)6~Rr@xiQdY_$5!lLN-XuM3W@A=vUI7O8Tw**vC_|@-=3<ax$fsYDd'
    '4~R?8BK3B3nL++|^@vZ0twf%sV*1FK@7J*(O(}szq^R<b7L6ePNb*B7c^}@k8R_5Q1Yn}sg;LJgcDRn6`u7dOr1(hu#|A1oo*iI8_#JS;sO3Y0n{~_K9_sTe'
    'u^%(Jz9XLLkLt27G9X@o_pkxxlQG=V1(gw~-a}g;#eemT#hCh-BEkoPxcXt#rE#s^3D^#A#!rd=ZhtqygT{|(!P|x>*wP?lym2E;x6TH4@}tK++}&r$Qg{uP'
    '!aJWczYQwd_!85v_}*vEDXz?L_#NK)hU1XRk1YZi-Q5xso=si`5hOob3{O#Y3Fvc(^WjRqO`dfiQg;u0)Z>g7U09ho43$t_!ejeKfFFtRUs~iFn{@-?Qw?UL'
    'VDwY(y<(3^_!#pZS|Ta&(GhYIKeGnG@u2>pQC<31LVg0Yi%x!3;+uTSdBw=t^;JNPlTJ2T{zRjAcKb5wJ+K{|;9Ic77QT(AKh_wQUAXrwrk<3ct0Md3-Wo`Z'
    'Ry`saKfdJpwR)gP0$q<7#Xy+bt#Sq$!F%t~`?YpyEWqStPcqGqKFE<ayWfWzXZ)f)GeEo*R%OTTFMiAU^k(=~iMwTfOqR&L1o~7Ju$Heoa=;HhfERc8iY6!B'
    '*)Zq!D-VY*vmR`N+6h$on=P#reGX4;{Du?@<Qxt7H>peWrA)Y@|LXUG`B*$WbJx=blP?+%;(aPpn*gt(^SNXlt1rl6xP-4?eVS%|`~sKgf42TH?2v>D-*F{0'
    'ezcr_xHk4xEWSQx@%Sslz-no<K8JZQ{n>7Y@_*}dh-)jk;meW}((}>!oPz^#?>9@vlk_>5^EuInn;ak4uQL@CE1M7I6Yx29+03dJ(&PX!;PC_VCz`{y2MM9)'
    'Q-;>AeIHum;2n4~sj7~tC{Oqc+V4K_^g@<s!5)&Hgt~thY2gt!iQjzM(Nd9s2S)mLo#5R4Fv4Yw<fo0nj{zQlUnR!66B=%1{09-A*qx-WMY;CHm=dl+v#Li#'
    '_r<a`Ce%3>%#Oc8jZ@gcmH1S-2~uI8iUWg5l>FoL`(U;KN8~<(0T1Om9J=J8E`Gjjzn~()(-l)kot-SaD&SN)v=eWfccQ7DJ1;kSoL%`=c#(bfzx6r9KXO;c'
    'nkos3p!3|R`{K4oNn4WiIseXEO1>&PCxG#@Yt6mn;d`cpq?f_q_}W*L{*RBQq)iC2NF0a<G#r1tNZp@0xR!>m`HVk+eUr11a|PmS3iKTlsg+*Gw2r{P@cN;}'
    '&CSm=8piZP`!IKI@><vzzK=KV#NmN2+vUa3<6^%jX-W$R!8%m&QJ0lNZ>IinTWOlZ%f|>W3lbqAJ68tDs%Jq}T-58fn&Q9e_Q2xM_+3ON{Q~eXyyC{2IE|qD'
    'b8+r-)6x+81BWs7f#$E&+%lR5q3=}rW2@BX6#txKW4AxC3nBVy@E-j43-1f;@P!TOJU2|JRhFGPmIP(fIZn<jF8y^(BgrQo<MVfSj^0dsoGkyns;6xUcrKs8'
    'IWzl(+Q8*7crISO)3pUc_Kjhf{A0M&r~5IcZqO;b{Qu{K{F;XjJm_&NMq)pt%Ur_O=X`pqELU42{GZG}65mz8h`js3`Ag9Fkwesx|Ngc(5`O$k-$BNY){M)}'
    'p9z%!@e%BdoHkbzPt6a2%{8#|cbg86-i&|FinCUkPTS?t=d`IO`~KXR4uPdLsP^T;2OvHY*P1@Bs^Px1NFEX7&$sj~5+{7R2(zfs$On?^9|tD{r{`oLdIXRU'
    'T-D3Fpr7$1l!~{HC^Jna42od(^JAaf|8$&5h0PUd<U7gr9ak^xOv(R%^l<?BPScD{F7+keu#_~7z60zC{s=$icFe6A!7ZTgfHXnHe$lRQI6{gBZ%M9yBqSU-'
    '2&o~yU&!`5E?oL$;WOBrfy!^8E`_3B9W$5y`AbC#@SVW1!*7DS@3l=l`~hz#u)6cqq=1DPoex}fIq5uuN(ZTK3xns%V^RL(OXRs8NiP6=CpaN9>g~;!ycRRN'
    '`Ue!>DHWM=(<~?lqIcZ(HLL@DZ7#4DRb42-qF8rEzpCQTdc-vo;emmV`fl;Y%hlRfptpGVxP8IMBZG4y=;zXuVJ}6fv4xF9amMWv4HN9(7eJ3Yz9RLUQd2y1'
    '74MhH3`sV146~#0cw+pS<ENO~U>$a<b$hCu(!V25<f~0>J)&EF2Jqa1e!n4OKe!VQN0ey}T-5U5L0`|FRdnN=o6*n_y!%Y}S|oHn9Fvz-I5=f9CoB*a$HR}G'
    'i`1trA_dXsTGu4LN&-hVq)jJpxxnjdec{O#IE?LY-e9Kw(dVl%H%Anriva(l>#fiH+&R8bD&Cdm*EaXVn*`A3DPs=Xs$bXr;BmaTV;6qnXO|nMUgoCk-K0DG'
    '^?&C(mwQn1kIxL3`<!M4cn;~?F5waur<fNG>(R-RKAnI4df48}kObTR@vHyLKOQUXkUYt}2#p_;{a)?7GZsV2KmK(-XGG5DlKR)yH8CPQI1*AAd+v1~&V>}_'
    'Qtit-Tq*S(a<y_DLvhGF1j3pufl4R&_ZMK8ivJ+&K<Yb)dxszITgf^>akS2+K+I|4{%#m*pjv0Tp{#l{@$uG=fkz6#NPGmlhUVIN3heO?@B=)Z*a3&?<jN*Y'
    '{$oXmLXxP$W0HoKx^)yOJ`(p-QRil<J>3Yehrq;|O8ZUX5v0Du^ym=|-QCGQ?CuAjr&Gu9M{CpSMz<RHD;4iWsIsmikmKXxJi~?T3lTg3-aET`favJu7|0aw'
    'N)n-mqtm}&@Z7bH%iGT-UV)=={jIzuG^EQgy57-0CM<3J4<UQ1x)<r}Hr*M2Ij^*mNV(sLUpUAdDsjrEy%LLnit(Q3Z-?GXU!^eo-0Qn<D~I=}Lb-V3)(R(g'
    'm`ct;k7GaCdj3pP4%|mY?2&wM$o>Fg<-NN@%aFQWa9XqI;~JSr*a+JQ)9?!?<HsV6&d|pU2rmNs!mU;Zh)fw9z^Otw_X$}_`9#!;qVID<fOebf;vuM<fITmI'
    'xaR53)ORck_?cY#RUEu`pvbL%>$&4xeipQzPQD>8)#qDU$8pFRZ^U5SgWdxg^U&}8qFGh_+rApuRh)*7NC;Wp7Z%6x1nol3!$3#W6)w1rxxy3H6~p8Ib#lKu'
    'zQw%v74_HnB}_XYO&Wd#Gx0G%)g|4G0ny_DKY|}~;2ZO5<4L;Le}3k!I-9_D4&!Gd@UhTUO!h2{i#wm3KE&r_kHPWFk7Z9yDA(iX9|vv|j!c{uN9P|$4)*3J'
    '+`5dPf2{C#Ily^A2%Ud4-56n?JN|$5kN?Dv#P!nYa@0cci2@owN?eq#jMIrF;m5yxX~_OL3wfGT?8cFE4EWc-$ChsLSalor$9vKuO<aDPB{A>mteSvukaZMn'
    'tApd8;}R86ZaXH0_RrC>^o~AnQvy{{<?E%R=IhP$N469AaQQ<s!h3<r@ZNf7+eImkkPDt(uPimGAY>bcmn~?EdDdo{0&S*~=jXojZnv*z7M!R?19y`3tA^V='
    '=GE?Bh>rjB-@aKLd?*+mlc)Bj6R4WeukI?{m1$Fo_+o<qm885*q@^Ymrgy(PUEI3B;PZ7jMV^LV51C(0Vh~**Tm^VNq)tpjjUhjzSOxyH5a&uU>8!O{{utlA'
    '4|{?ai8aT<1$cK@P%dYuw*NPT8YkHL`1hwzEhIl1>~m4An&DS@{8ikXs*wE-%<G3PcNpVer$S5Vd@c<Ms!@G6AHhEV^3Ne|y88@0O7;09(a&Y2<L&-ER|4QP'
    'Jf7mF&_|{P-54J6cBgDc2B8odqUA3`_Rr}QJ$%U_3aNVm`?;1*?c2tP0Q|Z_*^|a{kBaA^@!WEy#oIoX2jly%aEOh$mu<lKy+t*;ypnxbOTs(6)Oj|tKfr3M'
    'x^b}!pgsWMMIw5AJy`7SL&~&tq2%~Dt=seLup5#O1epTkL-$KgS9#;>f$H5-mG7Hzp>f&W=_<vz8_keEE7f`NYXT)cwwUueUnoWD#{d!=WXT)7)t?AC;ax9k'
    '!=JY{{g`~<#)d!fldSh4aaI~}j?Aw%6^`s*eNqHnhib@bz!+_D1sY<ZZ+`mLmKAQ!kEGkDG27=_oMv}4y!`*-BklS};@XN`>t<$kR~{W7yPV|ZUlt{h>K|#w'
    'N8&TK=N!%*VG%~hN9m(R+xq_ulHwy~-x4`hHB;a5R9t5y>?DYf$hc849kwG!O`uun7*X?{-)orD;#w-ai_URhGM;%_T{ngI;~kvdYo)~eCSc-t(~&EeV;|ll'
    ')eB+zBigot{3CI%+-@M=ML_C>2!MG~<K#K7ezqObV4~Vby7P5s{N)DJjvr=ZMD_zv(fKhpd@X}`Cp4Fds@~h(mr~#HGuwz;m<_?z;0MVztT`?tvJINRdm;;%'
    'dzSP)#Q1PT%1_!)r}je<H1P3EeTUuFl~$gEGH9Ql^Hr>0O!p^}d}9Ck@z_%r{`JaS^m+MVmMN@k+Dh`b#$2!JJf;3|i1p0LW@7}8gO2k~eXb|B1V+zJkAR9h'
    '`&Aw0m8FM3^YG$FX*~<)#^V_OZf)W2v{>(GcnO{JOk8G@VYWdAw3LlX2X?vx_{$Mv!FcnbD+0?9J_S_Btb4O+?Awo4n7I33_?xOAN_@P~Zjf1<gygwx!8xWX'
    'naROE9B#q8!(PIpyy_7F^m}Jk8DA;Ap@G!D3v-_rs&r@SAK4D>-Xm{`_zHmQ@hqFCkh~*>?mCo+M|WI=IlSpUzkR+Qtp1v@68ge|U4MhwU6lMIr*2aG6E5UD'
    'd<lHtt#_&NMHn=Occmk;2Nv(t$2^y|TJx{Ui_;+vDjyMDvFD(^gBbs>?2u`>Ifd|3>EOJ0>bc>n$1U&@YMnw(G$lR?ohh9gF^uuA+R~$NqN33pevWsa`tB=?'
    'cJIQ>&xzX^Ex`tf@D@7P!-LBXZ9dHI_`X7Azc%syYQy*n|7Y|$#Q7vO`fcIX1q;#qs^00UOG2xnN%|bjz9rzJI};xZc8G^)4k7tR3)0t<h1Kszq!OUj!qoNb'
    'sQM1Esh2T}mkXiwtK)q~rH(FCh3@0kKc3}kTRCS4!}D*s=-4au(;ITa%S-FW{^{XgiK*|<I<xcHIiCVpLXj$75s)(jaX&~49r?K?9i#7X>E^89_pX9AFj3`6'
    '1N=dJB*r?mxZ<l%b|dxrz&R~&?ftEf7(HQKynXrVRS(AnojmCExw1h->*MaH5Gz$ZG-2Q|(616-PGsXMJKGBY=OX<(K1;mLvh#WY$y3*9lvPvw0S-*5^lrY0'
    '_?m#uKRInt=eM}GLj%-0B0?mk{*hblnY&*jz=23TiO(TVwzWTQ!76m>7^KqN$}$cQLRkbVeFu>@Aje1MlFb<$Ba$G$x(`4E)n3mP8mC}fovpuZkF;SGrq9SP'
    'g{uO48YZEARQe8rT@t1K(P4p7*v=*-FAF*_+>+2r6xIx)+n28LXz$0Idu6k$6Gx3RJYMysXf$tjzMrW5F7J)M`J19NlFuI|uF6JG_*M0~++!y`1D=}zn9=!h'
    '73!yZV_|yh5k17mb2eApgfejR)n}?U&)q~o@8@Po>Mz+}mcdWusMcSsEd@X3MHFlpS|<e75y&_eo?`qF%^nY5!n>Y;yopz3F@_)S9nw|1TXGsUmctoWw7&e8'
    'AQ$Gm=sni4Zo}<bl70odbeVN$`Xkpbl-uzkSs1NTBy4%9=$hmLS>fd$IS;b$|Mmpq7rwB^$NKcjVJM0~eLWPv`dfIr>grWU|0ASNL!7pqSzH=)2=BR+W!KBp'
    'T;M_PgM;-;470|)W~Y<>Uta(CPk!~9s={|Td;uCivY+Lb+;k%P-})TlUg-sCC2;%@M(cCtuk|`&H~;#-^*O{pntu}flGZGX*5{PAgqHTp2L4;0L#p+E8Zjwd'
    'B!<@K=toA&u7sD9^f~|hL{>)GN6BOI{H&bkYeG+SlJo(Xeg!*+K>Z`}kM>2K*>DvYM*xhDpqHA}EBSOdj9T|%<wc2)(IpIW_C3hC5Ez7&?LQawvL$17b#SPC'
    'B(03aQcU0~d=xLPejF4O_g%`3-hb+R=O?_8iGjWF#xZkT_B1jepwETzu$iRBXa?Q-2W|rul~=zSN%{v?Y8??-pVJ|t)wWk#41M0Pz0r|ib0!{Uz`M^AQ;J(z'
    'PB5X@wQWo_d+DDlIFyxYfBlw0>5n|}Xi@w<Q$#;s4D?k2?%a=Rsv&)<z8}&Sqm=kK%}}Q3poH+Upng?srMK@zv(@lNy!&)c63|)u9COb0TWxM%F_r=u;XTPa'
    'OOH$j@5YQ1ICR}rx9%~#lbPCAD4x>)k=1YoWV{0LKLQG8rOG`g--(1nCV1BiFj;fO*a%bSZ&Q*q;-wh^f5XEOdzV&J-P?%qC$nas^QxD%2A^N$8E|)Ej}%1n'
    't3Nl0cI|Vo!{=A86ggC1dMAwLS1UHwFK`jPiqEfVZ-pY$U(Z3$mmz|hLdwNPe15f9ll{xyW@$9Pda9TyU`<Z~KEE2!ZmaOy6vMA_d8WSnxvh@Gud-0bRWg3e'
    'JoBU4JYN{i|4*^3d+sH28S0{wUp0QC)e&G40ne&07HXWUx2747{O$kaSMlmQipF!dXkL{?<HsDn5aqKrxA6O0MMXamUS21P_Lnm{TKC6gB=z6@t%&i%-C}K8'
    'Ig>DYpJa0w6M4y<{_p-)#6{G8^ZIfQHc51UEAhbjpWRO0{C9sV;%2ROW!sj+WIrw7_}qQRzy0_T)xY{%VLsvcTQ{f6VDj#2r?x-6cR7`Ao?j#Bd4&~U0I46-'
    'KR*#232}X&F}eb#7|tzg{0eCIM<Vm9i7(ISt67Vn`y+XXu2(f?-j3fNsj_Q@i1Ss<x(+OUYx?JOEvY{e26yiH3i6M{y(>90Mf~X!<aq-9V{p+ZRCcBr&Z5;H'
    'X)c1tb^N#IEaXr`_eVOZWu`17TLkZ%Lv=19)(4RBqrUR<bn$PZXuaiGpVv8G*1Ex`@$$za+jnvb3S;_I91(l^g-L9PgpV=vQzV(9UtMP|vfq~<sbd6n{;wPk'
    'Msl^r!pgIG?<l<b;@172n{vjXTq?Z`!M#v-roWYsO3ZS3eb8?jsarbG{@QJwddck0DJa}oE-WTr%iB)c&oSrPt?3l}*#7)ht1<6<pd$vE-^8MmCaJIUXUCJG'
    '^uWu{yfu5WIeK=y9%`IsfA-RJ%TRbf-uoL>uPG~Q$J8V2$S_pVcySeKral)nq*46kF8Egy%Fiwa^9*EEy+$|hp8sGJ3L{XB3kjmc$5^A^@2lr4f_Vmf1-+I^'
    'l%I1sGCREiYM!;*m_HqnEP}SuInF%s!rr(QC*YNMxU-eX?Cf4WOdg-fxj>wc*#*D9Rr{Tt8+N4dqVcN64KLmeN6zEdcXWh9(N2m&|JnXl0r_p>n_f$!@ndmn'
    '@9W)GxBk-Skm~ix{3?60bSD=Ncy36&e#(N45*5X9P`VI}yd{}m9XV%cJ-AB(oqsgty*Urkh{D(P*m_#uxuS)s!+lfj^?_hw3yI_D{HB^Mbxn^6qsPVE7O%bb'
    'CKukUNR@AiZUFuO#C6*umc+f@e=)dE;Eb;wd*gSebQ`>94wXNtu4UB>eykav{%G|M;}hG~^lbUA{h71#Cqc!l6E(>X%@*H;mGJ8I2yw?f7bs!+0N&Z9a54Gy'
    '75LM9?EgusXPbirhyHM`Ib6bsE))>k;UD<eJmw}sY=YSQafl6z-i$uyv{`FT$xZ}cje}6z5}%^?-3W5U<HsQew%x|EnDfT;{e31fHm@Lm0(QRaSj4I~!>?8z'
    'Ss%a68p%I`>#Z5$-m}-eWOmOJwU1==2&-HC@q!NGjqp5D{=Dn}GkX7Vk^1R(@lYc?#!hv9Q)8p(b8-)xCJyN#{jESf+qq+$b#}Tza4p{c)3Rmi&+1*w=;!jC'
    '>Bpk!XZN5*H2M{j{y;pRDh_;5`kX6=_HBFj<%MC5&Je^*owppRr}TfcdG61)K}H7cGpf2$^4x~|?WFuM#<%rFHOM~_V~_7j-_AvwKwl-KZ{&c$TA|Nh-a~t+'
    'ep0Lw55*r~bFx87^L(UU1_)0N8H*O?_ZLA*bjJT9HesuLtk1%W=^WP^rLwwf&wcm;)w#%{rGgS4OWq5+t;!bz_t{JOeqOhWtwu-!-Tb(V3tw{Tr&xG5o#Wzn'
    '&0F6YS1>ys9~D=B1r(JQ<=lW5;_(|QbIRZB-h+v&%gd5pJnBA+?+?(i`kGAr7ykdGKLBy=-&j1qI$^Oi8bA6j3{?uvi2Qf{k+`RBTgRJoC{zTUf2`9GZghdn'
    'N%_ZregaiGza8Pi@a|cvVc%Nv@<{nW%sG3@Gr+GB*S75v{#B)Ck-X(saK5|aFc=k+4PTe1kw;`f7P~=1cB9o7A+&$D9REGDrQ$Wv3R-zYvOl?E)u*@ZQb_#+'
    '$Rkz?4edN_a}n0Yn}^WS)3+`~V)&~?hRlly%3aVD1C2Z)**|C8viX~*F_ISodBo>qd@VfpUqJI|<q^sFagCz#&ZSXEUuBR-+^acsa<RJ~)P=Xg)@;?kL3quN'
    '##b%ENmosE^62gZcqjAV>U9peP!FxV7nxr@)s?|8kOKUsk-XQ&kvF0IGO@54o!3jzA*fA%@P%;wEl=yr8N4Kd>BlS(w8*bnq6prz02}{*8}aMS_~+QAY_<-~'
    '209JIf7P2?)mf;$1WuO4UV@~PCdbG1Sxe(N3=mx}*x_L3rqq1~Ct-aVocp{c&Ya63#E0Iu*2`%;H>i(>t?}mB$es;3m4O(a(ZvQVnr%EGP!8U68DF$5el8Oi'
    'dVbnxzB$8h7YPUB^7z}K+voRT^7u!lmfhLDw3>uJFzbkrPT^N~w5IT}=_2RoMDRTKGE4LBY=VSYX!uZ&`#)w%x~X&3A^l!~55<J_&N0K}YRH3`DqpRetvl2I'
    'aofwdKg^PX=)O@MU(#T|)Ou(eGnLLw+MN;~W%Ingp7lcBJ1FGoxc^8a$0rn?6^=(e&&gX_osftZ#LG<IxGLXnpO?M2V0!(5QcO!$?n;V>!tqA%G|Kp|yo1R<'
    'Cay9T4v}<(U*Ns>ALSBd_CA>X`ah#zCI0c`M{5~zLrFA#EIi$;&HHj6Nx%Bf&tk<UA9iQVxgvi;F}Omvl(xQ;T>p5~U$Dl#S^%x@Ty$<Im8bt9ZG9&*5<gP8'
    'Cg+GZ3!?R%i#F{2+4rKIw!V|hucoqf9`UvlM(aD7-wP=+K}+%VonwmniQ7{!`TB>Ettxu&dT8rA$@nqnYOPYLg%n!fDW$@5ysIr9U*9P!e~i!H0i*AHJ?UE6'
    'TGB{c-${;-j7#~Ydr!-v^_@zNOpKf4>DPCP^RtvJeH4VR@6;?FNlvxL=;wavUoC&<a+_{_r)SOEE`=NMwDp}_$oWktXQf!Zlqg!?>7Lm4&P4ekZG9&>|F}A1'
    '+3F`oB4~YQTTy@5otl1<z7u`kuo`Hm<R3l5jpL;Yk$uvG^nGge_NEouH{fWzUuM0ob@T9L%(>{-Pm2*Z%?x-Mo%<`>(W}a1MfJ4xq~!eLVvZmaoncY5o-{xH'
    '@ZGqtNp$N;LlzhLo!Upgo^<`{o@L`*#kBRL#P~>zJ@2*|#GcL)LF-BHg>yYQEznESlVbBQak?}0kKe=A`1c=0-a`;bK^!+d2<hL3s_48=lh|&rny)^v5}o6$'
    '!lv&HRK(%yS09cgY@WMK1g&4yl8XElxvUjmzdAitT^^JqhSsn8q;dUWePoTVUu{s*Iqmft!#m6AKd#|#NT6H4YJV_>iM`teU%z@#{|J=D`@delig`~&-!=wh'
    '@x6i0J(=qy0pcUz>)*?mOm%*^1-(_F!OJotv0!12=a1j*66pNnE)k;*Lyn8!H%q8+lJ*3}Uv5E-uX{xX;x7l3=r46Y+pMbyge(-Xyru5hZxnrw0>dR6QRxNf'
    'I_eXVAJPJLt%ua{HiYxH%GlTxF}m$#``qR)=a^-_1vRc_`<*2LZ++nFc=PF{f?>v{IT$|o^21{XYZ~hyMpo=Pi+%Wsf**gCo7TCsAaMX7zZDNTuM<4(LGCOx'
    '>Zr)^v48!`RoRo6ylhP=t1?^l1!xOiK0<EI(OZhm7~Rqb(@oHA{rK7C>rvwdEwrr1!(#CJcQ6}{u5tCp=q7ySrmX@tmcfSvX!NP(LiX3^989_fs}Y}H(5Lzi'
    'i^{aGX)Anq4h_F|Ze)KQ5*N`v!;9$rfV0@WzAmHg!v&BZEx-492p=ePC5$g&66voC^tp1}Eost(cBquf@4ZUB2l&g8`ak(kbK1N|{N-<uzK^<*P^lDE4F%Hj'
    'd*?;s|BBxUTjbXuxD)ujKj2W{mY(c_<}%RmUnj@MN{?Mvqo)zx05}OT{|waUyRaXQqZ5zE#3gGdcVv+K59t|KEhy-`Xnrv4g*Rg7)T&%XM-0y8SG}(|c)N)1'
    'IBqrCnKr&U7=D5`!p!&D`o(WC=UO}V+UTyFAh-{YM=W^Cu~YL6245*kL=^B2dqBTAu<u>-+eb=#WIC<md4F64y`P<veLEtQFbvfY{?2a@$nmjA-0HabWqEXd'
    'W3Bsx<E`;I&=xw^-vjdvoh)DcU|gRDo+geS9m~&Xzs#Fo3VP1zn!s!D^1w&MU~6*-Q#bIR@y{W?U)#^iv^}cD(ERF~tNx>}-NH%!Ie+zWB*(`~tbWfHl6e#G'
    'd3TD>TunP0TE|7B|13HG_)fY?XVtU_x-YozxqXu#m2N|!c<=Xh``Q7+Zp?d%{&{hFV{Za<m)fs~z*9AYAKQ#Jh^H?^&aVK}nY+y=2!!2%_v6mM8#_+U|IC42'
    'XD`~7Ipx0EK}LAKkxt&csfWaQ(c}Dre*WlqlK~S8ApYv(NbcX^ymhQpJbeLLx3v1(_pgI;PheYGeH_XCJ1Tk{kEvH8{4vNw36J~Q%EUc_uhHt`NcPY9t-=5J'
    'whWRl1C^&Om#iO%J_v<3<LQWslucg`ier3*hQ5mPd#blX5X0Yf6Qqeij6LFsW%qu*M&badn>aMK?#b&Xk71|3aKu?keFvL-)_u2)h#wlz6?P@}UA7XBpWXh7'
    '+RqCbB}PBNsk1w0qsFZh(kxNC_zG$!(CAZ1=2uq+?OGzVO$^<qa`)iL>KyB$+3A*1*E`7B`bS);1B#{kpQ?1`=*{5AhWlR|7;hmuA|S;2<@Up~o;eFzN2iWR'
    'u;Xh)e9mpyoX&AO?j1c7CzAsA<BbpyQplT(oIvl(o07R$IEP|jGi4e&J~F@hWkt+|%_gGgyoiMk^h->t7P?R6w^a43O?Re$M_F#Bdi?;R3kCfjw=4{5*DLae'
    'l<?+t(lYtmk@qqB+@Bj&4kg^og`*g#@Z;-dN`I>`f2&0clM!DC7t;6NT7&KiuT6w4aG%Swi>$Xxr_sI_gC1hZW3P`xYw-A+V+)MG$VFrLn^l)X*v!A%!J&AG'
    '($Pb|cV75~(c=ktr%c{`?@IFP`R7MiC-r7qrNIC8{NrDJsu+-b3!nRE8xvDe5FbIGDx<yx&rp^qNL%$E_>uUJt-NVbFuez(U)^D5UGwM2a(JcEKky@Q^K|S='
    'T0Gq=j6QFbPM*2Cq|qPpQ^e+lq%VA;;Kwx&rjKl{M{rUz>HCvU*gZK7RiUd2c=Kt#0*_HQ1}ELySuyO}F$l3y&BLm9jCwQm9fP@tqMa)co)oO&XKxhE1dZN8'
    '_XyZHa3bgprM}~8$ywPx_Jv@*0RJwz)FfW*YDaihIylt4|NL0>!0mw?=nO6oT>QQFbw9>`MZ|5_9*v&J*<BCqpEoi1eaC1koWg|teGSc%l={aA`)vUy79jgp'
    'E0{Iar$Qdv#lf%ft|Zaj-MZokJErddSL%*mg{R?aynX9g)57k<wHW+wt&lLM+^rU3XQiRDCG)GcY_g#qRmeFE=w9RNRF7u#RzfMX@Hm-Yt@thwk)9-h#^bTW'
    '+p>)N&qFSFc)YA7B-;Eu#&7AO#HI)8r{n0x<C#lYtp?rWpaZykuFB=<^@JY~>b#irZuGjt;!>EO79QtE&ckd|i$|S!#nE^?U00moxTXbOLko`!An|I~m_Tz{'
    'nH(CAFT9miw0uVqbeWNcKQLLJb2`1!YHv3t|KAQhNj<qc4mQL4eX2IGyiX**qvL8~LsMaC=L1NQ3jga?J*4Cx4-eGe+2f4p#{h5XX!9n~vFjP6NZp6~2ui;?'
    'hjBW>p9iVS0w)^vN0IaPFL0in&IYyq-YwFbO#GMtJ*IP<x#8&ci8Wp@0WUtvwZ+(dJH?NV<DR<nl5bcC!KHX{ZtiHcz+@x_A6uyna;<-$4t3$-#~*k3Uv*ar'
    '>HZh%JBV{*6KjF!r($6=e*Cpst!f)b*uVRCko?If+QXaq1<`e}ZM%J|Bw~vG?%%OC4#Y>|;(OW}e|)S8>8l9x5g!`F9>{2hLAL+u-$C5(B-iYNUTqae_wRTx'
    'Z@<*i$qYz<YM&%{9HQhO9h;dSDflDjNnn7sh;i+ueOrT}&v-NU)BQ6(vIdy*q(oo6#olZ$Sehyy;lWr<iI1ua3cjCMisS=<QZnbLb>X`AzHkBFyr1v2pzfZp'
    '5IW9%4|1#J++PJbP@R_vN1H+Zk$4j-A70zW%|q}Ns1QD5I&#OX=ovIgeGS`2-5LMY$VTB~$Ct>Wea(jt6^+J=c*7g%#1S?=bq}uEB|zzTBa}Aoma#s^j;`0w'
    '==1!o#p?y1W2AcTqalD_CH_%Iqnc&8B7&3LNMAc>{*)U$a3AiZ&dV-$(4Fy@J9gIPn%;GBbe;cVHqjUp*$~KbHop(0D>QKCFXE01fX^_{H?CjCknx&a7?grH'
    '!t>ertu_II=;!ihQ}3sZ0so)=G6TP~Z+cyrLdOq_{k6;FM7p8t)OoMLBuafp%-#h7ue%YySD+Y|Wn0WOxiOt?UXLeG=xq0mCP<q)54=B!(jWQL$GatF>k+?K'
    '8*qMu@8<1IbH&f+j>MEM|I&)_ZGFk$ofc+&=Pz81`512>dnk~Ii60BJRwiaBTa)^|{PPnO8ok<&AEQILHD$H_#N|qSe*iHPC11`!%s%K6dGK_t;R$?yfXP#_'
    '{fCAy^%i#y1Pkm<7{vDnFj^mS>AkBg+8>}N>%&G($y9uQfWhA%-9<%t(ed#@*o(QJTdw?{Kfs2+>#t_~<&?WQ_xU6Wf%pjG4%62S3SI`G&<b@LzRG0%>ebd~'
    'lb#9)egyfz^Pi%Nd$V<*NW2Gr(dolEk*}EZ$VE4J$HGe;P!#n(Nf4p*f7A+Yf7NmU$<Kj_&ZXdL_k|^WcD~A}_p4T<QF-vyC1^JuZsA@pWK;AQlSeeX7npX-'
    'v>cw#gT=W5hnXn-JG$-dZ(UYFbiMCL-*0vEm+I+ifH!ke?dK+*p#LLrPu;Lqs%|m|!8@R${CV+6%)!aS&^EmL982-Ol66}MeV=kd#eG#Q9q<qKzxlZt{~Y<Q'
    'u6H~q5WfY$&s~S_$2^&=g`ZH@r7OY|{K$T(-)=ky(IWr{8?i4p$`x&H!%BGZ!%BlW!@h|Dy>E#N6mA=3O@`B$sLnUPVG6(c@NHwh-E~oL-T;hMZbUg>Y)m|<'
    'U&z1X@y<d*Wi+N=;leqB19P6<gSDu2%LD;R{o~6mk;$%BlHj}noLYMewlTJB568!IG<edpUrd)qtnz_-8R;A6<FzeRI`|eWkEegEJ~`Ipd<nyg)IweM6HD(x'
    'oz#5%iDD2RiTi7c@G19Ib0xt%drJB`P<d&ePkJXL!btsG(kT9NVLXDKcR57S{kvRjpDvv+%bJ~^E$Tj>rZ-1e1~x*YRJ^6rVrJk^PK-Uho>C8VZy<hl9|1mI'
    'yJGmaTL|n<9gqL#_s#ID?PZtMm1GdUvyJriwzu#`JDXsr8Sf5<9IvRZE5PVpzNd~jBt%5Qxp?EgPVaq_XoJbGT1n2ml2`qfG(R!#!7NvICjU5YVyUf>iQFgH'
    'cT3vOg!X0{(d{FWF=lW6N+5rBygF*!96psFOlIzI1ulQHigCx!R{sC(`NzNN5gCx<jvr_Dt1EW|L3{)cAad}wRD;HI+CHk}_?Q||x?{z-0NQ_*LGs7F@PUJ~'
    '^HD{`M`k02K<Cpb_&%ytnqh|>pL~E&<7|I?eDZLKBUHVfMt+qAIp4ufxh%>Tk@Fpx2r;*<*gnJtz~^;n<X6f3s%pGo(0m7ebUv_h@udF>sT^nyt^6t(KOS<E'
    '$Uk=!(U${K>9eAd!6NBs=scbK$d86U4~0anA>X<5jk}m>b5zSU1wKtH-#G`_??eX@&;K!!MDt$1A`V5$KIwor(aLvnBKSj!{hD~#8N|;H<U1dH<g(uRsR=$#'
    'E8j`3?{Ix*T<9*2^o<5HL~XP|u&FZ`R>fPN>4c;eTTQvp`_J|(`(J)cZYSXuY`+&*3V!T!T;#tpNd&|>3DVcRz6tM_IpsjgbjCTEoGnVn52Zjn)cMt`$&~z~'
    'riSlC&_W@AbJro+h;j|i$Qb-Qckic^K8cq<AQXNyU2WvEEUOPHpw7!mXHxX5mES6(2K8hBKF$LvdSO4?#ZiTH*A<4k<v*`6JO*FG8*!@dmCW;tJm@}LV}IBM'
    'T$7XG9=vgid^<mloc{=+&P7-Cor6kOM!`$|m4770$EvRNfpMn77@yxq%SGK;YoSUe8vUQh@ljPXw*5;cq6-CuH3PlJUlgo~g>Zf8;+f|?3)#Yko@bVumuMzM'
    '9)+~<^mv=3_MKLRF>zHSv{AL#*cNug%LitPnG&?eFuG8K_$6bqDwU8vUjL4VWi{)v*9)NWW9n3-dK$Mo#DT}J%C(HYIJy&qA0HLR>f|4Ym|g#l{}O&A>sN_u'
    'i*M5PXo{Brh>t+O%9;Uv`yL;TuV0nCc6X$FjUYPz_;uQEcXs0%eEsU_Gi3+PL}T<`l5z>5qF(;^`qkZ*gJ(FuOQQ9wx(Xuc!)a^r^{ce>UPs4o9*oSPTff?P'
    'Jx5r*?i#*+b?MeqRztfnJ_eb8uJNwD<4)4A{`1qnk4IhTgfJRcXBT@#x_cJU)^~Cs`+=nQo*r!%5p+I@UnBk33jU_Q^_@<BfL|r<C9h)iisNq}{g^=AWNuQr'
    'UM*J~BuE|SoZnFVbJFi9_G`{Z>SaI&pUpZooBGYO;T&3dp}EL@zH7tz<6m@T(RrZ>-O*ca7wVyVRB^6qydLnY#G8L1?6T)WB(mRGk-p!soV!CTCk^stpgP}h'
    '=1}sFE1r9K8cE25_*ez<t9^QPm#sQTzO<Ne*Ze8@$8%exNAwEhL7Y1VbYMcp(LAx+p&`7P(rJ6|*3N2vG*4R6>2)ORXzuLz4b-^OJgG{{kz^Q`FH0TN@;&V`'
    '4x#aCGvS_o&|PR&KDem;eCIPyWy^*k_#*+kPgLbs>oSn><8bjjeY-D6eJ;r7K3o-J?eZrX_Qku>AAj!N<rYln_`w&nvT1YTF?^o}iz8F=GKCo5M4#xl1xI@0'
    '@W<UzOXc4qf$=dO-{3iQp*RpW#*H5e;vB-^n7$uga}H%#yu6C9U*+_ZwVPzc=qcJ?P8m4x|E*t@KFCYyk1XZtw^QUN;v)jiZ#nDZBQvei@%w3BId#n-MvfW%'
    '`@EpszR%isU|e6pi3iEyAyRM9e0+K1m%RGj#`yZxL(T1+oCW_~{Md7PMPZ^JKN>%3^_Lf38GDA0AFsqI-_>>EN8?8i+q}mcLT%~BkIQ4_tmif+<KxHvrFZSh'
    'A|}xI@#?;VUZw{eNca)M3vGX*JA)tJWG0Oy4f28e1R2L}CdO1h(<>0~IvROdR%AbqKlJHg^$vctkNOfd2_==smtgfZG<^5S_%T(qZT=irJ~Vy|_y6g`V7wO|'
    'P{;XwvD#hMC+1+*wOhb$L5EM}kO@_NneL)45Fd%Lr%;(m{Ae)H%Mv_+zKZL4c#S~|^oGi3bTHsCMZYSybaT$7m57cA>~QOsu)Cghb%U<s;kgSs^^=yj=VJCd'
    'F2BO5fD{N<uUDXJ^UA~sb3WHvuQ%~|$sPD4BNdJqze9<S!B;r%o>oKr!T<sYh;*w{Ywv(<sPNq2%K?i2YQmz49b&hTKJuXAp@~^W_~7|ha41zAAV}Y)@T;#c'
    'rCXk#<OT5&_}~>T7Fo=D%Lj(&jC1D$Oa)$!ronsY95<|dfAtnGL---yh@G}6y&}$xX#az28`2kbUJHV_@p$gI1X*}*1E&A{>2gh}$cRd)k5-<I93PqWhc~2L'
    ';6v-ck6x&*$$5GL(!)FVCEqQT75Ow5J+Ie#E=)I%dQLmfM#hhoyf&FJA~NXyPm#e-%)ZWlPs+3X<!egDkJn#!JTFKDb*RYtGdsPd$yG8JT21G<Vbu@oAF*OV'
    'v*R03<J8+MN7Qv^nb$+%$1R6X!g8-V{yJw8Cfq3fk-w`)?sAMq>f@wHU!NOiSiSmQJTxnx4OE=l{{C3KtWFs0j<?<yFJj5={mP{KU&N2Z=VG|Qh*8pr4~-w+'
    '*A6UC?A=MikN^A_B>uQ8vYHRA``uIW%)`Q{kT%{S>vIfW`sTKM7ewQot1G%$Wt$sl*RzrJIb*`*^H&|0MC<2n!impb59H9UXJbM3yIT^v%!L~S(e-SS)|Fh$'
    '9<8+N*;o-$>ui0s$+d$r=z2D`FPgV~BP&StY=7zX$nmi^X}D!)jUc)|a-e8X^9_ag+38+T>r5)~-KWmq4<Q3u^>Z9Ze$IM(Q!~p02@nT>BJg{C#R^rHA?Oi-'
    'M*SQ)K4#BZy42z`(ia2db0eBd`ro*^K(2Uq*y8n0BikFJC&~BJx|3h|5!y?j!ezv`L*`eT-ya?>^;-t`K)~d+O^eEE*Hp|d--6=xti!xs$h$hj^zw*R`MK=#'
    'q5|M^cq635PJXX#U`5BBK{*|-eZ%?m*XwD!yY&eL;n(Xa{YsGt-HNFPT3s|*SR)jGU$2+hzTP&I55t#r)~=ef+})dmcji&8_nG{ofOd?x1_R<lbr0a&5x07o'
    'lPypsb^gCAk>W3>e)&Ah+X6(_n*ff>D#mZ3ws(_!+c5o@pTW8_{&H*EVuJ79MDmD$iJb`Z*jKjmAcWh0rNeLa=&LA99g%F8x;QgK-t7G4P~)^1T+6kDx*+H;'
    'zrGlXepNowD&l39G<v<K=-fV*A$NCnx+T>0ls0U9vG;s4^ohEE=kXLu{;{kqY2fK+X|PWulfEwguH2j-mJaFDxlec=K6S8g!V1d9I~P5el<LlQ<iPO91{S`*'
    '&z^ww@a7pGw}p#t8{hx-_=sLF?Aw3q&eT7a%P5|eF+k$uFVcTMJa@R0V`dL?Y@pGfmF!P`^TFiE?ITE>1Iqq|KlVlbJmm>Ny4ZbrTALg6?;y^N&w+XheZ%7D'
    '{vBntHMxxok3$_0)w(t*r|4IOiqq1z?M3t?pniUO#Z?0b-Vmry6T3ch<OV@}B*u@`V>=@wb|Lz?X%LUcTHoxqjD%z~Y4FEne>scq*Zfvo5<s6rRW3IvD8))c'
    '5qReY6TJ$TK0g6W|G<mCl?2vSL)Tep^eZ6ibB@n@k?(&J$(MnC1&Uca``5<bfJ5+B$|8eqLb?{FZ|yo6*ian4|Cg@`K{%13Uwt+t=H&AmiI2c(=UA%gX32Hw'
    'v-8D9#YcJjW2=7h)j}oI^$$e7EIB@Av!7HxkS?K%A|&>o^r*0EO@I~YjB|`mclErxJfQ#<`o^`&t6aKe`T{Pa&g+e2Qux(B%ALMZ?*-BR){<kLyV6BcVch(&'
    '$GG!Zk@FZH@ru>vi3yIt**#CxxLxDwLW9-!;jgSz=US6MkbflZ=QXEBcc_!=j{t+4cVNBHfLsH#lA1qGy#V4PF<zOT>h^iH6{%w|0C7N1blO?&7356#+rN7T'
    'KNcxet=O3*fcEd6V;*w-W|9}(euP{bijSYQ$)Ve4^vpERBd@v|7^XV+>7MJL<R9yn3z{E&DT?mD=KQq%>PUP7ych3E0TsFzZZ7?W=Bqcg?GTCSxenRmjWgf('
    'M)#lwH+ucOthjShe}f-fgEv1FR<?()vc&i;8S*S-;R%j~j^SO8RpzUt;bY9XsQzcl(R)>w;U>K6^(^$xus+6$&fk0<EZFI7oCMA4-2T73{_&su>Sv2u$NXYm'
    'G`||U&Gm^zK|emf`dnhU9+L<knqOTz`88|om<`?h>Z9T4oFnr8AAVI6+Tdr!Jc;I4H@$P{3>G?y&#y{omPW9)^Pu@v>B%adSoO>J{HiQxMTmzX51L>7-YD5R'
    '-sFMLuZHZa*p^J-LG!EnlHS(Kop$5%t8r^xM`~EP&^SV0;i!P7co{yws=VryTO}6{nqU28xAYcEQ4>DD%75q`H@nyZG{34T0y`ZWb;0LXcU0Srqz-VR<3LG6'
    'l|uB!WPE;AO*gK7)dVYg+->djzC+$I`26bJgacuooeF4vRj<ByzNjMLS24b34bLe0oHHpa&gV%WbuR$`mt`OKd#aKI@59aOE!gljTMTpF@DyPv$(AmM=TPGT'
    '=9?6KPU405Pjm0{fb#}W<jC)ljF;4io1LE~>b$W%anXEjQDdl^&T+T07oVJSEfAkyWp~Y9q&$M*S2-hW)t+uH$LCk8x4?p_2|Q?i)#~2aAn^x|`24D=VUdaB'
    '1I)bMlcqVX$T)z{uR@XAUmxa{M)RxJM)NGLuNuSWS4(BNp0+H*@T+RB(qTWJ<j~EpDzxlfs{A>CZhrN_*|o;un)LIlyDs#kS^j9p=T~n(leM`PEr#Y-KlFdt'
    '_|7VhZhm#szTzu)OhfVc)g?;Jz80y>|5^O_K-r7Hznce*A61^(<YkE;`Wrv~4;~=lN8<e5<39MX;v$lN1br1CE<)ne&9hLk9!~#`EZslqDaE|#{vBl>-6wrZ'
    '!twi_CLVhk+-%N<eoqJXYb)|!@`6_CQLnQ>pkF1fwSlWn%<v!=x}S#OZq92Czk=bXf9a0~VUmW3_@g8ARw?b;hxk8&|4aLpAFEeXCgjYEJvR*U{?H}GN8(<2'
    'elgRx;};R$1UQQXvkx4T*Gz`gcxO7OQg~T1e)M!?{J!!9!jmF!p_Mgy>joDXw;!B|b-rH5ePQ%|etn4ZYmiD6WX?&2Blc(O&h+o7Nbb59q`Uy^=P>E(^ph9Z'
    'JWLy*6n3h3b-N$vbBM7gmc93Ym^;#s3HZbtuiI6j)_ET~#ZGm-hba{Os(|@hqxj2Gpsx~e&}OVHd@<MJJoJTa7V8?-A1U@zFGI<Tl-cQlQR9|;>Wh8Zod?aT'
    '4=}p^@$T(B<LPT~5#B!HeC^;p!$=V{@7%)DoV~Ur8(K@}{;H=f`N%FJ2ey)?I-g$gq3Bm*maw}gIm)8z*{`iDIxZIw3%B51Df7qqr%wejJlm&T@3^ARrotgo'
    'RO_}ThEm^gTI=R<Aw^j*KfzACNr&y1N^KV1{y?k6^6WSNu!9dv;fy$AFt}1-FJ@lnA89UbIFJKb&ZD6pC&x!_Mim0Lw>Y}5Vpd}3Sy!7>h+bZ{YsYink4NmF'
    'mGf{$EZ_T-iSr>h8V6Q6mdov{Yle-eeAKO?9#i~Ruly7khtBYUeGr`6O-^?c&M$C>eem>ua{4bf&GF|z?}IHrA9=P)C&CSQzpTn9sc#}_0_`UevY+$j@=po>'
    'hadk{|459F>PHFMZ9`ljK7u-&(N*pzIaCAxUH?e@$2NR^XRfzZ7+wE(nA4jh=avhp{t>;d?K`GG{*m~{{FF;)xHOUc2XL79`taU7;njZ7BNZ$T^w28Mo$3FW'
    'G!<Pek%9Pd)RDet(th1p_BI+?t&GhhN^4C~`afzbc|P#SLe>j#2sc>OQn@P77@Ah1Z@s*dGAR)MojSYy4mD0XZCHgZ=rSCNSI>5>p2e)`BF2AZi;>)7@rRG$'
    'CI;;DjP>|N!H-#2lHKn6%7Nzs@;lRuYCa0Mzl3F}#<hK&rr<}mvgcN+tjKc#NX7SuhS|H&aQF+}FLS=Uu6xZVOkOCq!J^S*S<&qJBcjH=4`7tv-R}dX;*D?;'
    'wjJa9i1GQW&o6G5k8P)2FH6RcCL!<L)b<OZ>t(~coLlXkQt7Uj9dY%%u|DNKsa_VFAHNImBZ=>PHQD{;1ET9KCw+ZCqb*y={W&B|h380p#(0ta{AtUcD`#5~'
    'oCJJ0k4nEz|1dBD!PGh;;auRqN{m+yW}W7cJ?IyP)OX~aS=teD=qY4Fz^+>f9fIDBKftZKo~`|PNPXupcrGsrme?-JqMPr`NfGlo=NULVoP-+Z_Uh0ECDkv`'
    'e(HR6ZW_g(+#$%`(b5>nHvpcyv*Y`@Ex+zSs&rm2YIIsv`*GRq;v;I@{L3enWxIyp>q6f=di{J}<!5yL%4U{@p>cZ)VI3--TbH4h5+5&lT-#syY9V;<uK;>>'
    '80@pN3xqlF@IPbh^EpRc=AhS~+>3MG(;8{BJNKb|cht7MRs4C9RL}j-Pl38;%hCM*_5O1I<X2yN)+(=e<3jVR&8-z?5njFc{HpK4x7Yd?@uK-vZ9kTzsm7yp'
    '^Q%sduz2C-Bz%5V?v3Csr}-bz{OYZ&EtS{w>`DCUU-*~>iT@Qb+B(8o+~7U|KK4&t5`4I83v_Nh4SY<-kA0d?MBQzW^Bus)f_L^?s_R{$jaS(ad^JaXFrXq}'
    '4jQkr8H_9$8xA7jRm^*^x<<)Ao;vzQIM|C1?eoJO&Ybq>yEBx9iyxozGA%lf@e2_e&gQjENP-tp#s9$u9=)0PcuwPEv)*1Lzba1po+V4|oMpBlT(AJ=dKOj-'
    'CK~E7{g^5bLmx|88sRfkeX2?MV={jH9?-W$_&(yN4bD?e+qVU;dlCR=veLJ2{TaQ{`tjg(sF8ubadU48eH0tMOB;WX^N&Y%TsP!BA%eyqp^ES7t9p~^#~=KE'
    ';;bbO;^U8eB}K=Z`WT;>OM=S^d7d`G8))GVa{os`lO}ms9OR)8{IT9>;bh&<D)=H3cAN*t1K=-5j6H9MF1oYGAbk87;5%2mDOYT4f!i5r?B`_s$Q}K*$Mygx'
    'm@i;I*Kp?zFkJ5ed*k8iQF9Z$kMA(L<-H|K+2G0)*pN<K9Tc40vhrFNbeBL?f7Cfb@t5lk^^Usnlnd?esHPvvG`H;mY)hcZ!?2W6^f`+8#hl6qfUhax5$Ws2'
    'kH39dH5yH~&+iv^nJ1RI8Sol9$MIZIOZ_OE3>nb*Ty*UXB1SV4;0L(8!Bw{!F2zZ7eMg0EL+WF-gxTef|CjsAF;Me*49H~pRq7I#yp&TH_3hG!jotw{Rj@I&'
    'K7g=~(jQqbZnNu`x6&Z)00)i5()q0$s!E~xRO@9hT@dJVNLm{K=1_52q(3s)i1plU8$y4Xz*Ttro!XCgu8~EYXkH=Q$kXL(R6K<1t9;aYYP;bk2K2ZD7k)#h'
    'x1F#CGgZBJU5f6EK4&1gc+#^TsqX-avY}%~9_-!|39rQ451cIas$43ai@x5C<OK!J<8d$}Bh~ekvMKSgI3#DR_zL2m2b@!v3=}B{D5lLWZ-Scl6GcDyt-dzH'
    '4OIGE50!k1K4;06N7tJ(kvIT$swYSH4}TO&goE&|<XES@@CW-O`n+uQ^pEv|{3=MD+V8Emff64(ZkQ}_+J@kr_ki-LD^z}WBn*0rcfF>u*X~dR#vi~!Bk)ad'
    'LlSI-H;!kw{|8~0ezgB{30yyT<i-`~Cqz|m>F@~DcMxxW;JV<esqaX>8Ym@l8te&c((gjWB2?E99-`zQ4@TT@e!?z{_TASM`}sB5_cAmG?|h>ycmGLRHpX|K'
    '#nO69o5Ch2iyvp)5of6Gg(jvh&BKBv^0e-4c)bD*KW(yp_5RCw%5^p};68z$_Qf`*tfwcM;6OGiUVT#sh>yhh5s~4uvV{q$KL&m_ON*4B=w&^G)!C^1IR@bw'
    '{i^lvT$z#iLg+k&b?I6=XvuN75RXSJyeRSAs|&+Z_`Gwf+qJa^_N4L$jt%HX)py*At50D?eDH69=TcnCo-20?4q>D(?!0ewlwZ%J0==g*UPTdd>jdDpkQsG6'
    'ZqKLaS6wW)PG4X{^i_bj^XggqX@6NE-8$*B8y)%A9}J~C-x+iD#OHIKx=;Zg&z+XjH?H<b;6Gdcn5B{V;g<*}8b6lxcnm%d?ZL;7_Z&YT*Z(mOjUQiM5Ix7|'
    'e3)+hIM!$QRUb~o$B+BtF2!)xVDKYX(Br%%T!;SB=UBD(0{lo^FLP3Q>*E9Fg8Q5T>*=LNxQXjUD0&?Y+{udYJELkoA0rQQfqewv&SJ3wH)mTH=-@gUx?VP9'
    'KX@-}7F{074fYYB>-~9M-7=lmAF|M)q3b30??{#CQCZ@R>?1(e`yh9CcdeB-^i7+Fu9w{3s{E^0SZ^x8kBF|9&2d<=l+hTviRU-@`t^j@xH-nx+<T>o;)jE^'
    '@G)AtNpk+tar>?v{M&&~ETWrqoSMrmGf@xmQTJoAa;4~3$4=zx6sU@Vd?#2a6{f8Lq3bedw;!PLddfdOXN(h8K%;mg>PLG%4Ucl6^MMIZKaMfVw83UneR(}h'
    'V2Xa#-0?84(lX>eK_7WJ_-1*&U?>dX?dQ=Oe+7SG;6Y!{$B}FE#m-bxzfcUnn$Ms|nn1*O+`6ZriX~kZ<a5DYO@}%EJY=3vciqe7Cnd+O9V><>8L+>v>P0oh'
    'e>ML@^1Svb36KXK0PstEDCgUo*xC8gqVl;n3ClhW_y)j14D^jFsQj>e`OqEwI)9<Di!N4K7=7nfp5EZ51`n7G@4V5U<YL(3fvK1AmonxMQBH(3>EzYt`s8Nj'
    'Dm;ee7_sxZ%A|?n58z^-r<pfB59|}5U&xNtw_5_snxGp5?0L~cuaLs8+Sscddzy~)>+1mfkA%=KLTbqWM`wG>L+|7l7WD7C1F1#n8y3z^pMx3~&9Kk<RzoBl'
    'iTD28__Pn5s9^k0_O~MLUAtdzy>jj|2Z)bgA!`+i>RxdUAoaKUiyC4?@JG+hb-EQw-01#RQ6HkVRa$343e^3kcDPdVk6Go+{yMFQKaxD@>+PJqd;8Qb!^Axh'
    '6UPTFDgLVhL5w_o%aQz!5b0}6MSsV@F=wb0Z{M1yJYgoICW@X<Y>!i`rmd@CRu-!Jtg@~mkn!V*@-sPV<w$-GJeS0Z_%@?&w;@wH&uLD{R+Sc3m*Gr0`-wk#'
    'X>T_$-3ZxJotFs#mnr>`HSZnPzqkiE*8@UzZ}o0J=^q(%`(oy;h^&~Hdl6cV_dJjLGcQnHgz2Xd!0xbP#~&-W4=*uv&3Q8BB_GEBIq`#WLsVNKslN_}?^N`p'
    '_^;l4-s8ZQjOf6?4zN7at>l$`7>rv#=TK(VzG{jEeLW4y`aFwu6(qlDOdNkzLh%QPR^6!|9gFx1ft`wLwQZVfaWqtncRgRJ*2@td1We!0%!tveeA&=CR%%@q'
    'GwA<F+$+6(+IrLu3ZZqOhgEY;Uk;W)y)0DsX;Mwm=k!D!80>zC_^$zH*wrSTD>g2VnO$FO)I5WG9^TWRTLs0j{Ow~jQ~x-(KkU>FLvFN>QAs>cnK{QZQeQf5'
    '>hlyCKfco6x{PyF7@Vg7b2wju!RB$P<LvGe#Xo9DnL3p9CqlS>eJ9q6X1Xt9N9%gL2_-MHItrn=)P0T*Kceufb*Zx7f4Kv{9)dgIDK<~-*4`G3KhMHLvUm<2'
    'U_oE+G>>A))48rt3Eq02aP4&L*~^6<mz!ZBks8+vAEwSbR|Qe@t2}G_blfL_?=E2z>Fe)z#eSriU!2`}3iaMo6O0P)9Z!OF=p6T-!jG+;(U${-InemA^^KC<'
    '<C<=K{J4dOkYC%)g~pFVi>+@xnY5!DKi+26onp}Y|KP_fDo-z+yZ8x>AG=PpT#A;m!^e-cl7#I~K5?M&WBq<ZQ`H~m@$n;Hp-yUv$Q(3&?0;s{?P}osZ~REy'
    'OA7mU@V8yyMB~S@U;4|px%vJZKN9yl!4*-l7lZ`R_|eN|&9^R1FMRx{=pf0xJ&YHPAMc7V9eOrqgpVI9wzl%e9>w6t>|?Ii_3di@jUS1dOYwt^MI~DV(D+eo'
    'vo@1snFk3!{__*cSSIJhB81)#95ocD-_<>&jpxYmvHz3cjurC4Xgs&_edlr&tq^=Xclf#|`;uK4Joo&imaDqVKkytmK1%PBik6*_LgTq}d8!NMozA5j&&gQc'
    'ys9y;h=k`bJ`~)Ix-<Ck`?vMcQ}yBi&jCldhTZSZZCsfK8PSR79KFqV6}`9y$IyxA^gsKJ=CVfP<GH->jOC}jF?jBF{$i#dA8Sc?4zo|>9HYd?<FOacE9D{n'
    'PV-4$e;U%NU6q-DA9qqtH*>F)!T8zR3oq1PA?5_<<Hen=zWw=^mScF?6OTmJkGIFujUQuloy<J<ci`j4bDMtaNLX{B@nfD9ONk{*BR+l%`s}@qF;yCkA039v'
    '!;ilU#>bDtr=AVp+scYQpJuFG-D%OCM>l@lCf;*A`F#XFew171RwAkQzurHGX8w`*ej}~VJi2$D9i4wPm)EYHZh-%te<ZHusle=0hgV9X^N*|DJiOFui~q_$'
    'l6(ir{;PX0{waE3j`ZOGzJnXp17lVW6~S&)`v_ra2*rQ3ecR91XQM#A1;G(-Ru9Jxs$7QG<Go+uCf{~WB@7>NVw5@M*yaSf=hM3M%UVu7&!I%Bc~3eakn10p'
    'ye#S6kb&vzYNmhh9$RoCe%-If^e5+VwR!0Kyz<8LjF_h{tU>2}Hs8EEnf~?h-}zO&1dx9u?#)Zzom{r%E22LFZ@*(1<CX)3&tWF&JgI#EsP7=g5041`Mki+k'
    '?|_U)$8gTQp0}N_6BVyVxP6JjueM&{jkDf@#D6f+54^l=l(!=mF2eg|#>3Yu!|gG8y*Ke(OLiB;llpM}rFSChSC{A5{fJCJ@|{5MbmL-*1s{Jbw297q(qf^7'
    'yQJH~@cXmV80VG!J^S2_4A>uUM65`~(11lZS~n3sl==LLXF9CLNF$$1*5_;-PSg+Gg!n>%3|a3$t$3qHDO5zwbI*&X@T<;3mZ7C)2<`-Vb=A2wiqrk!un#V;'
    '_v*KN-3m;6WN8dN{9;=Vq(t3M<4iU1&mqQwQ$p@m5A3DDdl&@wS$Fu8;?uWKA%RN2x;s^uK-RA&=PZ7ln~dmJK?jZHHWAF(mHCh^ow)Ozs%4pJavQ0SDn_?t'
    'UqHc+B{xo39AH5F<w4(%Al4sq#S{|ZS)Gee=dgy>8|&YHO`4s~1~raBID62=Bn)=J8__)_n#9eA@k8IA$7ffsl|u3{`se5D;3u6$9s}rkRxg*4IH3ZO^6~%t'
    '@M*05F=Ea0pVhD4_qthOyMqmlA2U@AAKv`kg^wTSUC>li?&3t_$09`&?=*;h|J6m+3v*K*B;e!6iOAmZgw3DP_|e~G@y6$(2l4#@1mXy*j(lW8`vbUfwY#0r'
    'xJdE`KtGqyFXrjZ_ygo_c>kw$9)cf%@6Mse4<m9eSVJLt*mb5k_+59Vzg0=^ImrPfL{|uKr@_k;CrbEFLY7qH1{b{7CE-Wn=E5lA%`z2(=vRM$bA#??(`b|3'
    '&=*}A^*LnxSi$>BPSI~Jx=)Vpbrr7)n=Q~vyaT|X!8Jqae#|;^aL_t+cvlHrH5a?zc|0$t)ITnROS65H5&Q`1N!AbDd$?(T5FEiy^&YmgQuH~p|F69-52UK='
    '{=eoqW2O)ql$2;jolBA?8dOSTN+^<$h@?^(GA2_ph0H@FsbkDgNrp62qR^m}l<<4+-G{sHN#}9@{5{Y2{`bE3+1=xueb(A*t<T_efAQ?yQXu|-c;47G(0g`?'
    '1**fzSR4t+YVdv6J<0kCTKDc&vtuUD1#}Kk{jqxJ^0R}MB5<AS`RR7G_n9uLZZzCC@CwN%GmYVM^vc|7Brjn8B)~U1M?7{=`(6&B$zWa$;ulc9@GA+O%iy{&'
    'R?VdD)Kqj*IDaTkJ`j*SeBfCF;(<G#e)DG69Opj{`MZ7NF2&BLz*#iI=&62gPX@#1%%YQDhRJ2+$U6L-bW<HoA7B@`DuX|43V0uYPn?<6(ap1Ka?l6(BgP3a'
    '&2fB=Z6x2~qf4fN=K$t6y0g-<)z+Lrf{CthOV+DHBWpP6bcMwV@;XLRCWk|X)*+`9PwpIuL5>stp0sROa6LPG-dJKfzqNa$o>(7X^DO@I6lF13AD}!gK>EVr'
    'IEM8BW*qF%lp%)_>jUh|a7{27X8X6-j}!VMQ|`;#*rnS$R&c=cM<J6v#jbU_fBGZCA0D^9V70rv@VUY@P;IJj#dYd@hu%jnji$eP<?;Dt-ZI!cDT(^+O5cbr'
    'L!ZhKF<hUG;_FJSA1T4QS@~(aZwQ9F3kCfv$0c_;_Z1*sI2=5QoK4f0>wNzsKgS2lS0!Nj^ZmY=_k$8dkClENAwIJo^j9hI+$^%*z_JO;S3p6tYC)Nu*o*>1'
    '9=DDZ_a5X&O1^q_Ae(JC4&e1kWnlk$Rx4hqLXk4u9C0(!4)P-<DAlW{zw<W3^t6G#<?bDY8D<ZR5m^?-&dV)jO23!(#USwvp0ks0=C3P}c#PtG4!u-<(E6*h'
    'xfk8t{S)JfgJ|ks<MfV4FJp3gsL=0y3_(7w4G5SVZwp%Y_}f`7?Zrux^KpXWPchf(_)nLkC#Pov#q(F;;i1FHZ;?x+KY2al_|=FvnodC({4lTQa?{7CH))3_'
    'r@sRIp4t5w!L{p>iG3UlFPN~i@-f5xhS6082AR)G&<-ZNzEe9w<#$l>$M98mhncIV!n#mLR}_jQ9WF=uarvLLIS=4hDY>Vw@tSynxB$#QR+w$Zey2GB%_dEN'
    'QyA;78d_8q`Xvd%K2??))t^l|BPSQnq33tBK<~EGwE$xM+!HRkTMvkg!uV?Q;!igV@5W6|*A&j<vr1F-mbIXLB>MN(Vb3t|N2cbh#j&0uVVLgaTk7|>=Uu#2'
    '()|XF#rK)2l4<?b7?aN@2d-myJfOdtWvL;^we=LSFZL=;?s?IcBk<pM`Pqd<K`*YMdJNvroi8ppa_v1ye9qL~a@Z*S1fBlTf2O}md3Xz%P=|+~+2Edqdrq{~'
    'r}=k?`5mqgQJvX#)8PK<2I1aUOFRpS`5kvETYK_aMPYtN$+}+7TO7NI`5g^Y4>`Sg#|fWf_@`;MGTFxw^E<wNQ(BlC_Z!xYFzJdJ@RkMr$3LW7#{3^+W81#A'
    'tP+NO)=F9>%=P`>|KaCj$|R&Y-e2u65knW;#Ln4Z-om>!p0ublgP4y{8oxoXXzC-l|M6l~xQhGYB`Blum06<?Ioe!<rm^7g2*=83`T$vX&tBu625{RXD^L%#'
    '@_A>N1fiKk=gHO`^UHmIOrh&nFTWuq>feM6<NIagKw5uQ<O++z)vGgLJc+hhg!&KVLWc3|ne)%>nzaa}>VDImQ+`y_`m3>hUDp>%U^suE7b>>N=*hdBY=-@3'
    'b+#7m%F23$rs3lx(|cO|*yE?JceG?0I5!l5In;c9#b~EgVqK_a4yUEUui9bV&um+b_g;6dG0eLwn0Y?9*3O&SFZ)9mYK-6EHamQcYmETh_t8C&6gP72CSt^g'
    'tIMYR@4Xel;<<N-^z)oj%pX9W`n}B3#HrWiDi98y33#t#{ngB(`uVlHrD0#Xof)t5lhtdHeO$Qy6iJjk?*I5fuBf2i7Q@F$Qor?h%FWqkQ;zPy`L=kameTkg'
    '3um7RSAT-(R{_2Ls;adtw!VJo0w%_ud)y=O^3SzdlY0)(I$aZ^L-!8(BK$;sk+3ec1`kPbxPEM0l%r~v?T021U8l0pv*|y3>F|a#hX=bawE3Y1MDMMT+ocxE'
    'PCDJ4hn5Tb{c5$)c%lQN;mTd3`d<9x{}%s|RzD8ukH|c%#quLKb;czmOUHlKMf-_>P4UXde?NAo>r30qzmJjGmX8>7OyFNSmLDBLPVGPGi`9?7zx1(#E>BQa'
    '1Ujic6!e_-YB}6zX1<9?ap3CCuriuHfFj$_YjYn=M-}+iDW*4`eN&f*l(GM*hcb@OdFECAHPl5I)<X#)GuuU8N0A$B6Zn^o@jIFYwhkPz6@_&pU(9qtgUu@t'
    'Z?*~Qcup+7W+i{A|CWL2VuOEO!T9=a?a^dp5ex1*H>ZOBDkV0v_*hqwbtK_A&Bzw3vz7a`5H9>WhqGY*NQr}kRlKTYg`kf}0*Ks8#&(MjYt>QvUUc<<V<_ma'
    'Qj$^1#r1Z)JAuz3h9`NVu_KT}!jRY>QGRrHu;2VCu)pqmcl)TBlCi}4&mS5C4;uOYK%jLIU(BC3Hk4D(Q*?1TFBSMdQZ9bRt(DEYo(X~L1o!2w;ytE4JM7W0'
    '$;3(M_bj*0U|V6F#IRq;#-8mvilvho)@|8+e%-*w_GUyMuS=U2M#Hbp-MNv8Wv?uF?gv2M`Hg>-j=~j0f#~l$ew~BeYY`pZ^3%pgY@xGqkU2QLB~tVwn!ent'
    'c4i~7gIIsu864?O$nKc4@+!(GKWAX{rR^*9G(sY}&h~->=fYYAVSJSm_p`yM!Dy6Gofy|48;|v!HB|jvy1Bvq5;BuGmjCbWIQA(1F~~a@4$MGG=g_N-(P%Ti'
    '4!oZO@Hv#Ut#~F}V3V;ZcyEE@Bge4^k9)fk819c3e9*gpyUYVgCt7i9dvaB`CmkP{b=i)Z8%Q#`l)?LwWJu!G{T7WF5v{P=^g*zjdl>nz>aU(Q5t&uE49kyT'
    'UZz&L*1x&OoT|V2$6SvYi$9)@{vyB5a=~-G%gPDmqJjI7U90KOcd3mSK8F%tx0-0=?pVbE@)ek~4T~-I*?g0X7HHw(!Tx+2K4)P4X{SxcFdbFUw<MEq_B}*4'
    'AX|wVHIE%Tn(Mjg_)u4!T<7ZA-asv;;P!o8708d2T)-{ZlA-xc0-k?#tMn#5dz^yq;KkuB-ENb|>&G*s>-t$wL|{DW;Nrty{P>Qc2}I|~X~}$Nm%f+^?|c2k'
    'y?GgHDv|3fIR7Baa@zb+ZOiSd6A>7W0r1(b-`JgQFgpx+$DqE9KTpbUQidD)fWdX)+O|!Kb`dBa(fL(`d)3aF0YdP3d6|2R)8oahRJ;?M=Sm~XX#5VowkW2%'
    'L4t7q>bpiwp154v<nj*m-0#n<bmK^}LEMN|v<pn1J<sbmT*rGqeSEFJpkd;<&{+MLaQ9^KmzPuFxsddnh`#sz5tGv=f#Oehqh49xz*E!~htKh1eni9PXt0#G'
    'zUbnH=Ygx7uX-NQeS|*4=`@s9R|5SUN<2SR{<b*rDdyV-7^DR{EH^tWU!XDg{J855EkAa#&lr3di|OZV2Yn`gCZiUmWMY4r($WhXdfxWKIt}IhK}$Y0RH2-B'
    'TrW!-4WILH%POAl%G_XIM}p@U;dgFBX9~l0wZraio6ANmXgi7iy(MS92l<hb>&yolN=*B?LH-B$@tj%HZZi!f663NyiRnja34TMMJigVj!@XL|gSbE5eX9HW'
    'g_d9Ny5OVcHagpK5JveHZM$TyR19B3gNRlbKeD)<Ui%i_*Ix4O?{~>v`9J+tB79Eb&gd-$L#!}9r(wAERY~t-Dn5q}2UluN9>?b#Ts_Jf(1_*7nc$pe#dAM$'
    '!4YIqIu1}BsGxCb&CuKgl#tg%Qh6}gh0^(VjL0U;u`#lw;&cAscaG_=wzf{+8Slgjt`h`s(W_^cviWJD9z+{)P~2*!auhe*SIF+#F50Db9nt3aOZ`YWZ(K+b'
    '&(KO0g!{5_kAqp%s&67(9Q6IixZ(ydf272_u;xRJm(~lw`XtEHw&k+VZzIp}eQxI(kRK`WZjVM!_H9`~INq6tHn~L$-b7OH>&TcsIVGNN=IvJGP{4Erz=_pM'
    '`R+WK%nHN`#}lLCLdWXIjoj65hu&g3TmUhz=~m8n<5(~;zRI==&EkGa=L2-mRVp#`Ni`zOhQlo%&w%)ku@!7e9GDLm;FH{TtnGbmoq!PbEgvBdcpNN9WIEqr'
    '4YB7vrARO<em;(!`6v%3?MeHSW9Iy@50_a-L3qTHFr<aS`mtC%+uvUOAX37@*t!ClpYP80R-l^re6@#JlWEM~YR3#I&P;Eh^GE6i?@?n#P>FaWRlk$&{C1;|'
    'JU)NC=cjv>#e6EbpS9HQ>rZlB40&(|eT3^D_c`{`@}pAMoY_9hK|Mz*rGBrfsDER{(P(rF(O;%9t9y~7Hwo?=h_1Vws_j&Uc;Wl;8#8J7(U~Li(#R}K$KWPt'
    '3$_02`1Ctza=c!MzoJ%lXP=^hKT0@fJMB?%aKZCI1X}mvT)x<mp%ipdybx%gxmxjg`UHkgj!y!;4<5P~U9L7IB8=+bET54T-ja9)Rc7$K`0lu%eVsxCs?Ol~'
    '?cZKMj;BJ(Go5YzQSv#;3eO+YT87*We9{rQDHHh5P~tx2ncn6WyAs_h4)PAj1KcO7RipZHkbb@id~(M49Rr`NhB>BT{*Pd2@515rST*Vza+#05j%YvCNUI;k'
    '+r`xFH)8$He$a1m>~B%`utBu=2%hXA+3n9tdIaG5(YweaU-ipnl(4U0>s&6ko|6Qx(>KeWw!t?6A@qZ<+}YB3Mw%V&^Gh*bUFKz*fG%M0+)&xewbwQz3{_r^'
    'i<74NY5D+Kh=;;bMXX;9_%dhvCAK3A&Y`!7>bZJ{*SS4b0&xCcwZ!skNaP9B5!b*Xzvt7W_E#yfIrU-1uX&N!{1^zXRdq=R*f$3w9dq#dIZm|s<BcbN4<1Bd'
    'ybw?Uj6RW1^I-KuHWFPYll$Vx@DMThIrMJjQgSZbj$E3J%j4goY4b<>z3h^lMl!HJ;uE2_%_q4}qjg01)skz8p>O8W)w!9{7Wp3@$085paKHB@omM}t^--|('
    'xryN>LB>d*U3y4<&qXROl#xE$Kxg-gp2fPzW?9D8y_vReThxy-G?16RehlLIN~<5MMe_&M2jswW0R0>B=m5WfUsb3CFa0_C{uDmVaeX=Y1Z%`N5A!(&JmR|S'
    'dlG}!8c;26-1$4Y81Nq{+oNlq6OZY3jAx?{_UDzKN;1oH&@DKfSkk>GG=1{ur(IoFxv{wc_`OV)WgDllxuH{u&Kusg0`m$h>F~#wb$W~B6EhG-^-xD<zFezz'
    'Hw^tkqz|n0?vcK#7+t?}xa$TvhU+-eO7z?ne_BMm9-`At{%&WtTHO63)rS+lpQ&&A$xLJT)kWtV9yK>&eXTm`_tsP6uIv-si8AV&5P-CHUFEvLFiz?}gU_Mt'
    'MV?O!r3II>!1-~n%+S-jJ4%T8k1K5Yz6r8Q!1x@t;pWjB)dj@-$8BwK4`0-X!2HLyTdluyJFgS-A7d-_mAI4&!u-c{sXCEs0^Y>@$0DaytCw%3<3Fmee<8ej'
    'c>=?@^hk@tTdst@Am%@wU~zDE))IyJkD@M_;pzbi#QaBpzRX(&TY2E?92DI?rSTl&{73OOs<ND`8z;sejm;moR?XD-G+zj=bJgdveE!}2Zesk=Sp8VkxU$fl'
    'hYo-A#+JV;<;*|ek6fo4eo7Rf!ylD~%Xz3;D=>^dIwbx|#_dG)#Q3AJ{HV5D`{RJG5R5-^+N<lPyEBj&fAn1NmS0*a9S`+$$)meEO^p-dkH+%j^7kQkw9d=I'
    '_#<DVuhI%%^BKk;y{O2aQo6ciV*JtA{L%Bdo#-sDsWAR%ak5yg&+JTw@kh0@YS!Bayqp++G*&-GsN5W?yCMPOkAkYwZyZ+n2mFz<V0^i9-D?<sbS-5cpKm7P'
    'ebR60cLHiQo}3teB!-2PsK?GvpPJL*j|ATLx){g+{1F}h@pKWbzZ$%zqWGN*rV9=HY@D@6H3I8BkS1<|D^%Zb&AI8A1biOu5uY<C6wrX4!|{kyHxAM8t8r-_'
    'CJ*0Bf&B?4_cyM$EVWt`f|@cj_C6dA9bP7;lQX&d39pM7sC>F>H8DPiGta_D_bcDO9iKz@`PoSx*9RzXO|~(giRDLduwJ9GrF4cw4kF5ps|URH)AD2Sl}*l9'
    ')iC{4z`tI0raWK@(u6F-^TahS(e&kZo-;7w7{KZefIR);=d-+ENd+?SM-BIc=5J+s>+s@}CR1VjyyT&Cbst|mrRrP#Q9_ROSARG4|JrOL4dcs~AD+(q;bkhK'
    '&S2aMJi#3vcUBK2)R%kG6r&=wj1GVOL13xr$EW^(@FwU!UyBn@Bh%l&`NOpQ#!PZs%H;TS(EDkyHq>n0+RI2A=U?(8<)J!S*~Up8pAP3o+oorqt?s8zE?+^v'
    'mv14Rp1Ls=$>YSua}pNdDG%pa_q<V?S}b3I-wT{4sch#Igc9nu`Oki{Gq;xs&R1)>UL6@0y^aiV;Bbs{8$o`g#JklkmJOcz7(WL%MLO%CBg1lck@Kv${URR+'
    '^j9glfPYItq170F4y-FP4Kg&}Qixn*nP8s@VEHPyWu8R)PjT44b8p|%+FN{2kRKD|Ri-ihob_eoQ@(u|P73Uh=JWx_qE9ghp>9I&k&hjE!gTl*`PY{YY`o(+'
    'Ih-4`uC<=+bRds0nnx6uTgYp2Rz4Jh>$2suS2yt}zed{ef1h<6z^_tr&l?dRIbJ@D4-^6V4UX-mKCF#Y{t?|dGsuh9U+oqwx{~B94DJi4$DNtBWlED>(TxoL'
    'ePS_2m6hM(knIev8<G!kYc9`45C-%3wtc3{TX$bW9a-q(C)vG`rcZ9T;KU&(Z%MGvPJy|B%c?1JR$V~}=awhGHU&g4WQOb2C+$%YBPrVuPNKS+yyxrwsBd(>'
    '=JJmp4e`!jkE#(B)`~Z0*_|rl`_JkFm~gTCZ#l*c&mRw{TvAz=*FvlhFwHN>*7y<^d`@fF|I2RWcE<g!PT7imF0M`>o?mIO1$|~>`3}z?Z)o)+$9|a;>&p$C'
    '%IxkaV1f1JX6bER_NC+)vA*2NkB;qMJy>CVIYHjlPu|u!{#{>=^0J#+9yo7f#|G=m&D5OsM*770zw66UE@HaK<1;f21Ymu+wR4KkxDB8DQ(x}P4VwRBV#MW;'
    '>*YXS2GglZI+>`vxaJ}vx|II;iP*5v{H+Xs@jAcq!Fc`vf23Wz`DAN|4wAc=v3PL4WPL@UZ3wcDL0pe)sDU|`V+q=UOXBUrQkvuXa+^$FetIw+;Fd^W!eW}z'
    'k}X@{fI1TSD)+5R;@v<JgyVVKaTE8#pj7lQ(G{v4U$%>N;4xh9>}UC~<+^%1Reufd|1@4dwq#lwTD}#8{hxX^p`x4SW+IH@7^m-$c(Q1%1HvdTy>R;`QSF+t'
    'iS^#b^mBTCb{E@d2*7%82Jx@%#Jo-*)_Zfy-g>tnhEDIzgoi9(alC-4_eS?9zD3dW<rbQ$T2}c0y*H8y_3O^3bz7gEyM$N~&M6eO#Yi2a^TkpOmzlEerzg6J'
    's6Re}YRq4z&knCUt=<qQZym?5zFdZ^{=N1M4aE9#(n5Sv#s_&|eYvH5ck}#I?@{&T{=mzQ<ws{N!}YIwrh_;E@Uk1ztw$oyg%i(71xEz;i&fIieJrN!zgF3o'
    'gG@@#7Fsv0D&4t3>k_fPT%)9G#r~Rqy}lgXxi>7AJkIYZ@oSH>{mBIKBk&=89vX4He^?jkRw1}=Z&?nB#+uWeE6T2SmbH<y(ETFx@5`X_D0v*eTG+M~*|!4o'
    'jcf;X=ian>snunuBp;q{aSZ5_Q#NORQryj#T^K$P_%T^l@&xB!E<}=<@puan@_7CDV7cMwb~&ss3;MvLXB#f1EkBQ5CE7`^5|vG7U*d-M;nw=Xtpgzs5hlDZ'
    'ufq)*e$_(#-PMf`g<)NRbAv87w3R~<!hLv9MrFkJDi<7owyr(Ec6j?s)EYOBAYZsk!>^{@8MSvT!1`>U{;<n#KPQx!HMzbl6i1X3+g4wja}iNtaNP~{Tm6eU'
    'eNZQ&K9s6aAzy?vo$r@M{`%_A3)7HI95`JXmM1`el@fnglXQ!!pJRF<;C%D*;ylC6Zwin;JU=Jy3Yb4q{ZLLTJX>6W^&2ixzvf#~wCR4qt3UjTsPoma`f-ZV'
    '0hjN{eDM6hd^Jke<+l0xk4QA`Ib3gN)npp09~XAFI@_0F`Z>T?aWB`E7jw%75EFczL_R^QA9au4-!98K7vz6E?0WK7^<-XQST{lbh;^t)Sn1^QF|_XCPpJjz'
    '9}Xic8GP<X%5A1kn-WJH&$ssP`hBd4&ObQy%ZGTAqrnIZJMO%3qKY<u%rG7_d$Lsk#*aHodVZhIXNgWH^6yx=b@+15bh^3TDr8Zp*z_mJThgDpP~-Y?bG@Ry'
    'nIkct2GE7lIHvcpUN>WMeJ1F>e6R?6Z}q?Zd5uJ$R~?QCXpYy9u{EulUIrLn7R-TFXIpH@%eu<2Z)DhGm(`(?5ybPG3D$XO4y>|iOGF9z;~6QEr=t?4{KxcH'
    '0soN_4-B063tdC7{0Itk_2V}`e*fr4<v-Ht`3IHK@T*aUg)7f(#rpgJu@bn(g#1<96iE{$xZj$-df&|W=zJV>l&*`c+L(r}n?v89^SWFE@*^dd6dYd=ag#{_'
    '#K|Mz_Z0JOc#`g;JGgOkOY+{*@%*?VGyj}-k}TYpH7BWhB<C^C5BO##E|h%wA*#eZf!`emR$LSwmhDY^C;|K3ZBQ!{@2b6t*5Um^>}mjhmGTmGE#iNAwi4^N'
    '*igUz&CcB|xbguij5{xT>3^l+bEI4b-hZEs_4UBh3W}ECly|p9BpIyd8iW&{t+l#{{$yrs-4}6{J*65A$PFg?{nE$w6X?rPf|9Mys-+Lbv3LL+A0M5XlI|6F'
    '3ptIyPKvMMSpDc*rk?hGx-96kf&HoFVWlR#CIy*P{)hTy&6oPuCY|+0Hxb>Jut)uC*2r+eearaKJMZJI^CpKYgzk%NNI=c~Q(>q%QN5G&HP-8+(o{H}zZ^AL'
    'uTkKL^h-}rSC7pfAAAttGGB}7{DQjr8>@=t!jbhzkrcu2Ra?ArI5>MMeD34rsvmawbQx7Ax-WxExGwfI(&5}JP6dB>Ym@sY-&rJ{RzGUX2_*M)3V}EQGX8re'
    'Q>M>%{D{w|e!3ao<xyrhPFyHVd3@HZ7%5`My+=P1$)quV<S8Y3tLraf^Bd4<zVWPPswiIv7575#gX2ldkGuBf_=haU&Ql;V6dvXIv?a`Ya`_RStEW$id9oyn'
    ';dKSWhO5lf^AIg|JWraHhR->zz&>w!7<QfDeYlmp<<tnj6EaM6Zg}=tXYNHZoeqwhc$3wkZPDl&2Cwr!gU_MF^P@XmK31<{g7G;O6<c`BJ{A+}=Zwrr8<M#o'
    '3gdHX>d$O)ce+ljpQGDjgc{uwg7tG)B1N|6yZwVdveg!`zVr?{TxiIG5B%&-mx=Xr411CkS%n8-eyi!{UqMn%kN)nDOvyc`>tYNfE(^l`$hQP7K3p4lPW4Bo'
    '<0Cp%)9Od``A$vdUl>ml;O?if%;_o)j6`{f@{a6?Pl99{onF{Rt__Lv)Gkl19)S0+M(3MT^E{CW{g}u4t9c1OHTOLeh5eX0R!&1X?NS)_V@{tkeVOCas)_xW'
    '$L5dK#hIHf>4?I9%&!&hO6v9}5c@H|ih8~;QvWMle*}0Q&3WlnIk6w}7(PcnEO$5aJYm?6*?F_`mAHb7#D2_cn%(ry{^W-Hb7a3gJ|1_Ii19}~E8ep6es6<$'
    'eqpDN1no8Qnix+ah{ZeMy~eg}@92DR(to#o3Gcr#v7XHs|FN0HM@c$W0@kxZPZj(Byk0i3p3PW)m1F-@(JW(eSkI<cC+*FAUvH|O&0jxv4)l%~If}#kll12B'
    'SMC=N|Im9OUu4%D$FCl8GE`%2lLXHlJS}vvtvNNmlZpqTdw$D9X!A$Tcuq@|57OZIIf1a6Dc8)~<UYA`3iKSddMQ_|nUh7V`#r%reS6X7>Uq`3Ab!r|6HU_x'
    'DD_;g^}$*OTqnr?`<J_9l^LZ?uAd9f5rclsGpY_mONmx&e889Se7yv`KTlZXh;B<iPpqFK6Q5igbBfMie#Q<(y$yN)pr6C~npaG!miOO|&-v^8@#|aVc*(~k'
    'c>aj=C$BDl+)O-wRPrE&gamWI@3rlblkXyLSTj6-d??R4>!bQV%pVtU*d438L^pqYw|Ilfk#4iU>yuO7HH#HT+@8oW!8!)9;^_Qe%5HzxC#S@vBEA#WoEKPO'
    'ee%G**1#OW-GA37r{sc+9d~LCFYv?q<R1=%I49jX@OOQ3N?aO=eeHkekpQeu{`uvL`{DT{sy_K&KkHxHhJU-p2G<|+t#yo3S9kuYr~M*_rY|=XSjd#iAp`5g'
    'a^8Fxbu~K&VYDyHn;oQ7QWk?U8n>Fe77G@vU|dgI*jl#9sO-VSdfH?8(f#qxi{2K3Fn`|lxqMx$Wg5e{<#+{sSII>vRab~Ep6lG8%^zzvZE=;#;-%}Wn}?3F'
    '9*ajNbv}olAJbOr0;z;3lyxHg?y>sutaQPn<*Tr{Gw44mo#==wQSn0~iGHa$+2dmDHx@X)N*_MAT&TYlIY7dlhc`Z;@gHZO5n3@#nhVtbpTKiC)zO>sG?if-'
    's`SbG19JzA(bWvD8?@=x&MT~=>NL>dUhdr^kK<Pt)+KyMND+qnOx3TNlhpzrBKL6pPOqPkevZBDTUVP|v%!7=%rEaHH@#QyYDUlC`syUHBwBvtD!=j6W<3XZ'
    '?qCA=WU<~g7asq~ook`ze(xkp_M$t9#5{0bBWd3m1s~z_;`#@BKl%yePfmvd+Se@pt3M7|grl8AbxCTH_KLN~{`K?63Hgtdy<Xh5JN3$K5*&ihNG`M6TA2EG'
    '{v+ktsJa+S+4oL``Hz*hd2R_evi{D0q&%C(2JKc3MRAz_cp^UCuf^J&%76SjZtKkT{p(%}z|YBL_3aO}S@Fbt#8T0+n4Dp{y7QicNwk7iJi~m%;#4VpO}~(d'
    '`G{lnqrh=~A@sE%%tt)IS}%QS<5^-pqSV8WD;o9a_=t6zeyKHR9GjSrIM!b+uMvxE@Dqahh>n`O3oF<+Q2B^|{VcmQwI4kw1mBlmhc64vSXVkRe~t%>2eBow'
    'Y)7PIVgB5n?jz<`=2cG2pX0^)SAyHf3s81(m_HZ%b<teT(t0X?ju~HH@nL^X-P^rWrFpUWD!5R^0IMOx(0ue2zWzt6LH$Ve(_q$?zLX98ElEJZPOZcFhscaN'
    'R0W6gYxkD~^G8aq>)PP^;@x=((9Z=-iFtF&^1oaSM$b&9?+o>2d3_S}l`6uJ3<mqa``jPg-ND+7Wa8`b5RBhJ$yale+Zvse<iT|U=SDZFqh>!u`>1nLy1Y8C'
    'lvY2g%sV)%)=?Jl{6YPQtYmpCBNL4f;+CGxA2yZ!APV#Ry9$2n3A<l9xj6;2&dlQbTdS!1sL-D}>gf3V@!XGdpUo>I!F7U#_Zl9~tsID>>b?E-6MuBeN1e`L'
    'SdVZp{*Y!<Q1aw-X`ppdS<Dw*&xN6uM14!8WT9hz26Q;0^K0z{HP;;@=09HV+&<#*o{ndWbUCN$*Y^_hAAdZ&Uo}HuD$IZEaJt<_9wIZ$e|&7$x@X6;>;K1p'
    'B+{1?Uz8!TZZ?)5LA}HGZB)dj-ju2@_lK_%GZybuqo2E95yf<AfUnZ%HeV8d=Q%`XDgE<ZSTF?gBPIS!|7GqpI*joj!TYKAV8e~7>$?%3Mf7>cx^V#TJ1DXF'
    'QdQ=<bHNxtpo#kZ)fLwknW&sbN>u3A$$Iva$Mti<*^@R^FT(cIJ@Bee`A|)Y6+$Ez66}aW)4u&?>!I_z^AtQJZqU|(B;fq@8iM;kexwBJ_dnV~Z=Do{eUA3Y'
    'ES6N`dQa6=q|+xUdP^pa)sH40?8{%-V|plA0BSIs>D{ZdhbG6jfX><GGhN~bPNpG*esCXm1U;)MVukUY!Wt`ng)>`H`3QggY+keU>qBKaUo6M%GoJH1HllSn'
    'y_YyIX#UvFKF8JlnLKz70RNy|eRgZ`i4IgAx31yGAk827*DU_b@8OuvIN+t7IrOb|rF}U1o#;AK?fQ;~D+t2Zc}_ly=Z9&*<nAZz`?Ye~;&1l8NGj2aA-TMV'
    '8=E-c{lf2lk7u)20}_GLrSlIVX;O1ICI82r8{AXlhv~$3P{02ga6tU#&NqlG?!E-sje`1-l0SSkw*IQk#`u<Rz`8^)S6=NFqz_*Q4rkEts}ee*MzwP>{z?k<'
    '>np#ud|veG8luJEeOX_~w$Of48k&XQ*K_)4^T*M<8%{|bz<852p#K<IwTvq+onai&!NQk{OFBcT_1s@S%z5|E?W-Wc`}(lr!95o=3Xp{i<^iL^GAHIp8^o0;'
    '&s8Lp745lC$6J2rb@ge@f_$`>e**mY7(T~hQ7CKqI_z8v>i^QHsn4HWIDyXLnZ$KM=i0Ovy8Hsu?2rmp#@1O{T|SriDC+<8<#Z;{UmL@(Rusn^?Bc=lBRC+x'
    '-gRG5JVzU4CL;b^;j3Aw^o|Y2uL|0Toz76tMf5muJh(Y0Y5L@KOO~_t-N1O0;N;SfZDeI$7e<UPPkWe54*bmq*H?ZYw_kq2euL_#MYp9oi)i>&)85r;HfOPV'
    '6*zN8_3@csjgLeKd3TOJx0CPOAA#qV;>q2^ORQ6o)i`_rnXR0bA9Xd?UF#mj`12;<&)=Ap;?wgSwZg|O!(du|3{I8(loUD*)_WONVA{%b^YrA-=g@mS&nJXG'
    'XE1egI5=qC_TyWHI$f?KFDKGp=fTcXJ6A-gKdTaf_1BZ5!#cx!ZXu~`I6T*<DlmVf<j3b{g&*6OV)H;S$@|sn^S!U39dXC^ujZgMe9rX&RpVqApie>qJLFl?'
    ';Hv{%_LDpJLUEGWtg~a`g{zZ`2hcj5sK+Bp%(=*XobDdk<{_=WYF%F$x?XS!xK7~w$1Z!j@6&}aV*NFa#XsEKMZd#*lJ_Q41>Ij1Aun+CaomPSG<|^OxBEu<'
    '2gN}Bcnq8~<AkNQFG!r6e=t<9nm*ez!!I-oiDAX{MK~dSa{tt-{a;+9Ks^WgmbI3P3x21^O|EW%>N2b0{!07J#mH_PFQoMJQ`-E|F}!&Ben$bIFXsdL<4m8n'
    'WvGWDtwg_6!|f`akT?qK#O-&C2q?3SM^3TO|DG)03mQJhz<Enx%Kz#>>J9g&eqX|^zOjA%rOD~b@j(4!pD*lv_PNLgoS&UnHzzGWhRWORzU#wD=Arddtf%Tu'
    't6$=T$W2C%keBv*CGtbZPQ8gnClx24b-!6ohPtuFFs%3fpV0@P1i@{0Zs)%GI08~3IDnkzQOb{=yPv8L@Yhd>++)^icOE!D?w;WzXLvM#$~R@3sJ}|t8?qke'
    '>la=CzL8jW<$UamfSp^<pk3+{_*;zW=fn(GCT2#9z`k`qMNb&7l{Q3ehz4dtixW1_mgI)}h*Q2M9MugjMLbw1@Tna0M@~ik`71-Pb3O2>tf_2>Y(83#JZGN3'
    'r*e${s5eK~Jue>fX9qr&bNHS1C+SuqPRtYdMsi|FGl4nuj)%G^JP$l6*I}q8)QB|T>($hD8vik>ZK}_TY-tde)2QDIuuhfED7%H|F-?G59-BXI59Ay887>9m'
    'Lf<1cQUlkHpl^uYl>Ix-1YIT3@zQH%vt*Q<yMi1fTK6l2!`;4B0A9BsV{5&7{6FZZCY-B!;<K3<t_OCg=ubDlUyUZ=&l`jBG<~_=pXW}t3(0`{avH?fXRoyV'
    'H>WaOkKaf$PWX80CUTU7<Mk{^r1@LDVl`ahsgC*2$AbRc=1l>XC!!h7S5hwQ(MBy{D5Ln?2xkG$Rmlm+exely_TO6^qUd~ipOZM%mdwtl;%w+}Jro|;nExYl'
    '*Wkgz9_(BMd@A}LIv<U^-GK7q=CWQl3V=QUCD#>j%ifpw!+4>f?=$nPp*yQ#82XICdUgM%eY4w?Ll9L4*In^eUQ*}!2nomGfxOyo(DZYv=D4nk-zNn3DLN&L'
    '(w3`55%VSzf4tx{8ybRfSr-R8N90&9GVI?m^=qmA6N6XOI)N^}47$+#k-cJ=s}oLRaS&v-oEav8e$SmIcaDbVh$3rcB<@^88D95anLkqQcg%}Vf{(jK;8O8c'
    '=E^*m%9Ov)A1Ti&&5eV3XSNVLe{AvYuo&8v`S<xFB|jQ9PuD0+oeIw%U$(4WUp*uF@A|8h{p+FrV(rap0a$;vqKSXYv99z#^jF(g2y0UNtCV;9)56XI*8Ldn'
    '1^6mN`mip~FpEMmd2sn^vwbPG=A^_pt-zI5+GJtaZ%Td(=VH6KD~K+G``%yU`2pu?KFB^E91o@WJg6Tj`KrBGZk0(N#)AiwQHifN6;!#3&<I>zMrz+n<9D=m'
    '-)R#L!1Sg8vBt%Mz4EEBDU!<IysTZ{Q%?QHd9;tgbtL)zqxB*s$OGJUdhKZd{Z-0&W9t_46+NZcbsnOAudulI*mB>8$T0rA5flaLM@leD&a6CBp(qFH7N9I5'
    'y6H^n%kAyRW~K@BfyVOVTRHOvQw}*$e}MfN(W(96z1Br^Qu=F9{lTLk`o`zcTVxg!ZodqK)9T0BAGfA3)kuT-15{K&dwy>h*GfYO@wsseg^<rJJY=Z;2<Yz9'
    'T>k3<5=ykrC3exF8MFD}b#|zNBA@Cy>fG}0e_7h+Z25Qm@Eoz~woB3L3S+c^=(+1wALkX-6rwxd7)J@X9Bo7{GU3h*^4T=~95<ik9GT~_elGBdbMhKdnHgM*'
    '*x~bWoF$F_`0VOe(N~|a^9^8TcCEekB-H#OGDvh^w3c~Ava#^O=cx^%dY2DNhN8|y>%MF&?ffCa4X=wGHD{{~&12aAasRx-4VO;E5bHY&&k^Ny+eF9n_pqpF'
    '5BeHMjHk$uzj8RCf&)J9@BQ^LpFQvj!*~h}hkTnNw~bUh#a};_S+C_i*aiNx`Z))*zwcA(!SW-x0WN&gQf<VWiTyhqo_%;OzLXvI?^q_1$ZW^&%&>pQ0rRrd'
    'SG94(xGl>d_16Vz{dD>{Wd>`lTmgQS?(Z8arS(^L+&E}h5=8>n3Hqz=)OGB1c9<dsVg$eUc3s%RJPtZsb70`UyxR*dqPz<x;AxKaS3M<f8InHH@iZM8$a~MU'
    ')Sx#e;!TeAS9d!EZDGBM;qJi?o?tef!5Oh1okFCKT%mt-ZTL|-URwRVlgE=c1*6x9<`MqOT94m-PPfjA`Ji-AM<lX+22PLQ=6&GbL5W?x$`P+$ox%85fTy|m'
    'yTF?+=SRpgW}IJ9Rw<3&k)?HZ<J&IGS7{fROV1D1JsWTpos_RTRPT_x^NxSI(~9)t@hb)o06vG3k7t~hy%9&20&y8s03r=Nvre>hAOg6#Ug_gHpf5+s$3<sI'
    'g-g#0!ML7}Pge7-TmKSyiN7iePjf6ke&$)I*&Zhh*W=8H&auX4NyPqo3vbpseU$kD`yBl`s`J@d?GSNYtyI6qYsW0QI=<!jUZXxz4#PZ6^PH`-gqq#a4t#w)'
    'RzFg5-43%?E2a%#_XXe$I^(?pWmmYP`9!~T3_%Jt69wSBbJ|pSN-$T><mQ3Ux_Rq@Zb$eYM`Yya`<Cr)4K(~}g41!*sU*zL1~^k4k=)1G?cRtM;`8{v3|jp-'
    'qtWVk#TsnR4fJJ&3|eLTb7IgQqAOffuG+$F&IQl8kIt6(wr2NT<UYQ?VR@6LpEI*+Z&KABDR4glP9Z%obx%=x1me%&eMtz=F<hCKfsz?q=OnxB7k|ZO#Ckfx'
    'irLm-%p=$7a2Wp?{v+jCjap|k*NW2Bk3X`m4JxVMr1Br(4^?NJ8GB!ldq+>CToZwDnz75xMRFG3L*-fj^pP9K=LD?~*m!ugB*-5?5z*u48|VIeji^0d-_q0-'
    ')Q^<^b4QQBqBs>{Sf8BLvunmtg;KN#e-0%b0r(usOQh%L&OFNs@M1B(u#jc&#|Qe4(0NRM*45+u$GSHith&#zz7STLR5h|&+M#F{ZjG1lZyJ6z?TKd5gZTiz'
    'f}JkP%N;-PTE?R*7~H>mT#Jq&o;FA((F*QeBj#%AoN)X(CUN6VS#$E_c)!qlU9+h>bivXFBmmckB3Z`M{E@Y4(`;@xV)rE)#Jid+KFgJ35JG)fh53?-lD>4j'
    'f`F;=S4S?yOwM;0TF1m>xtTNnH8tMRor`3hX#LeBXPcLS@<Oox{F=kNwk(^Sh3YeykF(6gj}P7sMfVfcWunu#?%0RZ;d5B_j;{J7`U1I*@0V?S0Q7Sxxrd!M'
    'rFh@8X<%Oio~G47;e_OarHC&+zUJSh@jIH|FUSwG$8?3?Q@?%^th%UXM<T-L-)9zEq*&6Jj54Z6EUDplT(tKl!#W!dYxlW`dsQH+%y|Bz512nnP<uIfM*Z_K'
    'o<BHGsRZc@n~OZ7>L1ZP_tFG1lNeU7dVG~DG<_%r?kCovUa@4y2vaPQO*psg-@D1!dlc3`zQUS)-^2DYx{ATSFW1#LU#;~lv2Q}}o4EyZcXPsZV9n+xIlm0g'
    'iTSc){Z+SZD|&Y%@WQ;h&gqKTXR{pt;LH9~T{5R#^xMU^|G<BoVBLQTzZz=uH8*1AFvyPpzq%^>!wOT6y~O^=w>no8uX3g16>?Pdb-!vir}`t))e+XiG=D3D'
    'dv~g<(||4prd!WB^>{j)xECs@ft#}(_)YWg=+*b+Ya=OueGl|4Su?*KPD7@l=ZIeH)$<MsY+J_(*GVcT1S0nIrl9AEc7*k;g5Kd>EbzJwT{*^byKf@89Q660'
    'oHs(tkKH?*SIkk6g?0J6n;tqmU64YoH<{Yv-@Q+65Pm=37i{E9{+*7zVV~fA8SAg|u1xJb@md_#>zT%NnCIro97Ku1c-L*8xogLfD@Y;x1aq3P{J2z1(<Zu&'
    '2cFZ6ZgO6#_VhUNo(NZ9c&6*uCp$?vKPI%q1ajQIgVy8vdZn`bn&bE!&%wC+8xLT*I})I-?g`#i!gUE1W>6n!q0SwRQJ160Ee6*qXU*F@9CVM^ADP8CfJfnj'
    'Fzk;k96ZA~Up<mxe`GQ{N7pB#R$_nTa5<4K-F0;S$n&-sux^ccK<tnFEQfjUr==|H7wXmY$$0P2r^NoqFV$AJ$B)waBVQ8s70gMxO6-s9l>S0Oq=L>L`S2~z'
    'gzbAuiT#mfBNN`*eqx9Hk>{LQe|Jt}GQ)h!FY;NhUpq7s`y+FY<a2x%rSnJTk+}3yPv9T?k)IeKC!R)qhwFdW_$TVq%DjpFkzFMYD$0q|`6EBcI&bV4mCd+6'
    'vX#DIf8am(BWr89v?RXng5%4=MZDjQ_eB05e)VtqtCV~7*?5`p6cUynL8l~xBz*dvm_Jp2m0pLtALK{MJDGBF?p=$EQ$T+FP5nA^{hsiJsai-Lk$~sHRllZx'
    'yUhjnEmPPYi}|lfo*XY7?w`-d2&r40PStzG<IAb_BPIVYxY%`0Cj!$a0r>I*JYV|e-YrLMxc}rCkL#2FPC}He6s6!ksZsR%7kO5;NIM(u+-sN)@Hv!s`{@xO'
    'JyQeh{0bZxcC_Z6Ts&HaoWtYUwpWnH^~qfq{5;|?OA_`+nzbr<ZL*LtnutGVcO(IP4&`NcSDmjbc~b(!twieACg|%$91jXmEBw0DQ$QbpvKb6iyXG1$6$9}H'
    '#Jk6-0&M|pMaUI=ABuF1mLJv48xOYZ!Eh*fVBB<Ji$u`+bI6v-<Q*s;EM}4(mAssQ>M@8z*{Yj%LM1Z>ttMLG_F~4ib*?OMU7Z~yVs&_JI>S1n3f!_!ZtJ{7'
    'I!W~LT=w@{fX|`ikA!0<G|YA7VBW;j69a)MYAwiZ(x1Hh@%r(1q6_J~ofOQwSO2B$-z(XP1moAezDdKcKGS9mJrjxHtLnjsuqOFOf4@1y{vC^*e_F5nor`QH'
    'T5);p7n!Bebbix&Y;(G24x~)3{)f*Sp>i9A(r+?6*OR`}D9k)72sI>{lPX27>-AkKN_QUa_`UnvuE*$kJRYyq9pG0f=Z!(5ifJncWx#t2Iz729y{SznO8=Mx'
    'm%5QOna1=1%x-)UC5z4l?=5&*tXk>Tzs~!k=0tzl0_XK7k86_PeuL2R4o+8{+{x)5L+fUC9UG2Y7ljTGollqQ*IS=#Vu$Zb&N|i8;<vUTBSgAMtS)}<?Nx>T'
    '?fN+r>aSAX?Y+AKN2YD~4e}$n!L_N8KZAJB6YH;r@rl3NIFAR`Uo};AlhOA!Ce~loI3WBg#f|QMZ+3i=QP-dHcl}k$wwv0L*RQ&82Ar<~KHhPt+geAgzuLC$'
    '<I`*{X;^>N^=ZV(uX(8q_k}_@_Dcy$KKQ%-D&^u`cWP>N`HRB!9CNE~ciGMxf9S7zO{u2o=NQhOm%IN3<|7Lbh|ZoDZl|gGpe#glM3W5<c9sQCrPG_5y;fVU'
    '<pEMXksj(8el@QCb)y1{IIM?y=GmQn?U9a12!r*NnRw<*b(I{3^|Vtin`B;|5lXC2%37b$?Ji8GPg;JgU+r<3167}t9)BE3>#wfxMyI;lW9JmWpO-TDnUTtU'
    '9O)$DdvDf%{dH~wojz$o4pZ-E8*_AuIKjH;prXKA)*|q_0Y3MeXAQF_)+Zg)C--5kMtaWD>67Lh31S~w>`$yunsCcSWF(&nevd9MOrMdQTQspgsW5habJ{k)'
    '$>W4DtWV14G^6bM!0U<iNyqBPsU>M{LC1w*ebT)ac^s;NCmGfm-DkajUYBzw!|OJ0I27D2kvp+I>Dc^Hwdu0<!BYaTK50ny_r5*zZ&39~>Gn%l8BIUOv3SMr'
    'iX3bX118N$xyA$c>~pBRCA?14cz*1tms9pE#&nZG9nZV3U&d{~5jje<&xGbDd3^syH@BRZz1?ZIeFDm;KS0<?-c+M)(Zv2;if)JJwZzf+?oG+l5iv8n^nd!R'
    'f15v2_J)E`;Ol}Dzv21g4$s>8yirAepFdLa)#WLlbQcu~!ShGqSI>rfbn2+{N4mOm^ft}^QKv{=_+S{uM{EOm!ephJisK_RWP<r4Y0UpIEAH_1`D6ij{>VvM'
    'wr-JB37R~Suku)aEI-0_M^6{SuYz^Q2W?``*PceLi1L8{`EY|zo-A-&=4`Cjdvhyha(I1c-AY4^?sMcw^aIiFg~XT%JJ)c*@m09T?)xsu*GLe4ekGeh!><-^'
    '+*q-3`wVbju7NSmyt4e!_o0)U*F(Rj<|Ta4t!a>2N6^h>oy!0|hm!w4ZWEVeYr_0V+CiV@6e-o}@g0VJfID(pl=#_08D8g`9_(>Tshm2$qVI3y1AGo8R;I2#'
    '=3b$P;iT_UzxEXSa#s1yBjh~3KK>a`^M5o<QEWLq3p-E6P`|!eb>QoTFE<fsyuRGRgXD4lj=2@t8Q*S6f%^jRtDekG7R%o~MPl&tve}U|{|>7JZ_jIN*gO{?'
    'Rz`&`L|y-V89739N4%~q=bWip0{1OHWLR_VQHx}lSJ0=_d63ID5<N+@;*GQp*HVoEIKD1AeTJ-b@GF(qL$_Z(#nSWvz9xMm2f0lL_XQx1qSPHp3c;Hh=Jndn'
    '+0T9N+f9T~ofwa2-E!)CV$f?uD}IZrtPS+0<2yIMRqtGN*JI*&;F!PF^xJuxQ)=jZ{tRWEZ}BP>POP6h?*CYR`(kQ|D6F5mvEcU4s~g;?`nins<Dc5(H<<gL'
    'Bi3J?VBNIW`g;x(YZA{NtyyNjeB#gjU-CyL=QC+Cjp1{&G`MdU?7{LQh{BgPvnoDX8IIJ?!22U>mX7O_pI<3vxkYX&%wzoB@A~VV_iL(76&}Ypt}iFWDY1*~'
    'pcp)N-}(DOb)aG@GASG*^n7y`4eXI;KF6@1@7<;iiktK<AjUKB`d0rtH?U(b`}40`?6-Zzct+sEN1iYI)vXTQ!~Dnlf{fRX94{k(SDoUA>s755o8}khPf;&?'
    'e64Yy)sIJu4znqU0UZt!P^Mh-BUx>Qdo22j@V$17e$(1Y=gW8CnvUGfYq@9w9%n=8laAGoJAbk#d_9QqS3sY1<>5>9qPo6R9nQagkcXzPpE+^D=U#5)!rPQ3'
    'R~h!xGN_4sy!L_x@`8l#=l<_M8Jj;^@T1L}Ernoz>w|F$j<$myNDk32X)UrY*7NxR$8(>1RaZ0C#G{PPWgm|^T7P1RLzv{~_q9_i3HV!4VrAfE7Y)(_Y##Uy'
    'w6mKcRfWW3ko&R|%s0mJ<Aa3a*w@)pz~2Yv8*BSQ%E@*XNcS{?D?BXN(8!lYhbt5+boqIW`3~B_O5cYu-TZ;3FL!b;f0%VHR`1kcX<m};=yP9m2T^}cN@&Mw'
    'eMJ%Yxtp}dZ<Y_LLIUvkIENHke$>-ONW%u0&nW0Sx1|{;^V!A|<GDii%@0Xu<%j#@8Z$l8qD3yCO8EDPd<EcFDd+u6?zf&t)?<FtexTLP(kN;CJaKY*Jy8Ex'
    'SbH8D)4)lTg~7Rj<%=PuS7+i8Ms<ay72>jlq~lR<dAvRc<lk{-yVwQkt=PN>ycE(&3ilpvGDWw^6a12B?!epHGEO+pt<MgR(?1!5C^I<M`cLsYMo%sIylm%h'
    'nBUPqZPVquCQZcp<nLz|?e}(Lh0je(UWZK!7PDelpS+suV{O-_cw&9>kyG6=b3F%Oeuv)dh^w*grho7|yeuEk_#O6beYxUFzY*#hFflL9drtZ?bvN>r$OkCK'
    '_WXmG)+z9Q(oHR%q9k+)eK=<VeEFEa)!IU`+;tYL4-6*z*(Co*Bh%{9<Lva;nG^SxmLEB?Mgr_auzuhbP{J0w7z8X0MM*^c$J4{pj;?B=<7t@5k<z%Ad7;Pg'
    '^_58}$d8o#&wYRKn=X<htWTB}ZsA@lRg6gB*WF8?%^!QT*Pf0#jPa;1Q@`%bwWu?eUW}F##SsnEcuvz(I=#2N{XgsC{koBz`1+&y3Qa%fMfC|`xzCu6Y9RIN'
    'ug}PJ3WxI<#%b6}`d;ghF+-IZT$dj1Ilv^Hj9L(_a2Seqa_9H}=VJjS@w}B9XAykdqWV;g`FDg2M10|M7J_}M^vtL8@d<|^a&owHgLfqjzsgkTv@d142#mAY'
    'tG`a~fTj{6PK4K&+x2DVeN8&uPl2#|&uj#m5iVx>=Rl4w2mMt_{x>qR|Ft&+<I8~kvR1oV(RWoR(uSXtQu5VUeiVqbXt!Dia9J4d_4@tAd*hN%qm1gy%}7_;'
    'H&p)-DJ0S7e}@~LwEUQ~tlsXoDwZDsPh+cTOUrFPhsp6?p**+WUwfv<?M%dva7E-o?Ncb7?#0sUwK`_38K@%u+|UqB9`}FzTIbZwx(?Ib1B!W<UL~td;mct7'
    'z7(~xe?ja|f|=RhK8fi|B3Y+$z~_yb&*heS21KB$MC*3EaSna-gbv^Muh-A9eL^1B2N>FadyC=uA&?)zsby#I569!Bc}P7QzJA<Kn?G*8et%7UixkL@v#4Kp'
    '`bA4}m_<<W2!H)VDrK2yZKA8!ML5@no12_M1abOZUY?~i|3{JNR2|(fm|meJsD@IxywKum^d8d$x;t!GzS35ls-_$*3fFVISEXK7u4_h*;`tqdNdTWi`9Ft('
    'eJek#!}Q(&kD+pX_(GSxFHlV;Ts)ZRO~bE#j*(nYaa#n$5isX>aGg8T?MMo-&KPG%UcyrOuW-NYvU{Y$PQC`T4_A+qa>{A?IWe!ETW$9Md4~j?z9Np*1@^tN'
    'q2fE}`m!;hwE9u3QG5-XH8`J>z?CVuv^Dx|_a#2pPY_3jWrxnGOu2$CWN_WmA5BGl@tx>iJnp695lw%UTP?Trl#~#-PT)|~Hp6NQ%bcvq`Gr8gr?hx(q~alm'
    '$<>drZbG(VLv?c`vXsHP(_yiC_~E6&2%&D`rH1bV7v$-D%^F|HDR9V?AUZ6#=cgS>(_j64bhUnh1I9xI^Xg{Ki=u1|50IsJy#<3XTK&j(;`{G!w`IV-1{15s'
    '+<8%>#dg#>@UNc@=Hg_(&r{&Msu}d1U)-XT>H|!d2R5Ik<;TvQ*Y5IbVEO>SNx4U?S@3*BHZm#y4(NSw=Kdj+@zj?X2X4*ZxJGCbAAAl=*?EsQqOlScz|G@J'
    'J08*IkHSe8^~KN50QUv>X%$&`D)yUXAx)$|^z)JN`6D+I6E_+q0PBb-So`a5WIaXf+ZJP>`&dq93VfZ%_lC?``yv9-WbnQicdBq3JWNAZGPpmLie88e9U-IM'
    'M1z>z*{vJzm$3ia^#T6s-(h~t@@{+ZFL?ea6l!?z?383;{|<R4>9eCr0<eD0`NX_1@}6{J{|>G6oX;()!mxf0+jRMxHfKzU{X6o9{0?|=(D_(5MSe)HQ1}PE'
    'iQ>u`2A{0yct-nw*&Hd!yUeiO#Gdz0$jW{}fA{a8<f~^n)p6>|0`Ocmzh$v<P|iK7e+S;DV%#5D=sv0A_io_df&Jhr1#8(~A?2t&UYG9G9rCz;M|jR)b><In'
    'Uq}Ekr^O@nKGEPj+C7=R5EPedOo|KQPF+CGFxVFwEdK26-kUTzULo`xI7iAVzQi7(`jh<e==rdCzR0_=oVQj0?sJz_Nrb)(DMJi#dYYu`p)`DsQpfD2-OsVQ'
    '3}nPF7AqSXPbM<l_li@0E3@f(5i)}X_uOkDY5Wf5c?m<ye$RpD+)|s^zMptM$$9{aTf!xQoJub%kY9K{$*$Wp{A%MlrQ)L^f^gm;b?Y2_A|6MKL;Yst<greI'
    '&TnGj?rUswlbnz=qWM+CzJL&}g*@;&j@89ZOWVUyNmYD&u>tv!vi)5P3$;2UG28$UYMH0L<z!1&GrAmKUu|}x@jFyw)(7Tol7Q#A)7{gS@2X8i-H86a_r9^u'
    '=Z*Y?>v(Zz_C>~So6$SCbC}op9Gd=WSMSWzOFJ-q0HBbyH%%sZPD32Sc;}a0`zr&apHSzy^!H^Et$vJEm#OLBht<_!zdVY3XmRgl$mH;=P#yTd=;_Qw_ft^9'
    '=e}Fd{LPy4-LTH@m+<0dt*kZZ8=`}4X`ac}@Isz{d;R!V{^NrHAE{}vzhM4jti7{Hp{5Cy|45I2Xd;i}a~eZKKX-YKA`t)4FYwJ<&0$N#cr_l+{*s2zso!+g'
    '^3^s_Kazk`glrT0vMhghq(>9CF1(h;fAl67UG9F%2HrEEcfWd}X6yM?%m|^5y2oQbRV6EK`1xwQ{-C$!K`rtg=O65~;SPCRUrze9UvQ~BcFs1We*Y`NH=*iF'
    '5|YSZyz@8Tr4sdHJF<qEv30!bT(@YiYeA1N(O;){8<XaE{kTc>`^DAkusrYq#JkVW<A&_p(Pj8Kjq5R*KQfa}!JrTaonP@sjs>a9h5Z=jUv(Z)iD4~BLVpsi'
    'aBtBL3R9=UJ4shvyfBm=J-L27^!yHruUW?az8cM7#`$SZX#)5hO8m(|_DQTgzz_Br&?Ah`lXhnjDnPlJC)l6-SUu1&*V}x<3QTtw?9Ywcs}`2>RH09C`f_pg'
    '{Q#dsiRboD1ldHHF&r3B&{j3)9(q~Vf*Rp`97*RA0e+Q|3uf}J?S8ih%X2_Eg-_?J;U48w>in24RLSxKe`HGTsc5+uHoHz7=7oMbE~9>^rUm^&C$`<0=HH>X'
    '&(PR`1LNm_3U1qp4IZ2MLJ&sl|0OySPhUn{Mb|R8uF}@#)78Lh$UFw~z;R@~uWElHdXs3yTZ69(PFW*xABMbPfNPII3)RmCK5tacV%8kzKVI4Sl*RHFHkSqS'
    'V|n54QhJW*4C_hGG)%OSmk39fGPv$~{Au27y{yUQN9cX%e9Nw~zU%5A`-Sck#IC+VP_7roAK7rP3Pr^t{*V8tLx1ia*(wb1Ih6NPa<zuj20JW20zbbr3FVJF'
    'rESsUc;DfIY8wAhZQ0diYc|Y>qnrA5`P{mU!Wk-v7E!%^|IAaK!VVJLUp*vaJ}Sdmi>l%H52mG$K!24IyAl>|+u-&P%d76-@5pJ{g<MTW-cIBr$A<Oi8p^_#'
    'o>Y{Fb=xjv*O?sNi~gxMb?*sz+#gxN$I<==A_4mqs$W>=)+PE7y@0FBsJOJT`f+x8h{p6H%&!m(o-~G>mVXLJL}%jHDG6wf*N@^q?TW9>!E|?k-pl;na-*8Y'
    '2UL6x9qyh3!msYxRbh9j0L&FgV5d9BxwyV9JVV^qn;>pU`$guzZizvC8C=(Qw%6yAQPSji@KC%<bLiyRwI%~4<WZAkjV^n5`@!)p`?-IEW?mlSc-hhybKG3>'
    'kp&FKp9Z^DJzl#ODt_gUKFnDCxOk)Fk>M)Pr@%h78)qTL2Tn80W6;d?HE`B`j!t92;cV=iX#S77>t7EhyutVgAg`*c{m5Xu+k<{4;e3G2uG8`(^FXa*0~eN$'
    '!NlKrsC$3bmGcbq5jQOgGVQ&WPRv6!V2abZn$ZeBzx7H;gQAl!szh{NOgp_vFTslgK2JST+VlRxoQFs=zAmHiNyqpdTTUArnXeE9ap^Pl>l3#XkG|t+M=s&!'
    '4dhvOY4xMTzR&exi!nSN$d3*mzAe1teQI)jCMfU3A1&PUWmhj^gZq!#r-^Hh*N=(e#o9U*SY53^{oZNoOs%2kxs%iNfa^KakUL$+114AJLhE*0+D}!<ik@6w'
    '_P>hH2{Ao(`sc5oFh1v-L}r?FN+YrU>cI<3Ke;Vuf%R9<&1?6)+Uv-${_5Ek^)c$zSBUjjIj^}oMs?8Xuj=|PXZ2dQmx|A!%O6j|Y5d2PW;(N0%p5|X+~l-8'
    'R9hiBVshs@==avRZhI6Pe*p1YgS#)3`#F{$c~1G5RhKfuJ|jZBCIem)EmYsyKXAQc`SEOK&@rpeSRDv(z5E=9dOB8hBD?VThnxUff3>v7EO3?prk?{Uzva_^'
    '8oS#bo*XU>ia%=nTQ)8z$z>RKujlKgvU*R@<a~~x_p?7YD>w1Pa->fie?MhQsedpf_nhA+bnnd;EboAR{PDo_N|k~qs1L3lpwwkd<ND;XPj%8IF&+x&>$TOq'
    'aoD-t1YOVI-#1I;vpeg`bcX%H)V0iJ_bjzXY_)OMN#QMx`9Frti+;57@-S7`gd`2x=iLi~0=e>#LY(-zY?QR-`25l4!>oD3J@POw<l3>QZFO-e$SICLb@#^S'
    'kJ}Qae|UIA3O<J&7#dxn@<IzWC(`-V?iMoTDHVkA9o!{{S7knWipb&T_yU(`{vA%QB9v1%PY35W2XJ4iN<MGeb`I$#x|5;ueI`9e_+i|O$D<dsmX_rrlk%H_'
    ')*andE*Z{|HMzJ1<#FLfEliRjUg&iV{9I&^#_!nbr>j+?KNZ%Y92SnR%&TcYV_ER{aqN~;@}s21ej)R2OrHb8*&GXOZb^?rKQMUCY?I$C$JKuT&0}V49U2?&'
    'Jh*8(B1$BD>0q@;COMc5?yLW2_#Kq^qjOx%UQ+5O%<u3Pf3P&|V;C{NWA3^#3+-b<Fuy~6u#zufL;IgT<j;F({*O(k^$+*Hz<6n3Q$0|Lm3P)npIl!7o<EZK'
    'EXoe5PLAgXt*g~U617hG6Z1QsX&)9o`dSE{dr3Hu-!!eQrSd!e(1{((k1R{?yH}C;V4mi=y!j(iz9ne&A2^~x+Wc`o-_KhI(z#(CLs#m|S%$^t=uW(E1o;Fs'
    'e=L$K%+ba0Kp+B`S`Juc6*^P#bASD;&ei_bnM1elU#;i+>Y0~_FzO#1&QayAbT0yRAX>5Xw$`DUy}#kP(8rW*91l6)q8CYj@?OUAtFLDz9$Cmb1?Ih^K7O}Y'
    '(Ityvy-=x5Dvvg#`k=dsR``|&+1zSjh3gg#EzTGdWkV{z<FB6|VQZd$@uKrpXtdJ$`9L;qV!ZwsesytAZSJkx+%R5W|Iy)950OMFUY{=SEcB+$AFr$a-m~E('
    '2iTwB1%KOdy-eZQF2t5d55-VOdru{qZjRp>bxp^QtpM#7$Nk=(5E_0pyBAsNeNh7JYk=e}S-!h;?c-MJ`#`VHv5$t&+1kqYk(pf*_Rn$b{}g|9Wgx>ku`4@V'
    '-RnHtQ9kAg^z+8>tBY5oGAEZ~?*qW~K6$Cuyk=u8`iAIzkS*IEu(OhmudjUhXX<I|_{sHwp><yp_kHNfO-2Jmaq-hXoHl=~aGn24KL*>^;N+z!uOgc?%^x*p'
    'a1L{)za^{s(Q0&|2xIH2;;Zg|*%1AI{0^e|qkmlO3p3L}c>dVHn*M98g)w#hNcX-N-U9tq%FCc4qTbVUY#7vKVBWHDU$@}?Xiua=o&Nl){fS3&9G~<3qV8+M'
    'lNde+IKX$f$t<|=Lk?L)6!6??udl8#q4RMZb?BFx)A@?ZgQLq=dR0IlfRcMmw(@^?(1z7zKv~>m<V}9MQxD>Rt3Rwc1vQz*=8v)Kkqa|AFkUm5tj()TRV+D~'
    'JGprTv=19+-HX{>eHLZ3A86=qrP!q7#W2tR*KOBzGcFXMv$^qg=VdT|q{I^EtPLmXSY*I+0M3mmY2`~ihU<`bc-@`2H-O(kdH<Yt9y(Q-g7N30sNajZA8FfR'
    '>4SLTd@8AR(wKjTi>Cgy*U^}sHh59Kl|>zx5t72NulvvALnVR1mFOS-R<jcTK8KQfj1)t~N9SRBM4(e3exq{1@QZrnA_<4HAz^1C%6akE=dQ=!HL>~%oMr^k'
    'dY$iWmnNrE4V};1&7K~VDM&<i60P8^$@9;iPlvnD+Ivx<sGuFah3hwv46o4gqpnuJ%o{n3_o4>ool_R2<ePa-uD*hPuj=!`gr1(O=qaKVo_4vmmk$rZ_zt0#'
    'caF(s-BiCxTz!R%=SP3@BZ+;=7=IO<MlZi-Z)@LngW>+TLybpwPhk{-FxuaU+Y!1*wg#nWozmUU(VMjXDyypJtaqm|eIS5$V!bc1Ob3ajo=^Y!Y1R@+{j`=1'
    '#y6C#l=8J>MbY={^!N=j@|@=1k*B!$z27hFJPhXCHy3VJ;Xij6UC++g-*@`>UE#aU31}>X_eCJc?q%xN@c)12kD9dnxX+(s?-7ebj{6+L^!$099L;R(!T;VH'
    'hX04h5~cOR@qd;L{Ci`BZGS-*vx7F4W==MT82MNDN9jKo#{!x4a_be&=00Kj{Q3WPW%fs`4msNGwXt__cKYANvfJ&mJLGsQOwW4zdGH@D_<vAK0Rj{Q6aWAK'
    '2mp97cU{pW*XYp%008+3000#L0000000000004ji00000ZDn*}WMOn~b6;(5WMyn`E^2dcZcs}B0u%!j000080C+KXT@=+r-Y)?F03ZSY02lxO0000000000'
    '0Du7m1pokTWpr|BV_|b&Y-w&~VP9flb7d}Sb8l`?O928D0~7!N00;nhF?U^wllzBP0{{Sg2mk;h00000000000001h0ci#R0B~h;W@&6+a%E;^a%FB~X<=Vw'
    'Zgg^GZeeX{Wo~qDE@N|cP)h*<6ay3h000O8crkZfo3zS|xc~qF^#K3?CIA2c0000000000fB^vt003lZb98cPVs&F_Z*E_3a&KZ`Vrgt?WMO1wb6;?Bb!B2<'
    'E@N|cP)h*<6ay3h000O8crkZf>DHhIp8x;=<NyEw6#xJL0000000000fC2sr003-ZbY*U1X<=V)W@c<@Ze=cNb8l`?O928D0~7!N00;nhF?U_)gLSbzO#uK2'
    '&I14|00000000000001h0niHo0AX@;WoBVxbZ>JnZEs{{Y;Rw3X=QU~Z(nj{Wo~qGWo}_)Z!T(YVr*$*P)h{{000001^@;C$^ifXR#^c6000'
)
paquete_integrado = _b64.b85decode(ARCHIVO_INTEGRADO_B85)
with ZipFile(io.BytesIO(paquete_integrado)) as paquete:
    for nombre, destino in {
        'metadatos_modelo.json': REFERENCIA / 'metadatos_modelo.json',
        'metricas_linea_base.json': REFERENCIA / 'metricas_linea_base.json',
        'perfil_referencia_entrenamiento.csv': REFERENCIA / 'perfil_referencia_entrenamiento.csv',
        'distribucion_probabilidades_prueba.csv': REFERENCIA / 'distribucion_probabilidades_prueba.csv',
        'latencia_offline.json': REFERENCIA / 'latencia_offline.json',
        'artefactos/modelo_riesgo_reentrenado.joblib': ARTEFACTOS / 'modelo_riesgo_reentrenado.joblib',
    }.items():
        destino.write_bytes(paquete.read(nombre))
metadatos = json.loads((REFERENCIA / 'metadatos_modelo.json').read_text(encoding='utf-8'))
metricas_referencia = json.loads((REFERENCIA / 'metricas_linea_base.json').read_text(encoding='utf-8'))
columnas_modelo = metadatos['columnas_predictoras_en_orden']
UMBRAL_CREDITICIO = float(metadatos['umbral_alerta_crediticia'])
umbral_crediticio = UMBRAL_CREDITICIO
ruta_modelo = ARTEFACTOS / 'modelo_riesgo_reentrenado.joblib'
sha_modelo = hashlib.sha256(ruta_modelo.read_bytes()).hexdigest()
assert sha_modelo == metadatos['sha256_modelo'], 'La huella del modelo no coincide con el original.'
assert len(columnas_modelo) == 19
modelo = joblib.load(ruta_modelo)
assert hasattr(modelo, 'predict_proba') and modelo.n_features_in_ == 19
if hasattr(modelo, 'feature_names_in_'):
    assert list(modelo.feature_names_in_) == columnas_modelo
print('Modelo verificado:', metadatos['tipo_estimador'])
print('Variables:', len(columnas_modelo), '| Umbral crediticio:', UMBRAL_CREDITICIO)
print('Integridad SHA-256:', sha_modelo[:20] + '...')

## 3. Datos históricos y evaluación del modelo
El conjunto de datos público contiene 30,000 observaciones. Se valida su identidad y se reconstruyen las mismas particiones de entrenamiento, validación y prueba de la fase anterior. La descarga es automática.

In [ ]:
NOMBRE_CSV = 'UCI_Credit_Card.csv'
SHA_CSV = metadatos['sha256_csv']
ruta_csv = BASE / NOMBRE_CSV
urls_respaldo = [
    'https://huggingface.co/datasets/scikit-learn/credit-card-clients/resolve/main/UCI_Credit_Card.csv',
    'https://raw.githubusercontent.com/baopuzi/Credit_Default/refs/heads/master/UCI_Credit_Card.csv',
]

# Primero se usa la misma fuente de datos del desarrollo original.
def localizar_csv_original():
    if ruta_csv.exists() and hashlib.sha256(ruta_csv.read_bytes()).hexdigest() == SHA_CSV:
        return ruta_csv
    errores = []
    try:
        carpeta_kaggle = Path(kagglehub.dataset_download(metadatos['dataset']))
        for candidato in carpeta_kaggle.rglob(NOMBRE_CSV):
            if hashlib.sha256(candidato.read_bytes()).hexdigest() == SHA_CSV:
                return candidato
        errores.append('Kaggle: CSV distinto de la versión verificada.')
    except Exception as exc:
        errores.append(f'Kaggle: {type(exc).__name__}: {str(exc)[:100]}')
    for url in urls_respaldo:
        try:
            respuesta = requests.get(url, timeout=45)
            respuesta.raise_for_status()
            if hashlib.sha256(respuesta.content).hexdigest() == SHA_CSV:
                ruta_csv.write_bytes(respuesta.content)
                return ruta_csv
            errores.append(f'Respaldo {url}: huella diferente.')
        except Exception as exc:
            errores.append(f'Respaldo: {type(exc).__name__}: {str(exc)[:100]}')
    raise RuntimeError(
        'No fue posible obtener automáticamente el dataset original de forma verificable. '
        'Revisa que Colab tenga acceso a internet y vuelve a ejecutar esta celda. '
        + ' | '.join(errores)
    )

ruta_csv = localizar_csv_original()
datos = pd.read_csv(ruta_csv)
datos.columns = datos.columns.str.strip()
variable_objetivo = metadatos['variable_objetivo']
assert datos.shape == (30000, 25)
assert not datos['ID'].duplicated().any()
assert set(datos[variable_objetivo].unique()) == {0, 1}
assert set(columnas_modelo).issubset(datos.columns)
X = datos[columnas_modelo].apply(pd.to_numeric, errors='raise')
y = datos[variable_objetivo].astype(int)
indices = np.arange(len(y))
idx_desarrollo, idx_prueba = train_test_split(indices, test_size=0.20, stratify=y, random_state=SEMILLA)
idx_entreno, idx_validacion = train_test_split(
    idx_desarrollo, test_size=0.25, stratify=y.iloc[idx_desarrollo], random_state=SEMILLA)
X_train, X_test = X.iloc[idx_entreno], X.iloc[idx_prueba]
y_test = y.iloc[idx_prueba]
X_prueba, y_prueba = X_test, y_test
assert (len(X_train), len(idx_validacion), len(X_test)) == (18000, 6000, 6000)
print('Dataset verificado:', len(datos), 'registros; conjunto de prueba:', len(X_test))
print('SHA-256:', hashlib.sha256(ruta_csv.read_bytes()).hexdigest())

In [ ]:
probabilidades_prueba = modelo.predict_proba(X_test)[:, 1]
predicciones_prueba = (probabilidades_prueba >= UMBRAL_CREDITICIO).astype(int)
tn, fp, fn, tp = confusion_matrix(y_test, predicciones_prueba, labels=[0, 1]).ravel()
metricas_recalculadas = {
    'n': int(len(y_test)), 'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp),
    'accuracy': float(accuracy_score(y_test, predicciones_prueba)),
    'precision': float(precision_score(y_test, predicciones_prueba, zero_division=0)),
    'recall': float(recall_score(y_test, predicciones_prueba, zero_division=0)),
    'f1': float(f1_score(y_test, predicciones_prueba, zero_division=0)),
    'roc_auc': float(roc_auc_score(y_test, probabilidades_prueba)),
    'pr_auc': float(average_precision_score(y_test, probabilidades_prueba)),
    'tasa_alertas': float(predicciones_prueba.mean()),
}
for clave, valor in metricas_recalculadas.items():
    esperado = metricas_referencia[clave]
    if clave in ('n', 'tn', 'fp', 'fn', 'tp'):
        assert valor == esperado, f'Diferencia en {clave}: {valor} vs {esperado}'
    else:
        assert np.isclose(valor, esperado, atol=1e-10, rtol=1e-10), f'Diferencia en {clave}: {valor} vs {esperado}'
(REFERENCIA / 'metricas_verificadas.json').write_text(
    json.dumps(metricas_recalculadas, indent=2, ensure_ascii=False), encoding='utf-8')
print('Coincidencia con resultados históricos: VALIDADA')
display(pd.DataFrame([metricas_recalculadas]).T.rename(columns={0:'Resultado'}))
fig, ejes = plt.subplots(1, 3, figsize=(12, 3.3), constrained_layout=True)
for eje, (nombre, valor) in zip(ejes, [
    ('ROC-AUC', metricas_recalculadas['roc_auc']),
    ('Recall', metricas_recalculadas['recall']),
    ('F1-score', metricas_recalculadas['f1'])
]):
    eje.axis('off')
    eje.text(.5,.70,nombre,ha='center',fontsize=12)
    eje.text(.5,.34,f'{valor:.3f}',ha='center',fontsize=24,fontweight='bold')
    eje.text(.5,.06,'Evaluación histórica (n=6,000)',ha='center',fontsize=8)
fig.suptitle('Calidad del modelo de riesgo crediticio',fontsize=14)
fig.savefig(FIGURAS / '01_calidad_modelo.png',bbox_inches='tight')
plt.show()

## 4. Registro y trazabilidad de resultados
MLflow conserva métricas y archivos de evidencia en una base SQLite local dentro de la sesión.

In [ ]:
ruta_bd_mlflow = SALIDA / 'mlflow_actividad8.db'
mlflow.set_tracking_uri(f'sqlite:///{ruta_bd_mlflow.resolve().as_posix()}')
nombre_experimento = 'Actividad8_RiesgoCrediticio_Integral'
if mlflow.get_experiment_by_name(nombre_experimento) is None:
    mlflow.create_experiment(nombre_experimento, artifact_location=(SALIDA / 'mlflow_artefactos').resolve().as_uri())
mlflow.set_experiment(nombre_experimento)
id_experimento = mlflow.get_experiment_by_name(nombre_experimento).experiment_id
with mlflow.start_run(run_name='01_modelo_y_linea_base') as ejecucion:
    id_run_base = ejecucion.info.run_id
    mlflow.log_params({'algoritmo':metadatos['tipo_estimador'],
                       'n_variables':len(columnas_modelo), 'umbral_crediticio': UMBRAL_CREDITICIO,
                       'semilla': SEMILLA, 'modelo_sha256':sha_modelo})
    for nombre in ('roc_auc','pr_auc','accuracy','precision','recall','f1','tasa_alertas'):
        mlflow.log_metric(nombre,float(metricas_recalculadas[nombre]))
    mlflow.log_artifact(str(REFERENCIA / 'metricas_verificadas.json'),artifact_path='referencia')
    mlflow.log_artifact(str(FIGURAS / '01_calidad_modelo.png'),artifact_path='referencia')
print('MLflow preparado. Ejecución de línea base:', id_run_base[:12])

## 5. Monitorización de inferencias
Se registran 500 solicitudes de prueba con latencias, eventos y trazas identificables. El registro evita almacenar información personal individual.

In [ ]:
ruta_eventos = LOGS / 'eventos.jsonl'
ruta_trazas = LOGS / 'trazas.jsonl'
# Repetir esta celda inicia archivos nuevos: no se mezclan mediciones de otra sesión.
ruta_eventos.write_text('', encoding='utf-8')
ruta_trazas.write_text('', encoding='utf-8')

class ErrorEsquema(ValueError):
    """Distingue entradas inválidas de fallas internas del estimador."""

def fecha_utc():
    return datetime.now(timezone.utc).isoformat()

def escribir_jsonl(ruta, registro):
    with ruta.open('a', encoding='utf-8') as archivo:
        archivo.write(json.dumps(registro, ensure_ascii=False) + '\n')

def registrar_span(traza_id, padre_id, nombre, inicio_ns, resultado='ok'):
    escribir_jsonl(ruta_trazas, {
        'fecha_utc': fecha_utc(),
        'trace_id': traza_id,
        'parent_span_id': padre_id,
        'span_id': uuid.uuid4().hex[:16],
        'operacion': nombre,
        'duracion_ms': round((time.perf_counter_ns() - inicio_ns) / 1e6, 5),
        'estado': resultado,
        'origen': 'inferencia_local_colab',
    })

def inferir_instrumentado(fila):
    traza_id = uuid.uuid4().hex
    span_raiz_id = uuid.uuid4().hex[:16]
    inicio_total = time.perf_counter_ns()
    estado = 'ok'
    codigo_error = None
    probabilidad = None
    clase = None
    try:
        inicio = time.perf_counter_ns()
        if not isinstance(fila, pd.DataFrame) or len(fila) != 1:
            raise ErrorEsquema('una_fila_dataframe_requerida')
        if list(fila.columns) != columnas_modelo:
            raise ErrorEsquema('esquema_de_columnas_incorrecto')
        if not all(pd.api.types.is_numeric_dtype(tipo) for tipo in fila.dtypes):
            raise ErrorEsquema('tipo_no_numerico')
        if not np.isfinite(fila.to_numpy(dtype=float)).all():
            raise ErrorEsquema('valor_no_finito')
        registrar_span(traza_id, span_raiz_id, 'validacion', inicio)

        inicio = time.perf_counter_ns()
        probabilidad = float(modelo.predict_proba(fila)[:, 1][0])
        registrar_span(traza_id, span_raiz_id, 'inferencia', inicio)

        inicio = time.perf_counter_ns()
        clase = int(probabilidad >= UMBRAL_CREDITICIO)
        rango_probabilidad = f'{min(int(probabilidad * 10), 9) / 10:.1f}-{(min(int(probabilidad * 10), 9) + 1) / 10:.1f}'
        registrar_span(traza_id, span_raiz_id, 'posprocesamiento', inicio)
    except ErrorEsquema:
        estado = 'error_entrada'
        codigo_error = 'VALIDACION_ESQUEMA'
        rango_probabilidad = None
    except Exception:
        # No persistimos la excepción íntegra: podría revelar información de entrada.
        estado = 'error_modelo'
        codigo_error = 'FALLO_INFERENCIA'
        rango_probabilidad = None
    fin_total_ns = time.perf_counter_ns()
    latencia_ms = (fin_total_ns - inicio_total) / 1e6
    registrar_span(traza_id, None, 'solicitud_total', inicio_total, estado)
    escribir_jsonl(ruta_eventos, {
        'fecha_utc': fecha_utc(), 'trace_id': traza_id,
        'estado': estado, 'codigo_error': codigo_error,
        'latencia_ms': round(latencia_ms, 5),
        'clase_predicha': clase,
        'decil_probabilidad': rango_probabilidad,
        'modelo_sha_corto': sha_modelo[:12],
    })
    return {'trace_id': traza_id, 'estado': estado,
            'probabilidad': probabilidad, 'clase': clase,
            'latencia_ms': latencia_ms}

# Prueba de integración local (sin registrar datos individuales de cliente).
respuesta_prueba = inferir_instrumentado(X_test.iloc[[0]])
assert respuesta_prueba['estado'] == 'ok'
assert 0 <= respuesta_prueba['probabilidad'] <= 1
print('Instrumentación lista; ejemplo de traza:', respuesta_prueba['trace_id'])

In [ ]:
# Reiniciar logs para excluir la llamada de prueba anterior de las métricas formales.
ruta_eventos.write_text('', encoding='utf-8')
ruta_trazas.write_text('', encoding='utf-8')

rng = np.random.default_rng(SEMILLA)
indices_replay = rng.choice(len(X_test), size=NUM_SOLICITUDES, replace=False)
respuestas = []
verdades = []
ventanas = []
proceso = psutil.Process()
memoria_inicial_mb = proceso.memory_info().rss / (1024 ** 2)
inicio_replay = time.perf_counter()

for inicio_ventana in range(0, NUM_SOLICITUDES, TAMANO_VENTANA):
    respuestas_ventana = []
    etiquetas_ventana = []
    for indice in indices_replay[inicio_ventana:inicio_ventana + TAMANO_VENTANA]:
        respuesta = inferir_instrumentado(X_test.iloc[[int(indice)]])
        respuestas.append(respuesta)
        respuestas_ventana.append(respuesta)
        etiqueta = int(y_test.iloc[int(indice)])
        verdades.append(etiqueta)
        etiquetas_ventana.append(etiqueta)
    buenas = [r for r in respuestas_ventana if r['estado'] == 'ok']
    tiempos = np.array([r['latencia_ms'] for r in buenas], dtype=float)
    pred = np.array([r['clase'] for r in buenas], dtype=int)
    reales = np.array([e for e, r in zip(etiquetas_ventana, respuestas_ventana) if r['estado'] == 'ok'])
    ventanas.append({
        'ventana': len(ventanas) + 1,
        'solicitudes': len(respuestas_ventana),
        'exitosas': len(buenas),
        'errores_entrada': sum(r['estado'] == 'error_entrada' for r in respuestas_ventana),
        'errores_modelo': sum(r['estado'] == 'error_modelo' for r in respuestas_ventana),
        'latencia_p50_ms': float(np.percentile(tiempos, 50)) if len(tiempos) else None,
        'latencia_p95_ms': float(np.percentile(tiempos, 95)) if len(tiempos) else None,
        'latencia_p99_ms': float(np.percentile(tiempos, 99)) if len(tiempos) else None,
        'tasa_alertas_crediticias': float(np.mean(pred)) if len(pred) else None,
        'recall_observado': float(recall_score(reales, pred, zero_division=0)) if len(pred) else None,
    })

segundos_replay = time.perf_counter() - inicio_replay
memoria_final_mb = proceso.memory_info().rss / (1024 ** 2)
tabla_ventanas = pd.DataFrame(ventanas)
tabla_ventanas.to_csv(SALIDA / 'metricas_por_ventana.csv', index=False)

exitosas = [r for r in respuestas if r['estado'] == 'ok']
latencias_ms = np.asarray([r['latencia_ms'] for r in exitosas], dtype=float)
assert len(respuestas) == NUM_SOLICITUDES
assert len(exitosas) == NUM_SOLICITUDES, 'El replay de referencia debe terminar sin errores.'

metricas_operativas = {
    'numero_solicitudes': len(respuestas),
    'solicitudes_exitosas': len(exitosas),
    'errores_entrada': int(sum(r['estado'] == 'error_entrada' for r in respuestas)),
    'errores_modelo': int(sum(r['estado'] == 'error_modelo' for r in respuestas)),
    'tasa_exito_replay': len(exitosas) / len(respuestas),
    'latencia_p50_ms': float(np.percentile(latencias_ms, 50)),
    'latencia_p95_ms': float(np.percentile(latencias_ms, 95)),
    'latencia_p99_ms': float(np.percentile(latencias_ms, 99)),
    'latencia_media_ms': float(np.mean(latencias_ms)),
    'rendimiento_solicitudes_seg': float(len(respuestas) / segundos_replay),
    'memoria_rss_inicial_mb': memoria_inicial_mb,
    'memoria_rss_final_mb': memoria_final_mb,
    'tasa_alertas_crediticias_replay': float(np.mean([r['clase'] for r in exitosas])),
    'tipo_prueba': 'inferencia_secuencial_offline_CPU_Colab',
}
(SALIDA / 'metricas_operativas.json').write_text(
    json.dumps(metricas_operativas, indent=2, ensure_ascii=False), encoding='utf-8'
)
display(tabla_ventanas.round(4))
print('Indicadores agregados de la sesión:')
display(pd.DataFrame([metricas_operativas]).T.rename(columns={0: 'valor'}))

## 6. Calidad de los datos y visualización operativa
Se compara la distribución de las variables del conjunto de entrenamiento con el conjunto histórico de prueba. El indicador PSI advierte cambios estadísticos; no reemplaza las métricas de precisión con etiquetas.

In [ ]:
def calcular_psi(referencia, actual, numero_cortes=10, epsilon=1e-8):
    base = np.asarray(referencia, dtype=float)
    nueva = np.asarray(actual, dtype=float)
    base = base[np.isfinite(base)]
    nueva = nueva[np.isfinite(nueva)]
    if len(base) < 50 or len(nueva) < 50:
        return float('nan'), 0
    limites = np.unique(np.quantile(base, np.linspace(0.0, 1.0, numero_cortes + 1)))
    if len(limites) < 2:
        return float('nan'), 0  # Variable constante: no definir cortes artificiales.
    limites[0] = -np.inf
    limites[-1] = np.inf
    hist_base = np.histogram(base, bins=limites)[0].astype(float)
    hist_nueva = np.histogram(nueva, bins=limites)[0].astype(float)
    p_base = (hist_base + epsilon) / (hist_base.sum() + epsilon * len(hist_base))
    p_nueva = (hist_nueva + epsilon) / (hist_nueva.sum() + epsilon * len(hist_nueva))
    return float(np.sum((p_nueva - p_base) * np.log(p_nueva / p_base))), len(hist_base)

registros_psi = []
for variable in columnas_modelo:
    psi, cortes = calcular_psi(X_train[variable], X_test[variable])
    std_base = float(X_train[variable].std())
    delta_media_std = abs(float(X_test[variable].mean()) - float(X_train[variable].mean())) / max(std_base, 1e-12)
    registros_psi.append({
        'variable': variable,
        'psi': psi,
        'cortes_validos': cortes,
        'desplazamiento_media_std': delta_media_std,
        'nulos_entrenamiento_pct': float(X_train[variable].isna().mean() * 100),
        'nulos_prueba_pct': float(X_test[variable].isna().mean() * 100),
    })
tabla_drift = pd.DataFrame(registros_psi).sort_values('psi', ascending=False, na_position='last')
tabla_drift.to_csv(SALIDA / 'drift_variables.csv', index=False)

histograma_base = pd.read_csv(REFERENCIA / 'distribucion_probabilidades_prueba.csv')
bordes = np.linspace(0.0, 1.0, len(histograma_base) + 1)
assert np.allclose(histograma_base['limite_inferior'].to_numpy(dtype=float), bordes[:-1])
conteos_actuales, _ = np.histogram(probabilidades_prueba, bins=bordes)
conteos_base = histograma_base['cantidad'].to_numpy(dtype=float)
assert int(conteos_base.sum()) == len(probabilidades_prueba)
distancia_js = float(jensenshannon(conteos_base / conteos_base.sum(),
                                  conteos_actuales / conteos_actuales.sum(), base=2))
assert distancia_js < 1e-8, 'Las probabilidades del modelo difieren de la referencia.'

metricas_drift = {
    'n_variables': int(len(tabla_drift)),
    'psi_maximo': float(tabla_drift['psi'].max()),
    'psi_mediana': float(tabla_drift['psi'].median()),
    'distancia_js_histograma_probabilidades_mismo_test': distancia_js,
    'interpretacion': 'Exploratoria. PSI no mide directamente error predictivo ni activa alertas.',
}
(SALIDA / 'metricas_drift.json').write_text(
    json.dumps(metricas_drift, indent=2, ensure_ascii=False), encoding='utf-8'
)
print('Deriva de variables (entrenamiento vs prueba):')
display(tabla_drift.round(5))
print('Distancia Jensen-Shannon respecto al mismo conjunto de prueba:', round(distancia_js, 9))

In [ ]:
respuesta_invalida = inferir_instrumentado(pd.DataFrame({'LIMIT_BAL': [100000]}))
assert respuesta_invalida['estado'] == 'error_entrada'
eventos_originales = [json.loads(linea) for linea in ruta_eventos.read_text(encoding='utf-8').splitlines() if linea.strip()]
trazas_originales = [json.loads(linea) for linea in ruta_trazas.read_text(encoding='utf-8').splitlines() if linea.strip()]
assert len(eventos_originales) == 501
assert len({e['trace_id'] for e in eventos_originales}) == 501
assert set(e['trace_id'] for e in eventos_originales) == set(t['trace_id'] for t in trazas_originales)
print('Registros verificados: 500 inferencias y una entrada inválida; trazabilidad completa.')

In [ ]:
fig, ejes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle('Actividad 8 | Observabilidad local del modelo de riesgo crediticio', fontsize=16, y=0.995)

ax = ejes[0, 0]
ax.hist(latencias_ms, bins=27, edgecolor='white', alpha=0.85)
ax.axvline(metricas_operativas['latencia_p95_ms'], linestyle='--', label=f"p95 = {metricas_operativas['latencia_p95_ms']:.2f} ms")
ax.set(title='Latencia de inferencias instrumentadas (Colab CPU)', xlabel='Milisegundos', ylabel='Cantidad')
ax.legend()

ax = ejes[0, 1]
ax.plot(tabla_ventanas['ventana'], tabla_ventanas['latencia_p50_ms'], marker='o', label='p50')
ax.plot(tabla_ventanas['ventana'], tabla_ventanas['latencia_p95_ms'], marker='o', label='p95')
ax.plot(tabla_ventanas['ventana'], tabla_ventanas['latencia_p99_ms'], marker='o', label='p99')
ax.set(title='Latencias por ventana de 100 solicitudes', xlabel='Ventana secuencial', ylabel='Milisegundos')
ax.set_xticks(tabla_ventanas['ventana'])
ax.legend()

ax = ejes[1, 0]
ax.plot(tabla_ventanas['ventana'], tabla_ventanas['tasa_alertas_crediticias'] * 100, marker='o', label='Tasa alertas crediticias')
ax.plot(tabla_ventanas['ventana'], tabla_ventanas['recall_observado'] * 100, marker='s', label='Recall con etiquetas')
ax.set(title='Calidad y carga crediticia por ventana', xlabel='Ventana secuencial', ylabel='Porcentaje', ylim=(0, 100))
ax.set_xticks(tabla_ventanas['ventana'])
ax.legend(fontsize=9)

ax = ejes[1, 1]
mejores_psi = tabla_drift.head(10).sort_values('psi')
ax.barh(mejores_psi['variable'], mejores_psi['psi'])
ax.set(title='PSI: las 10 mayores diferencias de distribución', xlabel='PSI exploratorio')

for ax in ejes.flat:
    ax.grid(alpha=0.2)
fig.tight_layout(rect=[0, 0, 1, 0.97])
fig.savefig(FIGURAS / 'dashboard_observabilidad_etapa2.png', bbox_inches='tight')
plt.show()
print('Dashboard guardado en:', FIGURAS / 'dashboard_observabilidad_etapa2.png')

In [ ]:
with mlflow.start_run(run_name='02_observabilidad') as ejecucion:
    id_run_observabilidad = ejecucion.info.run_id
    mlflow.log_params({'n_solicitudes':NUM_SOLICITUDES, 'tamano_ventana':TAMANO_VENTANA,
                       'modelo_sha256':sha_modelo})
    for clave, valor in metricas_operativas.items():
        if isinstance(valor,(int,float)) and not isinstance(valor,bool):
            mlflow.log_metric(clave,float(valor))
    for clave in ['psi_maximo','psi_mediana']:
        mlflow.log_metric(clave,float(metricas_drift[clave]))
    for archivo in [SALIDA/'metricas_por_ventana.csv', SALIDA/'drift_variables.csv',
                    SALIDA/'metricas_operativas.json', SALIDA/'metricas_drift.json',
                    LOGS/'eventos.jsonl',LOGS/'trazas.jsonl',
                    FIGURAS/'dashboard_observabilidad_etapa2.png']:
        mlflow.log_artifact(str(archivo),artifact_path='observabilidad')
print('Run de observabilidad:',id_run_observabilidad[:12])

## 7. Objetivos operativos y alertas inteligentes
La política contempla disponibilidad mínima del 99% y al menos 95% de solicitudes atendidas en 40 ms o menos. Las reglas utilizan ventanas consecutivas y control de notificaciones repetidas.

In [ ]:
"""Cálculo de SLI/SLO y evaluación determinista de alertas de riesgo crediticio."""

import math
from collections import defaultdict
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


CONFIGURACION = {
    "version": "1.0.0",
    "modelo": "riesgo_crediticio",
    "slo": {
        "disponibilidad": {
            "descripcion": "Proporción de inferencias válidas procesadas correctamente",
            "objetivo": 0.99,
            "unidad": "fraccion",
            "tratamiento_error_entrada": "excluir_del_denominador",
        },
        "latencia": {
            "descripcion": "Proporción de inferencias válidas correctas con duración máxima de 40 ms",
            "objetivo": 0.95,
            "umbral_ms": 40.0,
            "unidad": "fraccion",
        },
    },
    "alertas": {
        "tamano_ventana_corta": 100,
        "tamano_ventana_larga": 300,
        "avance_eventos": 50,
        "p1": {"burn_corto_min": 6.0, "burn_largo_min": 3.0},
        "p2": {"burn_corto_min": 2.0, "burn_largo_min": 1.0, "persistencia_ventanas": 2},
        "recuperacion": {"burn_corto_max": 0.5, "burn_largo_max": 0.75, "ventanas_consecutivas": 2},
        "enfriamiento_ventanas": 2,
    },
    "drift": {"psi_aviso": 0.10, "psi_critico": 0.25, "minimo_observaciones": 500},
    "calidad_con_etiquetas": {
        "recall_minimo": 0.78,
        "minimo_observaciones_etiquetadas": 1000,
        "minimo_positivos_reales": 200,
    },
}


def validar_eventos(eventos):
    """Valida el contrato de telemetría sin incluir datos personales."""
    columnas = {"fecha_utc", "trace_id", "estado", "latencia_ms", "clase_predicha"}
    faltantes = columnas.difference(eventos.columns)
    if faltantes:
        raise ValueError(f"Faltan campos de telemetría: {sorted(faltantes)}")
    if eventos.empty:
        raise ValueError("No hay eventos para evaluar.")
    if eventos["trace_id"].isna().any() or eventos["trace_id"].duplicated().any():
        raise ValueError("Los trace_id deben ser únicos y no nulos.")
    if eventos["estado"].isna().any():
        raise ValueError("La columna estado contiene valores nulos.")
    latencias = pd.to_numeric(eventos["latencia_ms"], errors="coerce")
    if latencias.isna().any() or (latencias < 0).any() or (~np.isfinite(latencias)).any():
        raise ValueError("Existen latencias inválidas.")
    eventos = eventos.copy()
    eventos["latencia_ms"] = latencias.astype(float)
    eventos["fecha_utc"] = pd.to_datetime(eventos["fecha_utc"], utc=True, errors="raise")
    return eventos.sort_values("fecha_utc", kind="stable").reset_index(drop=True)


def filtrar_elegibles(eventos):
    """Las entradas con esquema inválido no son llamadas válidas al modelo."""
    return eventos.loc[eventos["estado"] != "error_entrada"].reset_index(drop=True)


def contabilizar(lote, config):
    n = int(len(lote))
    if n == 0:
        return {
            "n": 0, "correctas": 0, "fallidas": 0, "lentas_o_fallidas": 0,
            "disponibilidad": float("nan"), "cumplimiento_latencia": float("nan"),
            "latencia_p95_ms": float("nan"), "tasa_alertas_crediticias": float("nan"),
        }
    ok = lote["estado"].eq("ok")
    rapidas = ok & lote["latencia_ms"].le(config["slo"]["latencia"]["umbral_ms"])
    tiempos_correctos = lote.loc[ok, "latencia_ms"]
    return {
        "n": n,
        "correctas": int(ok.sum()),
        "fallidas": int((~ok).sum()),
        "lentas_o_fallidas": int((~rapidas).sum()),
        "disponibilidad": float(ok.mean()),
        "cumplimiento_latencia": float(rapidas.mean()),
        "latencia_p95_ms": float(np.percentile(tiempos_correctos, 95)) if len(tiempos_correctos) else float("nan"),
        "tasa_alertas_crediticias": float(lote.loc[ok, "clase_predicha"].eq(1).mean()) if ok.any() else float("nan"),
    }


def computar_presupuesto(n, errores, objetivo):
    disponible = n * (1.0 - objetivo)
    consumo = float(errores / disponible) if disponible else float("nan")
    return {
        "presupuesto_permitido": float(disponible),
        "incumplimientos": int(errores),
        "presupuesto_restante": float(max(0, disponible - errores)),
        "porcentaje_consumido": float(consumo * 100) if math.isfinite(consumo) else float("nan"),
        "burn_rate": consumo,
    }


def construir_serie_ventanas(eventos, config=CONFIGURACION):
    """Calcula SLI, error budget y burn rate en ventanas por número de solicitudes."""
    serie = []
    elegibles = filtrar_elegibles(eventos)
    corta = int(config["alertas"]["tamano_ventana_corta"])
    larga = int(config["alertas"]["tamano_ventana_larga"])
    avance = int(config["alertas"]["avance_eventos"])
    if not (0 < avance <= corta < larga):
        raise ValueError("Configuración de ventanas no válida.")
    n_total = len(elegibles)
    finales = list(range(avance, n_total + 1, avance))
    if not finales or finales[-1] != n_total:
        finales.append(n_total)
    for limite in finales:
        acumulada = contabilizar(elegibles.iloc[:limite], config)
        corta_stats = contabilizar(elegibles.iloc[max(0, limite - corta):limite], config)
        larga_stats = contabilizar(elegibles.iloc[max(0, limite - larga):limite], config)
        fila = {
            "solicitudes_acumuladas": limite,
            "solicitudes_ventana_corta": corta_stats["n"],
            "solicitudes_ventana_larga": larga_stats["n"],
            "disponibilidad": acumulada["disponibilidad"],
            "cumplimiento_latencia": acumulada["cumplimiento_latencia"],
            "latencia_p95_ms": acumulada["latencia_p95_ms"],
            "latencia_p95_ventana_corta_ms": corta_stats["latencia_p95_ms"],
            "tasa_alertas_crediticias": acumulada["tasa_alertas_crediticias"],
            "errores_modelo_acumulados": acumulada["fallidas"],
            "incumplimientos_latencia_acumulados": acumulada["lentas_o_fallidas"],
        }
        for nombre_slo, campo in [
            ("disponibilidad", "fallidas"), ("latencia", "lentas_o_fallidas")
        ]:
            objetivo = config["slo"][nombre_slo]["objetivo"]
            global_budget = computar_presupuesto(limite, acumulada[campo], objetivo)
            fila[f"presupuesto_{nombre_slo}"] = global_budget["presupuesto_permitido"]
            fila[f"consumo_{nombre_slo}_pct"] = global_budget["porcentaje_consumido"]
            fila[f"restante_{nombre_slo}"] = global_budget["presupuesto_restante"]
            fila[f"burn_{nombre_slo}_corto"] = (
                computar_presupuesto(corta_stats["n"], corta_stats[campo], objetivo)["burn_rate"]
                if corta_stats["n"] >= corta else float("nan")
            )
            fila[f"burn_{nombre_slo}_largo"] = (
                computar_presupuesto(larga_stats["n"], larga_stats[campo], objetivo)["burn_rate"]
                if larga_stats["n"] >= larga else float("nan")
            )
        serie.append(fila)
    return pd.DataFrame(serie)


def evaluar_alertas(serie, config=CONFIGURACION):
    """Alertas con doble ventana, persistencia, recuperación y deduplicación."""
    politica = config["alertas"]
    estados = defaultdict(lambda: {"nivel": None, "alertas_consecutivas": 0,
                                 "salud_consecutiva": 0, "enfriamiento": 0})
    notificaciones = []
    historial = []
    for _, fila in serie.iterrows():
        for indicador, runbook in [("disponibilidad", "RB-01"), ("latencia", "RB-02")]:
            estado = estados[indicador]
            corto = float(fila[f"burn_{indicador}_corto"])
            largo = float(fila[f"burn_{indicador}_largo"])
            n = int(fila["solicitudes_acumuladas"])
            if not (math.isfinite(corto) and math.isfinite(largo)):
                historial.append({"solicitudes_acumuladas": n, "indicador": indicador,
                                  "estado": "MUESTRA_INSUFICIENTE", "prioridad_activa": estado["nivel"]})
                continue
            critico = (corto >= politica["p1"]["burn_corto_min"] and
                       largo >= politica["p1"]["burn_largo_min"])
            aviso = (corto >= politica["p2"]["burn_corto_min"] and
                     largo >= politica["p2"]["burn_largo_min"])
            sano = (corto <= politica["recuperacion"]["burn_corto_max"] and
                    largo <= politica["recuperacion"]["burn_largo_max"])
            recien_recuperada = False
            if critico:
                estado["alertas_consecutivas"] = 0
                estado["salud_consecutiva"] = 0
                if estado["nivel"] != "P1" and estado["enfriamiento"] == 0:
                    tipo = "ACTIVACION" if estado["nivel"] is None else "ESCALAMIENTO"
                    notificaciones.append({
                        "solicitudes_acumuladas": n, "indicador": indicador, "evento": tipo,
                        "prioridad": "P1", "burn_corto": round(corto, 3),
                        "burn_largo": round(largo, 3), "runbook": runbook,
                    })
                    estado["nivel"] = "P1"
            elif aviso:
                estado["salud_consecutiva"] = 0
                estado["alertas_consecutivas"] += 1
                if (estado["nivel"] is None and estado["enfriamiento"] == 0 and
                    estado["alertas_consecutivas"] >= politica["p2"]["persistencia_ventanas"]):
                    notificaciones.append({
                        "solicitudes_acumuladas": n, "indicador": indicador, "evento": "ACTIVACION",
                        "prioridad": "P2", "burn_corto": round(corto, 3),
                        "burn_largo": round(largo, 3), "runbook": runbook,
                    })
                    estado["nivel"] = "P2"
            else:
                estado["alertas_consecutivas"] = 0
                if estado["nivel"] is not None and sano:
                    estado["salud_consecutiva"] += 1
                    if estado["salud_consecutiva"] >= politica["recuperacion"]["ventanas_consecutivas"]:
                        notificaciones.append({
                            "solicitudes_acumuladas": n, "indicador": indicador,
                            "evento": "RECUPERACION", "prioridad": estado["nivel"],
                            "burn_corto": round(corto, 3), "burn_largo": round(largo, 3),
                            "runbook": runbook,
                        })
                        estado["nivel"] = None
                        estado["salud_consecutiva"] = 0
                        estado["enfriamiento"] = politica["enfriamiento_ventanas"]
                        recien_recuperada = True
                else:
                    estado["salud_consecutiva"] = 0
            if estado["enfriamiento"] > 0 and estado["nivel"] is None and not recien_recuperada:
                estado["enfriamiento"] -= 1
            historial.append({"solicitudes_acumuladas": n, "indicador": indicador,
                              "estado": "EVALUADO", "prioridad_activa": estado["nivel"],
                              "burn_corto": corto, "burn_largo": largo})
    return pd.DataFrame(notificaciones, columns=[
        "solicitudes_acumuladas", "indicador", "evento", "prioridad",
        "burn_corto", "burn_largo", "runbook"
    ]), pd.DataFrame(historial)


def diagnosticar_drift(psi_maximo, n_actual, config=CONFIGURACION):
    reglas = config["drift"]
    if n_actual < reglas["minimo_observaciones"] or not math.isfinite(psi_maximo):
        return "SIN_EVIDENCIA_SUFICIENTE"
    if psi_maximo >= reglas["psi_critico"]:
        return "REVISION_PRIORITARIA"
    if psi_maximo >= reglas["psi_aviso"]:
        return "OBSERVACION"
    return "ESTABLE_EN_LA_COMPARACION"


def evaluar_calidad_con_etiquetas(recall, n_etiquetadas, n_positivos, config=CONFIGURACION):
    reglas = config["calidad_con_etiquetas"]
    if n_etiquetadas < reglas["minimo_observaciones_etiquetadas"]:
        return "PENDIENTE_ETIQUETAS"
    if n_positivos < reglas["minimo_positivos_reales"]:
        return "MUESTRA_POSITIVA_INSUFICIENTE"
    if not math.isfinite(recall):
        return "MUESTRA_POSITIVA_INSUFICIENTE"
    if recall < reglas["recall_minimo"]:
        return "REVISAR_DEGRADACION"
    return "DENTRO_DE_REFERENCIA"


def graficar_dashboard(serie, config=CONFIGURACION, titulo="Indicadores operativos del modelo"):
    """Dashboard compacto de SLI, latencia, error budget y burn rate."""
    fig, axs = plt.subplots(2, 2, figsize=(13, 8.8), constrained_layout=True)
    fig.suptitle(titulo, fontsize=15, weight="bold")
    eje_x = serie["solicitudes_acumuladas"]
    ax = axs[0, 0]
    ax.plot(eje_x, 100 * serie["disponibilidad"], marker="o", label="Disponibilidad")
    ax.plot(eje_x, 100 * serie["cumplimiento_latencia"], marker="s", label="Solicitudes dentro de 40 ms")
    ax.axhline(config["slo"]["disponibilidad"]["objetivo"] * 100, ls="--", alpha=.7, label="SLO disponibilidad 99%")
    ax.axhline(config["slo"]["latencia"]["objetivo"] * 100, ls=":", alpha=.7, label="SLO latencia 95%")
    ax.set(title="Cumplimiento acumulado de SLO", xlabel="Solicitudes válidas", ylabel="Porcentaje", ylim=(90, 101))
    ax.legend(fontsize=8, loc="lower left")
    ax.grid(alpha=.15)
    ax = axs[0, 1]
    ax.plot(eje_x, serie["latencia_p95_ventana_corta_ms"], marker="o", label="p95 en ventana de 100")
    ax.axhline(config["slo"]["latencia"]["umbral_ms"], ls="--", label="Límite de latencia 40 ms")
    ax.set(title="Latencia de inferencia", xlabel="Solicitudes válidas", ylabel="Milisegundos")
    ax.legend(fontsize=8)
    ax.grid(alpha=.15)
    ax = axs[1, 0]
    ax.plot(eje_x, serie["consumo_disponibilidad_pct"], marker="o", label="Disponibilidad")
    ax.plot(eje_x, serie["consumo_latencia_pct"], marker="s", label="Latencia")
    ax.axhline(100, ls="--", label="Presupuesto agotado")
    ax.set(title="Presupuesto de error consumido", xlabel="Solicitudes válidas", ylabel="% del presupuesto", ylim=(-3, max(110, float(np.nanmax(serie[["consumo_disponibilidad_pct", "consumo_latencia_pct"]].values)) + 10)))
    ax.legend(fontsize=8)
    ax.grid(alpha=.15)
    ax = axs[1, 1]
    ax.plot(eje_x, serie["burn_disponibilidad_corto"], marker="o", label="Disponibilidad 100")
    ax.plot(eje_x, serie["burn_latencia_corto"], marker="s", label="Latencia 100")
    ax.axhline(config["alertas"]["p2"]["burn_corto_min"], ls="--", label="Umbral P2 corto")
    ax.axhline(config["alertas"]["p1"]["burn_corto_min"], ls=":", label="Umbral P1 corto")
    ax.set(title="Velocidad de consumo (ventana corta)", xlabel="Solicitudes válidas", ylabel="Burn rate")
    ax.legend(fontsize=8)
    ax.grid(alpha=.15)
    return fig


def crear_eventos_de_prueba(base_validos, numero=900, tipo="errores"):
    """Eventos artificiales para verificar las reglas, no resultados del modelo."""
    if len(base_validos) < 20:
        raise ValueError("Se requieren al menos 20 registros de referencia.")
    muestra = base_validos.iloc[np.arange(numero) % len(base_validos)].copy().reset_index(drop=True)
    muestra["trace_id"] = [f"verificacion-{tipo}-{i:05d}" for i in range(numero)]
    muestra["fecha_utc"] = pd.date_range("2026-01-01", periods=numero, freq="s", tz="UTC")
    muestra["estado"] = "ok"
    muestra["latencia_ms"] = 10.0
    muestra["clase_predicha"] = 0
    if tipo == "errores":
        muestra.loc[(muestra.index < 350) & (muestra.index % 4 == 0), "estado"] = "error_modelo"
    elif tipo == "latencia":
        muestra.loc[(muestra.index < 350) & (muestra.index % 2 == 0), "latencia_ms"] = 85.0
    elif tipo == "aislado":
        muestra.loc[muestra.index == 150, "estado"] = "error_modelo"
    else:
        raise ValueError("Prueba desconocida.")
    return muestra


def ejecutar_pruebas_reglas(eventos_validos, config=CONFIGURACION):
    pruebas = {}
    real = construir_serie_ventanas(eventos_validos, config)
    notificaciones, _ = evaluar_alertas(real, config)
    pruebas["serie_base_sin_alertas"] = bool(notificaciones.empty)
    for escenario, indicador in [("errores", "disponibilidad"), ("latencia", "latencia")]:
        simulados = crear_eventos_de_prueba(eventos_validos, tipo=escenario)
        serie = construir_serie_ventanas(simulados, config)
        alertas, _ = evaluar_alertas(serie, config)
        subset = alertas.loc[alertas["indicador"].eq(indicador)]
        pruebas[f"{escenario}_detecta_p1"] = bool((subset["prioridad"].eq("P1") & subset["evento"].eq("ACTIVACION")).any())
        pruebas[f"{escenario}_recupera"] = bool(subset["evento"].eq("RECUPERACION").any())
        pruebas[f"{escenario}_sin_duplicados"] = bool(
            len(subset.loc[(subset["evento"] == "ACTIVACION") & (subset["prioridad"] == "P1")]) == 1
        )
    aislados = crear_eventos_de_prueba(eventos_validos, tipo="aislado")
    alertas_aisladas, _ = evaluar_alertas(construir_serie_ventanas(aislados, config), config)
    pruebas["evento_aislado_no_genera_alerta"] = bool(alertas_aisladas.empty)
    pruebas["sin_etiquetas_no_emite_degradacion"] = (evaluar_calidad_con_etiquetas(0.4, 100, 30, config) == "PENDIENTE_ETIQUETAS")
    if not all(pruebas.values()):
        raise AssertionError(f"Fallaron pruebas del motor de alertas: {pruebas}")
    return pruebas

In [ ]:
registros = validar_eventos(pd.DataFrame(eventos_originales))
elegibles = filtrar_elegibles(registros)
assert len(elegibles) == 500
serie_base = construir_serie_ventanas(registros,CONFIGURACION)
alertas_base, historial_base = evaluar_alertas(serie_base,CONFIGURACION)
pruebas_alertas = ejecutar_pruebas_reglas(
    crear_eventos_de_prueba(elegibles,numero=500,tipo='aislado').assign(estado='ok'),
    CONFIGURACION)
# Las pruebas funcionales utilizan eventos artificiales; la línea base refleja la ejecución real.
(RUTA_EVIDENCIAS/'pruebas_alertas.json').write_text(json.dumps(pruebas_alertas,indent=2),encoding='utf-8')
(RUTA_EVIDENCIAS/'configuracion_alertas.json').write_text(json.dumps(CONFIGURACION,indent=2,ensure_ascii=False),encoding='utf-8')
serie_base.to_csv(RUTA_EVIDENCIAS/'slo_linea_base.csv',index=False)
alertas_base.to_csv(RUTA_EVIDENCIAS/'alertas_linea_base.csv',index=False)
disponibilidad = float(elegibles['estado'].eq('ok').mean())
cumplimiento = float((elegibles['estado'].eq('ok') & (elegibles['latencia_ms']<=40)).mean())
resumen_slo = {'solicitudes':len(elegibles),'disponibilidad':disponibilidad,
               'cumplimiento_latencia':cumplimiento,
               'presupuesto_errores_disponibilidad':len(elegibles)*.01,
               'presupuesto_errores_latencia':len(elegibles)*.05,
               'incumplimientos_latencia':int((elegibles['latencia_ms']>40).sum()),
               'alertas_observadas':int(len(alertas_base))}
(RUTA_EVIDENCIAS/'resumen_slo.json').write_text(json.dumps(resumen_slo,indent=2),encoding='utf-8')
print('Pruebas de alertamiento:',sum(pruebas_alertas.values()),'/',len(pruebas_alertas))
display(pd.DataFrame([resumen_slo]))
fig = graficar_dashboard(serie_base,titulo='Confiabilidad del modelo | Indicadores de servicio')
fig.savefig(FIGURAS/'02_dashboard_slo.png',dpi=160,bbox_inches='tight')
plt.show()

## 8. Incidente 1: alteración del historial de pagos
Se simula una falla en la calidad de los datos y se evalúa su efecto sobre la detección de incumplimientos. La respuesta restaura la versión de los datos verificados.

In [ ]:
def resumen_calidad(variables, etiquetas, nombre_fase):
    probabilidades = modelo.predict_proba(variables.loc[:, columnas_modelo])[:, 1]
    predicciones = (probabilidades >= umbral_crediticio).astype(int)
    tn, fp, fn, tp = confusion_matrix(etiquetas, predicciones, labels=[0, 1]).ravel()
    return {
        'fase': nombre_fase, 'n': len(etiquetas), 'positivos_reales': int(etiquetas.sum()),
        'accuracy': float(accuracy_score(etiquetas, predicciones)),
        'precision': float(precision_score(etiquetas, predicciones, zero_division=0)),
        'recall': float(recall_score(etiquetas, predicciones, zero_division=0)),
        'f1': float(f1_score(etiquetas, predicciones, zero_division=0)),
        'roc_auc': float(roc_auc_score(etiquetas, probabilidades)),
        'tasa_alertas_crediticias': float(np.mean(predicciones)),
        'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp),
    }


def psi_discreto(serie_referencia, serie_actual, epsilon=1e-6):
    """PSI entre distribuciones discretas de un campo ordinal de pagos."""
    original = pd.Series(serie_referencia).value_counts(normalize=True)
    actual = pd.Series(serie_actual).value_counts(normalize=True)
    claves = original.index.union(actual.index)
    p = original.reindex(claves, fill_value=0.0).to_numpy() + epsilon
    q = actual.reindex(claves, fill_value=0.0).to_numpy() + epsilon
    p, q = p / p.sum(), q / q.sum()
    return float(np.sum((q - p) * np.log(q / p)))


resultado_previo = resumen_calidad(X_prueba, y_prueba, 'Antes')
for clave in ['recall', 'precision', 'roc_auc', 'f1']:
    if not np.isclose(resultado_previo[clave], float(metricas_referencia[clave]), rtol=0, atol=1e-7):
        raise ValueError(f'La calidad base difiere de la validación: {clave}')

X_afectado = X_prueba.copy(deep=True)
columnas_pago = ['PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']
for columna in columnas_pago:
    X_afectado[columna] = -2  # Alteración deliberada en una copia, nunca en el dataset original.
resultado_incidente = resumen_calidad(X_afectado, y_prueba, 'Durante')
psi_incidencia = psi_discreto(X_prueba['PAY_0'], X_afectado['PAY_0'])

senal_recall = evaluar_calidad_con_etiquetas(
    resultado_incidente['recall'], resultado_incidente['n'],
    resultado_incidente['positivos_reales'], CONFIGURACION,
)
senal_datos = diagnosticar_drift(psi_incidencia, resultado_incidente['n'], CONFIGURACION)
if senal_recall != 'REVISAR_DEGRADACION' and senal_datos == 'ESTABLE_EN_LA_COMPARACION':
    raise AssertionError('La alteración simulada no produjo señales suficientes para intervención.')

bitacora_runbooks = []
bitacora_runbooks.append({
    'incidente': 'INC-01', 'procedimiento': 'RB-03',
    'instante_utc': datetime.now(timezone.utc).isoformat(),
    'hallazgo': f'PSI PAY_0={psi_incidencia:.3f}; evaluación recall={resultado_incidente["recall"]:.2%}',
    'decision': 'Restituir entradas verificadas y bloquear la transformación incorrecta',
    'accion': 'rollback de copia de variables afectadas', 'resultado': 'ejecutada',
})
# Reversión automática de la transformación errónea dentro de la prueba controlada.
X_restaurado = X_prueba.copy(deep=True)
resultado_recuperacion = resumen_calidad(X_restaurado, y_prueba, 'Después')
resultado_recuperacion['psi_pay_0'] = psi_discreto(X_prueba['PAY_0'], X_restaurado['PAY_0'])
assert np.isclose(resultado_recuperacion['recall'], resultado_previo['recall'], atol=1e-12)
assert resultado_recuperacion['psi_pay_0'] == 0

etapas_calidad = pd.DataFrame([resultado_previo, resultado_incidente, resultado_recuperacion])
etapas_calidad['psi_pay_0'] = [0.0, psi_incidencia, 0.0]
etapas_calidad['evaluacion_recall'] = [
    evaluar_calidad_con_etiquetas(x['recall'], x['n'], x['positivos_reales'])
    for x in etapas_calidad.to_dict(orient='records')
]
etapas_calidad.to_csv(RUTA_EVIDENCIAS / 'incidente_01_calidad.csv', index=False)
display(etapas_calidad[['fase', 'n', 'recall', 'precision', 'f1', 'roc_auc', 'psi_pay_0', 'evaluacion_recall']])

## 9. Incidente 2: incremento de latencia
Una demora controlada deteriora los tiempos de respuesta. El sistema genera una alerta crítica y activa un procedimiento para corregir la causa identificada.

In [ ]:
def escribir_jsonl(ruta, registro):
    with ruta.open('a', encoding='utf-8') as archivo:
        archivo.write(json.dumps(registro, ensure_ascii=False) + '\n')

ruta_eventos = RUTA_LOGS / 'eventos_incidente_02.jsonl'
ruta_trazas = RUTA_LOGS / 'trazas_incidente_02.jsonl'
ruta_notificaciones = RUTA_LOGS / 'notificaciones_incidente_02.jsonl'
for ruta in [ruta_eventos, ruta_trazas, ruta_notificaciones]:
    ruta.write_text('', encoding='utf-8')

n_solicitudes = 1050
inicio_incidente = 150
demora_inducida_segundos = 0.065
inyeccion_activa = False
correccion_aplicada = False
momento_correccion = None
notificaciones_conocidas = set()
eventos_simulados = []

for i in range(n_solicitudes):
    if i == inicio_incidente:
        inyeccion_activa = True
    fase = 'Antes' if i < inicio_incidente else ('Durante' if inyeccion_activa else 'Después')
    trace_id = uuid.uuid4().hex
    fila = X_prueba.iloc[[i % len(X_prueba)]]
    instante_inicio = time.perf_counter()
    probabilidades = modelo.predict_proba(fila)[0]
    instante_inferencia = time.perf_counter()
    if inyeccion_activa:
        time.sleep(demora_inducida_segundos)
    duracion_ms = (time.perf_counter() - instante_inicio) * 1000
    evento = {
        'fecha_utc': datetime.now(timezone.utc).isoformat(), 'trace_id': trace_id,
        'estado': 'ok', 'codigo_error': None, 'latencia_ms': round(duracion_ms, 4),
        'clase_predicha': int(probabilidades[1] >= umbral_crediticio),
        'fase': fase, 'origen': 'prueba_controlada',
    }
    eventos_simulados.append(evento)
    escribir_jsonl(ruta_eventos, evento)
    escribir_jsonl(ruta_trazas, {
        'fecha_utc': evento['fecha_utc'], 'trace_id': trace_id,
        'operacion': 'prediccion', 'duracion_ms': round((instante_inferencia - instante_inicio)*1000, 4),
        'estado': 'ok', 'fase': fase,
    })
    if (i + 1) % CONFIGURACION['alertas']['avance_eventos'] == 0:
        observados = validar_eventos(pd.DataFrame(eventos_simulados))
        serie_parcial = construir_serie_ventanas(observados, CONFIGURACION)
        alertas_parciales, _ = evaluar_alertas(serie_parcial, CONFIGURACION)
        for alerta in alertas_parciales.to_dict(orient='records'):
            clave = (alerta['indicador'], alerta['evento'], alerta['prioridad'], alerta['solicitudes_acumuladas'])
            if clave not in notificaciones_conocidas:
                notificaciones_conocidas.add(clave)
                escribir_jsonl(ruta_notificaciones, alerta)
        if (inyeccion_activa and not correccion_aplicada and
            any((a['indicador'] == 'latencia' and a['evento'] in ['ACTIVACION', 'ESCALAMIENTO']
                 and a['prioridad'] == 'P1') for a in alertas_parciales.to_dict(orient='records'))):
            inyeccion_activa = False
            correccion_aplicada = True
            momento_correccion = i + 1
            bitacora_runbooks.append({
                'incidente': 'INC-02', 'procedimiento': 'RB-02',
                'instante_utc': datetime.now(timezone.utc).isoformat(),
                'hallazgo': f'Demora sostenida; alerta P1 en solicitud {momento_correccion}',
                'decision': 'Retirar la fuente identificada de demora',
                'accion': 'desactivar la inyección temporal de latencia',
                'resultado': 'ejecutada',
            })

registros_finales = validar_eventos(pd.DataFrame(eventos_simulados))
serie_final = construir_serie_ventanas(registros_finales, CONFIGURACION)
alertas_finales, historial_final = evaluar_alertas(serie_final, CONFIGURACION)
if not correccion_aplicada:
    raise AssertionError('No se activó el procedimiento de recuperación de latencia.')
if not ((alertas_finales['indicador'] == 'latencia') & (alertas_finales['evento'] == 'RECUPERACION')).any():
    raise AssertionError('No se observó el cierre de la alerta de latencia.')

resumen_latencia = registros_finales.groupby('fase', sort=False).agg(
    solicitudes=('trace_id', 'size'),
    latencia_media_ms=('latencia_ms', 'mean'),
    latencia_p95_ms=('latencia_ms', lambda x: np.percentile(x, 95)),
    porcentaje_dentro_slo=('latencia_ms', lambda x: float((x <= CONFIGURACION['slo']['latencia']['umbral_ms']).mean()))
).reset_index()
resumen_latencia.to_csv(RUTA_EVIDENCIAS / 'incidente_02_latencia.csv', index=False)
serie_final.to_csv(RUTA_EVIDENCIAS / 'serie_slo_incidentes.csv', index=False)
alertas_finales.to_csv(RUTA_EVIDENCIAS / 'alertas_incidente_02.csv', index=False)
historial_final.to_csv(RUTA_EVIDENCIAS / 'historial_alertas_incidente_02.csv', index=False)
print('Acción automática en solicitud:', momento_correccion)
display(resumen_latencia)
display(alertas_finales.loc[alertas_finales['indicador'].eq('latencia')])

## 10. Resultados y evidencias de recuperación
Las gráficas comparan la calidad del modelo y la rapidez de las respuestas antes, durante y después de cada intervención.

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(13, 4.6), constrained_layout=True)
colores = ['#187f65', '#bc5a3b', '#187f65']
ejes[0].bar(etapas_calidad['fase'], 100 * etapas_calidad['recall'], color=colores)
ejes[0].axhline(CONFIGURACION['calidad_con_etiquetas']['recall_minimo'] * 100, ls='--', color='black', lw=1.1, label='Recall mínimo 78%')
ejes[0].set(title='Calidad: detección de incumplimiento', ylabel='Recall (%)', ylim=(0, 100))
ejes[0].legend(fontsize=8)
ejes[1].bar(resumen_latencia['fase'], resumen_latencia['latencia_p95_ms'], color=colores)
ejes[1].axhline(CONFIGURACION['slo']['latencia']['umbral_ms'], ls='--', color='black', lw=1.1, label='Límite 40 ms')
ejes[1].set(title='Velocidad: tiempo de respuesta', ylabel='Latencia p95 (ms)')
ejes[1].legend(fontsize=8)
fig.suptitle('Continuidad operativa | Impacto y recuperación', fontsize=15, weight='bold')
fig.savefig(RUTA_FIGURAS / '01_antes_durante_despues.png', dpi=170, bbox_inches='tight')
plt.show()

fig, ejes = plt.subplots(2, 1, figsize=(13, 8), constrained_layout=True)
ejes[0].plot(serie_final['solicitudes_acumuladas'], serie_final['latencia_p95_ventana_corta_ms'], marker='o', lw=1.5)
ejes[0].axhline(40, ls='--', c='gray', label='Objetivo 40 ms')
ejes[0].axvline(inicio_incidente, ls=':', c='gray', label='Inicio del incidente')
ejes[0].axvline(momento_correccion, ls='--', c='#187f65', label='Acción correctiva')
ejes[0].set(title='Evolución de la latencia p95 por ventana', xlabel='Solicitudes observadas', ylabel='ms')
ejes[0].legend()
ejes[1].plot(serie_final['solicitudes_acumuladas'], serie_final['burn_latencia_corto'], label='Ventana de 100')
ejes[1].plot(serie_final['solicitudes_acumuladas'], serie_final['burn_latencia_largo'], label='Ventana de 300')
ejes[1].axhline(CONFIGURACION['alertas']['p1']['burn_corto_min'], ls='--', c='gray', label='Umbral P1 corto')
ejes[1].set(title='Consumo del presupuesto de error (burn rate)', xlabel='Solicitudes observadas', ylabel='Veces del presupuesto')
ejes[1].legend()
fig.savefig(RUTA_FIGURAS / '02_alertas_y_respuesta.png', dpi=170, bbox_inches='tight')
plt.show()

## 11. Procedimientos de respuesta y valor de la propuesta
Se documentan los runbooks, las acciones ejecutadas, los responsables y la recuperación obtenida en las simulaciones.

In [ ]:
texto_rb03 = """# RB-03 | Calidad de predicciones y datos

**Objetivo:** evitar decisiones basadas en información de comportamiento de pago alterada.

**Activación:** alerta de drift (PSI >= 0.10) o recall inferior al 78% con al menos 1,000 etiquetas y 200 incumplimientos reales.

1. Confirmar que la variación exceda el nivel habitual y revisar el origen de los datos.
2. Suspender la transformación defectuosa y recuperar la última versión de datos verificada.
3. Ejecutar la evaluación del modelo conservando su versión e idéntico umbral crediticio.
4. Comparar recall, precision y distribución de predicciones antes, durante y después.
5. Registrar la intervención y solicitar aprobación de negocio antes de reanudar cualquier uso operativo.

**Responsables:** analítica de riesgos y responsable del modelo.

**Alcance de prueba:** se revierte una alteración introducida deliberadamente sobre una copia del dataset histórico.
"""
texto_rb02 = """# RB-02 | Latencia y continuidad del servicio

**Objetivo:** recuperar la velocidad de evaluación ante un deterioro sostenido.

**Activación:** alerta P1 por consumo acelerado del presupuesto de latencia en ventanas de 100 y 300 solicitudes.

1. Identificar la etapa que concentra la demora mediante métricas y trazas.
2. Desactivar el componente o configuración que causa la lentitud; conservar el modelo validado.
3. Repetir solicitudes de prueba y evaluar el indicador de respuestas dentro de 40 ms.
4. Mantener la vigilancia hasta que la alerta registre recuperación y no reaparezcan fallas.
5. Documentar el evento y el tiempo de restablecimiento.

**Responsables:** operación analítica y coordinación del servicio.

**Alcance de prueba:** se elimina automáticamente una demora agregada artificialmente al ciclo de inferencia.
"""
(RUTA_RUNBOOKS / 'RB-02_Latencia.md').write_text(texto_rb02, encoding='utf-8')
(RUTA_RUNBOOKS / 'RB-03_Calidad_Datos.md').write_text(texto_rb03, encoding='utf-8')

bitacora = pd.DataFrame(bitacora_runbooks)
bitacora.to_csv(RUTA_EVIDENCIAS / 'ejecucion_runbooks.csv', index=False)

recall_previo = resultado_previo['recall']
recall_incidente = resultado_incidente['recall']
recall_posterior = resultado_recuperacion['recall']
latencia_previa = float(resumen_latencia.set_index('fase').loc['Antes', 'latencia_p95_ms'])
latencia_incidente = float(resumen_latencia.set_index('fase').loc['Durante', 'latencia_p95_ms'])
latencia_posterior = float(resumen_latencia.set_index('fase').loc['Después', 'latencia_p95_ms'])
recuperacion_latencia = alertas_finales.loc[
    (alertas_finales['indicador'] == 'latencia') & (alertas_finales['evento'] == 'RECUPERACION')
]

resumen_incidentes = {
    'fecha_utc': datetime.now(timezone.utc).isoformat(),
    'modelo_sha256': metadatos['sha256_modelo'],
    'tipo_prueba': 'simulacion_controlada_con_datos_historicos',
    'incidente_01': {
        'riesgo': 'alteracion_del_historial_de_pagos',
        'recall_antes': recall_previo, 'recall_durante': recall_incidente,
        'recall_despues': recall_posterior, 'psi_pay_0': psi_incidencia,
        'senal_recall': senal_recall, 'senal_drift': senal_datos,
        'runbook': 'RB-03', 'accion': 'restauracion_de_datos_verificados',
    },
    'incidente_02': {
        'riesgo': 'latencia_sostenida',
        'latencia_p95_antes_ms': latencia_previa,
        'latencia_p95_durante_ms': latencia_incidente,
        'latencia_p95_despues_ms': latencia_posterior,
        'solicitud_accion': int(momento_correccion),
        'solicitud_alerta_recuperada': int(recuperacion_latencia.iloc[0]['solicitudes_acumuladas']),
        'notificaciones': int(len(alertas_finales)),
        'runbook': 'RB-02', 'accion': 'retirar_demora_controlada',
    },
}
(RUTA_EVIDENCIAS / 'resumen_incidentes.json').write_text(
    json.dumps(resumen_incidentes, indent=2, ensure_ascii=False), encoding='utf-8'
)

informe = f"""# Continuidad operativa del modelo de riesgo crediticio

## Propuesta de valor

Un modelo de riesgo crediticio genera valor cuando ofrece resultados confiables y oportunos. La monitorización permite anticipar desviaciones, actuar antes de que se amplifiquen y conservar evidencia de las decisiones tomadas.

## Situación de referencia

El modelo validado alcanzó un recall de **{recall_previo:.2%}** en **6,000 registros históricos de prueba**. La política de servicio establece al menos **99%** de disponibilidad y **95%** de respuestas en **40 ms o menos** en el entorno medido.

## Incidente 1. Integridad de los datos

Se alteró deliberadamente la codificación del historial de pagos en una copia de los registros. El recall pasó de **{recall_previo:.2%}** a **{recall_incidente:.2%}** y el PSI de PAY_0 alcanzó **{psi_incidencia:.2f}**. El procedimiento **RB-03** restauró los datos verificados, recuperando el recall a **{recall_posterior:.2%}**.

**Valor para el negocio:** reduce el riesgo de decisiones basadas en datos incorrectos y facilita la identificación temprana de desviaciones.

## Incidente 2. Tiempo de respuesta

Una demora controlada elevó la latencia p95 de **{latencia_previa:.2f} ms** a **{latencia_incidente:.2f} ms**. La política de alertamiento activó el procedimiento **RB-02** en la solicitud **{momento_correccion}**, retirando automáticamente la demora. La latencia p95 posterior fue de **{latencia_posterior:.2f} ms** y la recuperación del estado de alerta quedó registrada en la solicitud **{int(recuperacion_latencia.iloc[0]['solicitudes_acumuladas'])}**.

**Valor para el negocio:** contribuye a mantener una experiencia ágil y reduce el tiempo necesario para identificar y corregir incidentes.

## Recomendación

Adoptar un proceso de monitorización preventiva con responsabilidades claras, reglas de alerta revisables y procedimientos de recuperación documentados. La validación de cambios que puedan afectar decisiones crediticias debe permanecer bajo supervisión humana.

## Alcance de los resultados

Los resultados se obtuvieron mediante simulaciones controladas con datos históricos y ejecución local. No representan un despliegue productivo ni permiten estimar por sí solos ahorros monetarios o impacto financiero real.
"""
(RUTA_EVIDENCIAS / 'informe_incidentes.md').write_text(informe, encoding='utf-8')
print(informe)

## 12. Consolidación de la evidencia en MLflow
Las simulaciones, indicadores y procedimientos se registran en el mismo experimento del modelo.

In [ ]:
with mlflow.start_run(run_name='03_incidentes_y_recuperacion') as ejecucion:
    id_run_incidentes = ejecucion.info.run_id
    mlflow.set_tags({'modelo_sha256':sha_modelo,'evaluacion':'simulaciones_controladas'})
    mlflow.log_params({
        'umbral_crediticio':UMBRAL_CREDITICIO,
        'minimo_recall':CONFIGURACION['calidad_con_etiquetas']['recall_minimo'],
        'limite_latencia_ms':CONFIGURACION['slo']['latencia']['umbral_ms'],
        'demora_simulada_ms':demora_inducida_segundos*1000,
    })
    mlflow.log_metrics({
        'recall_antes':recall_previo,'recall_durante':recall_incidente,
        'recall_despues':recall_posterior,'psi_pay_0':psi_incidencia,
        'latencia_p95_antes_ms':latencia_previa,
        'latencia_p95_durante_ms':latencia_incidente,
        'latencia_p95_despues_ms':latencia_posterior,
        'acciones_correctivas':len(bitacora),
    })
    mlflow.log_artifacts(str(RUTA_EVIDENCIAS), artifact_path='incidentes')
    mlflow.log_artifacts(str(FIGURAS),artifact_path='graficas')
(RUTA_EVIDENCIAS / 'registro_mlflow.json').write_text(
    json.dumps({'experimento':nombre_experimento,
                'run_modelo':id_run_base,
                'run_monitorizacion':id_run_observabilidad,
                'run_incidentes':id_run_incidentes,
                'modelo_sha256':sha_modelo},indent=2,ensure_ascii=False),encoding='utf-8')
runs = mlflow.search_runs(experiment_ids=[id_experimento])
assert len(runs)>=3
print('MLflow: modelo, observabilidad e incidentes registrados en tres ejecuciones.')
display(runs[[col for col in ['tags.mlflow.runName','status','metrics.recall','metrics.recall_durante','metrics.latencia_p95_durante_ms'] if col in runs.columns]])

## 13. Informe ejecutivo y paquete de evidencias
El reporte resume el impacto de los incidentes y los beneficios de la monitorización. Se exportan gráficas, métricas, logs, trazas, runbooks, configuración y registros de MLflow.

In [ ]:
reporte_resumen = f"""# Propuesta de monitorización y continuidad del riesgo crediticio

## Problema de negocio
El uso de modelos predictivos en decisiones de riesgo requiere garantizar calidad de la información, transparencia y respuestas oportunas. Un deterioro no detectado puede afectar las decisiones y aumentar el esfuerzo operativo.

## Resultados del modelo
En 6,000 registros históricos, el modelo registró un ROC-AUC de **{metricas_recalculadas['roc_auc']:.3f}** y un recall de **{metricas_recalculadas['recall']:.2%}** para detectar incumplimientos, con una tasa de alertas crediticias de **{metricas_recalculadas['tasa_alertas']:.2%}**.

## Confiabilidad y objetivos
La política de prueba fija una disponibilidad objetivo de **99%** y al menos **95%** de respuestas dentro de **40 ms**. En {len(elegibles)} solicitudes locales de referencia, la disponibilidad registrada fue **{disponibilidad:.2%}** y el cumplimiento de latencia **{cumplimiento:.2%}**. El motor utiliza reglas de prioridad, ventanas y deduplicación.

## Incidentes controlados
**Calidad de datos:** el recall pasó de {recall_previo:.2%} a {recall_incidente:.2%} tras una alteración deliberada y recuperó {recall_posterior:.2%} después de restaurar datos verificados.

**Latencia:** la latencia p95 pasó de {latencia_previa:.2f} ms a {latencia_incidente:.2f} ms y, después de la acción correctiva, bajó a {latencia_posterior:.2f} ms. La alerta se generó y se registró la recuperación.

## Beneficios propuestos
- Anticipar incidentes que pueden afectar la confianza en las evaluaciones crediticias.
- Facilitar la detección de variaciones en la calidad de los datos y preservar un modelo validado.
- Reducir el tiempo de identificación y respuesta ante lentitud del sistema.
- Mantener trazabilidad de métricas, alertas, acciones y responsables.

## Recomendación
Proponer una adopción gradual de monitorización preventiva y revisión periódica de los indicadores, manteniendo supervisión humana en cambios que puedan afectar decisiones crediticias.

## Alcance
Los resultados provienen de datos históricos y pruebas controladas en Google Colab. No demuestran comportamiento en un servicio productivo ni permiten cuantificar ahorros financieros reales.
"""
(SALIDA/'informe_ejecutivo.md').write_text(reporte_resumen,encoding='utf-8')
(SALIDA/'requirements.txt').write_text(
    'scikit-learn==1.6.1\nmlflow==3.5.1\nSQLAlchemy==2.0.44\n'
    'kagglehub>=0.3,<1\nnumpy>=1.26,<3\npandas>=2,<3\n'
    'scipy>=1.11,<2\nmatplotlib>=3.7,<4\njoblib>=1.3,<2\n'
    'psutil>=5.9,<8\nrequests>=2.31,<3\n',encoding='utf-8')

# No se distribuyen los datos individuales; solo estadísticas y evidencias agregadas.
assert len(bitacora) >= 2, 'Se deben registrar dos intervenciones.'
assert len(alertas_finales.loc[alertas_finales['evento']=='RECUPERACION']) >= 1
assert np.isclose(recall_previo, recall_posterior)
assert ruta_bd_mlflow.exists()
assert (FIGURAS/'02_dashboard_slo.png').is_file()

# Manifiesto de evidencia para auditar archivos del ZIP.
manifiesto = {}
for archivo in sorted(SALIDA.rglob('*')):
    if archivo.is_file() and archivo.name != 'manifiesto_sha256.json':
        manifiesto[str(archivo.relative_to(SALIDA))] = hashlib.sha256(archivo.read_bytes()).hexdigest()
(SALIDA/'manifiesto_sha256.json').write_text(
    json.dumps(manifiesto,indent=2,ensure_ascii=False),encoding='utf-8')
ruta_zip_final = Path(shutil.make_archive(str(BASE/'actividad8_resultados_integrales'),
                                          'zip',root_dir=SALIDA))
print(reporte_resumen)
print('Evidencias generadas:',ruta_zip_final)
print('Archivos registrados:',len(manifiesto),'| Tamaño MB:',round(ruta_zip_final.stat().st_size/1024**2,2))
if BASE == Path('/content'):
    from google.colab import files
    files.download(str(ruta_zip_final))